# Computer Vision Course


### Learning Outcomes

By the end of this notebook, you should be able to:

- Explain how images are stored as arrays (shape, dtype, value ranges)
- Load, display, save, resize, and transform images with OpenCV
- Apply classical CV building blocks (color spaces, filtering, thresholding, morphology, edges)
- Use standard detection techniques (template matching, contours, feature matching)
- Segment images with watershed
- Work with video streams and basic tracking (optical flow, MeanShift/CamShift, trackers)


## Set up a Python environment

Create a virtual environment:

```bash
python3 -m venv venv
```

Activate it (macOS/Linux):

```bash
source venv/bin/activate
```

Activate it (Windows - PowerShell):

```bash
venv\Scripts\Activate.ps1
```

Activate it (Windows - cmd.exe):

```bash
venv\Scripts\activate.bat
```


# Computer Vision Course


### Learning Outcomes

By the end of this notebook, you should be able to:

- Explain how images are stored as arrays (shape, dtype, value ranges)
- Load, display, save, resize, and transform images with OpenCV
- Apply classical CV building blocks (color spaces, filtering, thresholding, morphology, edges)
- Use standard detection techniques (template matching, contours, feature matching)
- Segment images with watershed
- Work with video streams and basic tracking (optical flow, MeanShift/CamShift, trackers)


## Set up a Python environment

Create a virtual environment:

```bash
python3 -m venv venv
```

Activate it (macOS/Linux):

```bash
source venv/bin/activate
```

Activate it (Windows - PowerShell):

```bash
venv\Scripts\Activate.ps1
```

Activate it (Windows - cmd.exe):

```bash
venv\Scripts\activate.bat
```


## Install libraries


We'll use:

- OpenCV (`opencv-python`)
- Matplotlib (`matplotlib`)


Install the packages (inside the activated environment):

```bash
pip install opencv-python matplotlib
```

If you install packages from inside Jupyter, restart the kernel so the imports pick up the new environment.


In [ ]:
!pip install opencv-python matplotlib

> **📦 What just happened?**
>
> `pip install opencv-python matplotlib` downloads and installs:
> - **OpenCV** (`cv2`): the main computer vision library we'll use throughout this course.
> - **Matplotlib** (`plt`): a plotting library to display images inside the notebook.
>
> The `!` prefix tells Jupyter to run the command in the system shell instead of Python.


In [ ]:
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
from PIL import Image

print('python:', sys.version.split()[0])
print('opencv:', cv2.__version__)
print('numpy :', np.__version__)
print('mpl   :', plt.matplotlib.__version__)

> **🔍 What this cell does — Library imports & version check**
>
> ```python
> import cv2        # OpenCV: image loading, processing, drawing
> import numpy as np  # NumPy: all images are NumPy arrays under the hood
> import matplotlib.pyplot as plt  # Matplotlib: for displaying images in the notebook
> from PIL import Image  # Pillow: an alternative image-loading library
> ```
>
> `%matplotlib inline` is a Jupyter magic command that makes plots appear directly below the cell.
>
> **Expected output**: three version strings, e.g. `opencv: 4.x.x`, `numpy: 1.x.x`, `mpl: 3.x.x`.  
> If you see an `ImportError`, the library was not installed correctly — re-run cell 7.


## NumPy and Matplotlib recap

NumPy is the core library for arrays (including images).

```python
import numpy as np
```

A few useful NumPy functions you will see often:

- `np.arange` create a range of values
- `np.zeros` / `np.ones` create arrays filled with 0/1
- `arr.shape` read the array shape
- `arr.reshape(...)` change the view/shape
- `arr.max()` / `arr.min()`
- `arr.argmax()` / `arr.argmin()`
- `arr.mean()`
- `arr.copy()`
- `np.asarray(obj)` convert to an array

Slicing means selecting parts of an array, e.g. `arr[:10, :10]`.

### How images look as arrays

Most color images are stored as a 3D array with shape `(H, W, 3)`:

- `H` = height (rows)
- `W` = width (columns)
- `3` = color channels

Common value ranges:

- `uint8` images: integers in `[0, 255]`
- normalized floats: floats in `[0.0, 1.0]`

Matplotlib can display images inside the notebook with `plt.imshow(...)`.

Tip: colormaps like `viridis`, `plasma`, `inferno`, and `magma` are designed to remain readable for many forms of color vision deficiency.

References:

- RGB model: https://en.wikipedia.org/wiki/RGB_color_model
- Matplotlib: https://matplotlib.org/


In [ ]:
pic = Image.open('data/00-puppy.jpg')

> **🖼️ Loading an image with Pillow**
>
> `Image.open(path)` reads the image file from disk and returns a **PIL Image object** — a high-level representation of the image.  
> At this point the image is *not yet* a NumPy array; it is a Pillow object with attributes like `.size` (width, height) and `.mode` (e.g. `RGB`).
>
> We use Pillow here to illustrate the difference between the two common image representations before switching to OpenCV.


In [ ]:
type(pic)

In [ ]:
pic_arr = np.asarray(pic)

In [ ]:
type(pic_arr)

In [ ]:
pic_arr.shape

> **📐 Images as NumPy arrays**
>
> `np.asarray(pic)` converts the Pillow image into a **NumPy array**.  
> Every color image becomes a 3-D array with shape `(height, width, channels)`.
>
> For example, a 400×600 RGB image will have shape `(400, 600, 3)`:
> - `400` rows (height)
> - `600` columns (width)
> - `3` channels: Red, Green, Blue
>
> Each element is an integer in `[0, 255]` (`uint8`).  
> `0` = no intensity for that channel, `255` = full intensity.
>
> **Expected output of `pic_arr.shape`**: something like `(H, W, 3)`.


In [ ]:
pic_arr

In [ ]:
plt.imshow(pic_arr)

> **📊 Displaying an image with Matplotlib**
>
> `plt.imshow(array)` renders the NumPy array as an image in the notebook.  
> Matplotlib expects the channel order to be **RGB** (Red, Green, Blue), which is exactly what Pillow gives us.
>
> **What you should see**: the puppy photo displayed inline in the notebook.


In [ ]:
pic_red = pic_arr.copy()

In [ ]:
pic_red

In [ ]:
pic_red[:,:,0].shape

In [ ]:
# R G B

# RED CHANNEL VALUES 0 no red, 255 full red
#It is showing red intensity as brightness.
plt.imshow(pic_red[:,:,0], cmap='gray') # show the red channel in grayscale -> the more red, the brighter the pixel

> **🔴 Visualizing individual color channels**
>
> A color image has **3 channels stacked together**: `[:,:,0]` = Red, `[:,:,1]` = Green, `[:,:,2]` = Blue.
>
> When we display a single channel with `cmap='gray'`, each pixel's value (0–255) is shown as brightness:
> - **Bright (white)** → high value in that channel → that area has a lot of that color.
> - **Dark (black)** → low value → little or none of that color.
>
> For example, a red object will appear **bright** in the Red channel and **dark** in Green & Blue channels.


In [ ]:
# GREEN CHANNEL VALUES 0 no green, 255 full green
#It is showing green intensity as brightness.
plt.imshow(pic_red[:,:,1], cmap='gray') # show the green channel in grayscale -> the more green, the brighter the pixel

In [ ]:
# BLUE CHANNEL VALUES 0 no blue, 255 full blue
#It is showing blue intensity as brightness.
plt.imshow(pic_red[:,:,2], cmap='gray') # show the blue channel in grayscale -> the more blue, the brighter the pixel

In [ ]:
# GREEN CHANNEL
pic_red[:,:,1] = 0

In [ ]:
# BLUE CHANNEL
pic_red[:,:,2] = 0

In [ ]:
plt.imshow(pic_red)

> **🔴 Result — Red-only image**
>
> Cells 23 and 24 set the Green and Blue channels entirely to `0`, leaving only the Red channel intact.  
> The result is the image displayed with **only red tones** — all greens and blues have been removed.
>
> This demonstrates how color images are just 3 stacked 2D arrays that you can manipulate independently.


### Image conventions to remember

- OpenCV images are NumPy arrays
- shapes are typically `(H, W)` for grayscale and `(H, W, 3)` for color
- OpenCV loads color as **BGR**; Matplotlib displays as **RGB**
- `dtype` and value ranges matter (`uint8` in `[0,255]` is common)


## OpenCV basics

OpenCV is a computer vision library (originally C++, with Python bindings).

A few practical notes:

- `cv2.imread(...)` returns a NumPy array (or `None` if the path is wrong).
- OpenCV loads color images as **BGR** by default, while Matplotlib expects **RGB**.
- Use `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` before `plt.imshow(...)` when displaying OpenCV images with Matplotlib.

Common functions:

- `cv2.cvtColor` convert between color spaces
- `cv2.resize` change image size (either explicit size or scale factors)
- `cv2.flip` flip horizontally/vertically
- `cv2.imwrite` save an image to disk

Docs: https://opencv.org/


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import cv2

In [ ]:
img = cv2.imread('data/00-puppy.jpg')

In [ ]:
img.shape

In [ ]:
plt.imshow(img)

In [ ]:
# OpenCV reads images in BGR format, so we need to convert it to RGB for correct display with Matplotlib
# MATPLOTLIB ---->  RED,GREEN,BLUE
# OPENCV -------->  BLUE,GREEN,RED

> **⚠️ Important: OpenCV uses BGR, Matplotlib uses RGB**
>
> When you load an image with OpenCV (`cv2.imread`), the color channels are stored in **BGR** order (Blue, Green, Red) — the *opposite* of RGB.
>
> Matplotlib, on the other hand, expects **RGB** order.
>
> **Consequence**: if you display an OpenCV image directly with `plt.imshow`, the red and blue channels are swapped — the image will look color-distorted (reddish objects appear blue, and vice versa).
>
> **Fix**: always convert before displaying:
> ```python
> fix_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
> plt.imshow(fix_img)
> ```


In [ ]:
fix_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

In [ ]:
plt.imshow(fix_img)

> **✅ Result — Correct RGB display**
>
> After `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)`, the channels are reordered from BGR → RGB.  
> The image now displays with its natural colors in Matplotlib. Compare this with cell 32 to see the difference.


In [ ]:
# Read the image in grayscale
img_gray = cv2.imread('data/00-puppy.jpg', cv2.IMREAD_GRAYSCALE)

In [ ]:
plt.imshow(img_gray, cmap='gray')

> **⬛ Grayscale images**
>
> `cv2.IMREAD_GRAYSCALE` tells OpenCV to load the image directly as a **grayscale** (single-channel) image.  
> The array shape becomes `(H, W)` instead of `(H, W, 3)` — no color channels.
>
> Each pixel holds a single intensity value in `[0, 255]`:  
> - `0` = black  
> - `255` = white  
> - values in between = shades of gray
>
> We must pass `cmap='gray'` to Matplotlib; otherwise it applies a default color map and the image looks falsely colorized.


In [ ]:
plt.imshow(fix_img)

In [ ]:
fix_img.shape

## Resize

Resizing changes the image resolution. You can specify a target `(width, height)` or scale factors.

Tip: for shrinking images, interpolation like `cv2.INTER_AREA` is often a good default; for enlarging, `cv2.INTER_LINEAR` is common.


In [ ]:
new_img = cv2.resize(fix_img, (1000, 400))

In [ ]:
plt.imshow(new_img)

In [ ]:
w_ratio = 0.8
h_ratio = 0.2 # 20% of original height

In [ ]:
ratio_img = cv2.resize(fix_img, (0,0), fix_img, w_ratio, h_ratio)

In [ ]:
plt.imshow(ratio_img)

> **📏 Resizing images**
>
> `cv2.resize(src, (width, height))` changes the image dimensions.  
> Note: OpenCV uses **(width, height)** order, while NumPy shapes are **(height, width)** — easy to mix up!
>
> - **Cell 41**: resized to a fixed `1000×400` pixels (may distort the aspect ratio).
> - **Cells 43–45**: resized using ratio factors (`w_ratio=0.8`, `h_ratio=0.2`). Passing `(0,0)` as the target size tells OpenCV to use the `fx` and `fy` scale factors instead.  
>   Here `h_ratio=0.2` means the new height is only 20% of the original → the image looks squashed vertically.
>
> **Common interpolation choices**:
> - `cv2.INTER_AREA` — good for shrinking (less aliasing)
> - `cv2.INTER_LINEAR` — default, good for enlarging


In [ ]:
ratio_img.shape

## Flip image

Flipping mirrors the image:

- `flipCode = 0` vertical flip
- `flipCode = 1` horizontal flip
- `flipCode = -1` both axes


In [ ]:
flip_img = cv2.flip(fix_img, 1)

In [ ]:
plt.imshow(flip_img)

> **🔄 Flipping images**
>
> `cv2.flip(img, flipCode)` mirrors the image:
>
> | `flipCode` | Effect |
> |---|---|
> | `0` | Vertical flip (upside down) |
> | `1` | Horizontal flip (left ↔ right) |
> | `-1` | Both axes |
>
> **Cell 48** uses `flipCode=1` → horizontal mirror.  
> **Expected result**: the puppy is now facing the opposite direction.


In [ ]:
type(fix_img)

In [ ]:
# Save the modified image
cv2.imwrite('data/totally_new.jpg', fix_img)

> **💾 Saving images**
>
> `cv2.imwrite(filename, img)` saves the NumPy array to disk as an image file.  
> The format is inferred from the extension (`.jpg`, `.png`, etc.).
>
> ⚠️ Remember: OpenCV saves images in **BGR** format.  
> Here `fix_img` was already converted to RGB, so if you reload it with OpenCV it would look color-swapped.  
> For saving, it is best practice to keep the image in BGR (the OpenCV native format) before writing.


In [ ]:
fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot(111)
ax.imshow(fix_img)

## Helpers

Many beginner notebooks break because `sample.jpg` is missing. We'll add small helper utilities:
- Load from local path if it exists
- Otherwise generate a synthetic image (shapes + gradients)

This keeps every later section runnable without external files.


In [ ]:
from pathlib import Path

def bgr_to_rgb(img_bgr):
    if img_bgr is None:
        return None
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

def show(img, title=None, cmap=None, figsize=(6, 4)):
    plt.figure(figsize=figsize)
    if title:
        plt.title(title)
    if img.ndim == 2:
        plt.imshow(img, cmap=cmap or 'gray')
    else:
        # Expect RGB for display
        plt.imshow(img)
    plt.axis('off')
    plt.show()

def make_synthetic_image(h=360, w=540):
    y = np.linspace(0, 1, h, dtype=np.float32)[:, None]
    x = np.linspace(0, 1, w, dtype=np.float32)[None, :]

    r = (255 * np.tile(x, (h, 1))).astype(np.uint8)
    g = (255 * np.tile(y, (1, w))).astype(np.uint8)
    b = (255 * (1 - x * y)).astype(np.uint8)

    img = np.dstack([r, g, b])

    img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    cv2.circle(img_bgr, (int(w * 0.2), int(h * 0.35)), 55, (0, 255, 255), -1) # -1 means filled circle
    cv2.rectangle(img_bgr, (int(w * 0.55), int(h * 0.2)),
                  (int(w * 0.9), int(h * 0.45)), (255, 0, 0), -1)
    cv2.line(img_bgr, (20, h - 30), (w - 20, h - 30), (255, 255, 255), 3)
    cv2.putText(img_bgr, 'CV COURSE', (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2,
                (20, 20, 20), 3, cv2.LINE_AA)

    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

def load_image_rgb(path='sample.jpg'):
    p = Path(path)
    if p.exists():
        img_bgr = cv2.imread(str(p), cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise ValueError(f'Failed to read image: {p}')
        return bgr_to_rgb(img_bgr)
    return make_synthetic_image()


> **🛠️ Helper utilities**
>
> These three utility functions will be reused throughout the notebook:
>
> - **`bgr_to_rgb(img_bgr)`** — converts an OpenCV BGR image to RGB so Matplotlib can display it correctly.
> - **`show(img, title, ...)`** — a wrapper around `plt.imshow` that handles both grayscale and color images automatically, and hides the axis ticks for cleaner display.
> - **`make_synthetic_image(h, w)`** — generates a synthetic gradient image when no real image file is available, so that the notebook remains runnable even without external data files.


In [ ]:
img_path = "images/sample.png"
img_rgb = load_image_rgb(img_path)
show(img_rgb, title='Image (RGB)')
print('shape:', img_rgb.shape, 'dtype:', img_rgb.dtype, 'min/max:', img_rgb.min(), img_rgb.max())

## Crop

Key concepts:
- Shape conventions: `(H, W)` for grayscale, `(H, W, 3)` for color
- `dtype` and value range: `uint8` in `[0,255]` is common
- Coordinates: `(row, col)` = `(y, x)`
- Copy vs view (when slicing)


In [ ]:
# Grayscale conversion
gray = cv2.cvtColor(cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR), cv2.COLOR_BGR2GRAY)
show(gray, title='Grayscale')

print('gray shape:', gray.shape)
print('top-left 5x5:', gray[:5, :5])

# Crop (y1:y2, x1:x2)
crop = img_rgb[60:220, 80:280]
show(crop, title='Crop')

# IMPORTANT: crop is a view in NumPy; modifying it can modify the original
crop_copy = crop.copy()
crop_copy[:] = 0
print('Original min after editing crop_copy:', img_rgb.min())

> **✂️ Cropping and grayscale conversion**
>
> **Cropping** in NumPy is just **array slicing**:
> ```python
> crop = img_rgb[60:220, 80:280]   # rows 60–219, columns 80–279
> ```
> The slice notation is `[y_start:y_end, x_start:x_end]`.
>
> ⚠️ **View vs Copy**: NumPy slices are *views*, not copies.  
> Modifying `crop` would also modify `img_rgb`!  
> Use `.copy()` when you need an independent copy (as shown with `crop_copy`).
>
> **Grayscale conversion** uses:
> ```python
> gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
> ```
> The image goes from shape `(H, W, 3)` → `(H, W)`.  
> The output `gray[:5, :5]` prints the raw pixel intensity values of the top-left 5×5 region.


## Exercise

1) Create a function `center_crop(img, size)` that returns a square crop of shape `(size, size, 3)` from the center.
2) Display the crop.
3) Print its shape and dtype.

Hint: compute the center `(cy, cx)` then slice.


### Solution

This is a reference solution for `center_crop(img, size)`. It includes a small safety check (so it still works if `size` is larger than the image).


In [ ]:
def center_crop_solution(img, size):
    h, w = img.shape[:2]
    size = int(size)
    size = max(1, min(size, h, w))
    top = (h - size) // 2
    left = (w - size) // 2
    return img[top:top + size, left:left + size].copy()

cropped = center_crop_solution(img_rgb, 200)
show(cropped, title='Center crop (solution)')
print(cropped.shape, cropped.dtype)

> **🎯 Exercise solution — `center_crop`**
>
> The function computes the center of the image and slices a square region of size `size×size` around it:
> ```
> top  = (height - size) // 2
> left = (width  - size) // 2
> crop = img[top : top+size, left : left+size]
> ```
>
> The safety check `size = max(1, min(size, h, w))` prevents requesting a crop larger than the image.  
> `.copy()` ensures the returned crop is independent from the original array.  
> **Expected output**: a `(200, 200, 3)` array of `uint8`.


## Quiz

1. What is a pixel?
2. What is the difference between RGB and grayscale?
3. Why do we preprocess images?

## Exercises

- Load another image and display it
- Convert it to grayscale
- Print its shape
- Resize it to 200×200

## Mini Project – Image Explorer

Create a notebook that:
- Loads an image
- Displays original, grayscale, and resized versions
- Prints pixel statistics (min, max, mean)

### Drawing on images

These functions draw directly onto the image array (often in-place):

- `cv2.rectangle` draw rectangles
- `cv2.circle` draw circles (use `thickness=-1` to fill)
- `cv2.line` draw line segments
- `cv2.putText` draw text
- `cv2.polylines` draw polygons from a list of vertices

Notes:

- OpenCV colors are typically **BGR** (e.g. green is `(0, 255, 0)`).
- If you need to keep the original image unchanged, draw on a copy (`img.copy()`).
- Callback functions let you react to mouse/keyboard events in an OpenCV window.


This function is a mouse callback used with OpenCV to draw a rectangle interactively.

You click to set a start point, drag to preview the rectangle, and release to finalize it.


In [ ]:
import cv2
import numpy as np

import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
blank_img = np.zeros(shape=(512, 512, 3), dtype=np.int16)

In [ ]:
blank_img.shape

In [ ]:
blank_img

In [ ]:
plt.imshow(blank_img)

> **⬛ Creating a blank canvas**
>
> `np.zeros(shape=(512, 512, 3), dtype=np.int16)` creates a 512×512 black image with 3 channels.  
> All pixel values are `0` (black).
>
> We use `dtype=np.int16` here to allow for negative intermediate values during drawing.  
> In practice most OpenCV drawing functions work with `uint8` — be mindful of the dtype.
>
> **Expected output**: a completely black square image.


In [ ]:
cv2.rectangle(blank_img, pt1=(384,0), pt2=(500,150),color=(0,255,0), thickness=10)

In [ ]:
plt.imshow(blank_img)

> **🟩 Drawing a rectangle**
>
> `cv2.rectangle(img, pt1, pt2, color, thickness)` draws a rectangle on the image **in place** (it modifies the array directly).
>
> | Parameter | Meaning |
> |---|---|
> | `pt1` | Top-left corner `(x, y)` |
> | `pt2` | Bottom-right corner `(x, y)` |
> | `color` | BGR color tuple `(B, G, R)` — here `(0, 255, 0)` = green |
> | `thickness` | Pixel width of the border; `-1` fills the shape |
>
> **Expected result**: a green rectangle outline on the black canvas.


In [ ]:
cv2.rectangle(blank_img, pt1=(200,200), pt2=(300,300), color=(0,0,255), thickness=10)

In [ ]:
plt.imshow(blank_img)

In [ ]:
cv2.circle(img=blank_img, center=(100,100), radius=50, color=(255,0,0), thickness=8)

In [ ]:
plt.imshow(blank_img)

> **🔵 Drawing a circle (outline)**
>
> `cv2.circle(img, center, radius, color, thickness)`:
>
> | Parameter | Meaning |
> |---|---|
> | `center` | `(x, y)` of the circle center |
> | `radius` | Circle radius in pixels |
> | `color` | BGR color — here `(255, 0, 0)` = blue in BGR |
> | `thickness` | Stroke width; use `-1` (next cell) to fill |
>
> **Expected result**: a blue hollow circle on the canvas.


In [ ]:
cv2.circle(img=blank_img, center=(400,400), radius=50, color=(255,0,0), thickness=-1)

In [ ]:
plt.imshow(blank_img)

> **🔵 Drawing a filled circle**
>
> `thickness=-1` fills the entire circle with the specified color.  
> Compare this with the outline circle in cell 82.
>
> **Expected result**: a solid blue filled circle in the bottom-right area of the canvas.


In [ ]:
cv2.line(blank_img, pt1=(0,0), pt2=(512,512), color=(102,255,255), thickness=5)

In [ ]:
plt.imshow(blank_img)

> **📏 Drawing a line**
>
> `cv2.line(img, pt1, pt2, color, thickness)` draws a straight line from `pt1` to `pt2`.  
> Here `(0,0) → (512,512)` draws a diagonal from the top-left to the bottom-right corner.
>
> **Expected result**: a cyan diagonal line across the canvas.


In [ ]:
font = cv2.FONT_HERSHEY_SIMPLEX
cv2.putText(blank_img, text='Hello', org=(10,500), fontFace=font, 
            fontScale=4, color=(255,255,255), thickness=3, lineType=cv2.LINE_AA)
plt.imshow(blank_img)

> **🔤 Drawing text on an image**
>
> `cv2.putText(img, text, org, fontFace, fontScale, color, thickness, lineType)`:
>
> | Parameter | Meaning |
> |---|---|
> | `org` | Bottom-left corner of the text string `(x, y)` |
> | `fontFace` | Font family (e.g. `cv2.FONT_HERSHEY_SIMPLEX`) |
> | `fontScale` | Font size multiplier |
> | `lineType=cv2.LINE_AA` | Anti-aliased edges — smoother text |
>
> **Expected result**: white text "Hello" at the bottom of the canvas.


In [ ]:
blank_img = np.zeros(shape=(512,512,3), dtype=np.int32)

In [ ]:
plt.imshow(blank_img)

In [ ]:
vertices = np.array([ [100,300], [200,200], [400,300], [200,400] ], dtype=np.int32)

In [ ]:
pts = vertices.reshape((-1,1,2))

In [ ]:
pts

In [ ]:
cv2.polylines(blank_img, [pts], isClosed=True, color=(255,0,0), thickness=5)
plt.imshow(blank_img)

> **🔷 Drawing polygons**
>
> `cv2.polylines(img, [pts], isClosed, color, thickness)` connects a sequence of points with line segments.
>
> The vertices array must be reshaped to `(-1, 1, 2)` — OpenCV requires this specific shape for contour/polygon functions.
>
> | Parameter | Meaning |
> |---|---|
> | `isClosed=True` | Connects the last point back to the first |
> | `isClosed=False` | Leaves the polygon open |
>
> **Expected result**: a blue quadrilateral (diamond shape) drawn on the canvas.


#### Connecting callback functions

OpenCV windows can trigger event callbacks (especially mouse events). A common pattern is:

1) create a window
2) register a callback with `cv2.setMouseCallback(...)`
3) run a loop that displays the image and listens for keys


This callback reacts to three mouse events:

- `cv2.EVENT_LBUTTONDOWN` (left button pressed)
  - start drawing and store the initial point `(ix, iy)`
- `cv2.EVENT_MOUSEMOVE` (mouse moves)
  - while drawing, update the preview rectangle from `(ix, iy)` to `(x, y)`
- `cv2.EVENT_LBUTTONUP` (left button released)
  - stop drawing and finalize the rectangle

Key variables:

- `ix, iy` initial click position
- `drawing` boolean flag while dragging
- colors are BGR in OpenCV (e.g. `(0, 255, 0)` is green)


In [ ]:
import cv2
import numpy as np

drawing = False
ix, iy = -1, -1


def draw_rectangle(event, x, y, flags, param):
    
    global ix, iy, drawing
    
    if event == cv2.EVENT_LBUTTONDOWN:
        
        drawing = True
        ix,iy = x,y
        
    elif event == cv2.EVENT_MOUSEMOVE:
        if drawing == True:
            cv2.rectangle(img, (ix, iy), (x,y), (0,255,0), -1)
            
    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False
        cv2.rectangle(img, (ix, iy), (x,y), (0,255,0), -1)
            

# SHOWING THE IMAGE
img = np.zeros((512, 512, 3))

cv2.namedWindow(winname='my_drawing')

cv2.setMouseCallback('my_drawing', draw_rectangle)

while True:
    
    cv2.imshow('my_drawing', img)
    
    # Press q to quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
        
cv2.destroyAllWindows()

> **🖱️ Interactive drawing with mouse callbacks (draw_rectangle)**
>
> This pattern requires a real OpenCV window (not available in Jupyter — run it as a standalone Python script).
>
> **How it works:**
> 1. `cv2.namedWindow(...)` creates a window.
> 2. `cv2.setMouseCallback(window, function)` registers `draw_rectangle` to be called on mouse events.
> 3. The callback receives `(event, x, y, flags, param)` on every mouse action.
>
> The three events handled:
> - `EVENT_LBUTTONDOWN` → start drawing, record start point `(ix, iy)`
> - `EVENT_MOUSEMOVE` → while dragging, update the rectangle preview
> - `EVENT_LBUTTONUP` → stop drawing, finalize the rectangle
>
> `drawing = True/False` is a flag that tracks whether the user is currently dragging.


The next callback is similar, but uses different mouse buttons to draw circles in different colors.


In [ ]:
import cv2
import numpy as np


def draw_circle(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        cv2.circle(img, (x,y), 100, (0,255,0), -1)
    elif event == cv2.EVENT_RBUTTONDOWN:
        cv2.circle(img, (x,y), 100, (255,0,0), -1)

cv2.namedWindow(winname='my_drawing')

cv2.setMouseCallback('my_drawing', draw_circle)


img = np.zeros((512,512,3))

while True:
    cv2.imshow('my_drawing', img)
    
    if cv2.waitKey(20) & 0xFF == 27:
        break
        
cv2.destroyAllWindows()

> **🖱️ Mouse callback — draw circles**
>
> Similar to the rectangle callback but simpler:
> - **Left click** → draw a green filled circle
> - **Right click** → draw a blue filled circle
>
> `cv2.waitKey(20) & 0xFF == 27` waits 20 ms per frame for a key press; `27` is the ASCII code for the **Escape** key, which exits the loop.  
> `cv2.destroyAllWindows()` closes all OpenCV windows cleanly.
>
> ⚠️ Run this as a standalone `.py` script — it will not work inside Jupyter without a display server.


## Image processing with OpenCV

This section covers the classic building blocks you will reuse in most CV pipelines.


### Color spaces

RGB is common for display, but other color spaces can make tasks easier. Two popular choices:

- HSL: hue, saturation, lightness
- HSV: hue, saturation, value

Why convert?

- separating brightness from color can simplify thresholding/segmentation
- some tasks are more stable in HSV/HSL than raw RGB

In OpenCV you typically use `cv2.cvtColor(img, ...)` (remember the input is usually BGR).


In [ ]:
import cv2
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
img = cv2.imread('data/00-puppy.jpg')

In [ ]:
plt.imshow(img)

In [ ]:
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
plt.imshow(img)

> **🎨 Converting BGR → RGB for correct Matplotlib display**
>
> `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` reorders the channels from `[B, G, R]` to `[R, G, B]`.  
> Without this conversion, reds and blues appear swapped in Matplotlib.
>
> **Expected result**: the puppy photo with natural colors.


In [ ]:
img = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
plt.imshow(img)

> **🌈 HSV color space**
>
> HSV stands for **Hue, Saturation, Value**:
>
> | Channel | Meaning | Range (OpenCV) |
> |---|---|---|
> | **H** — Hue | The "color" (red, green, blue…) | `[0, 179]` |
> | **S** — Saturation | Color purity/intensity | `[0, 255]` |
> | **V** — Value | Brightness | `[0, 255]` |
>
> **Why use HSV?**  
> In RGB, "green" is a combination of all three values. In HSV, green is just a range of Hue values.  
> This makes it much easier to segment objects by color using a simple `cv2.inRange(...)` threshold.
>
> **Expected result**: the image looks strange/psychedelic — that's expected! Matplotlib interprets the HSV values as RGB, which is why the colors look distorted. We're just inspecting the raw channel data here.


In [ ]:
img = cv2.imread('data/00-puppy.jpg')
img = cv2.cvtColor(img, cv2.COLOR_BGR2HLS)
plt.imshow(img)

> **💡 HLS color space**
>
> HLS stands for **Hue, Lightness, Saturation** — similar to HSV but uses *lightness* instead of *value*.
>
> Both HSV and HLS separate brightness from color information, but they define "brightness" slightly differently:
> - **HSV Value**: the maximum of R, G, B
> - **HLS Lightness**: the average of the max and min of R, G, B
>
> In practice, HSV is more commonly used for color-based segmentation tasks in OpenCV.


In [ ]:
img = cv2.imread('data/00-puppy.jpg')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)

H, S, V = cv2.split(hsv)
L, A, B = cv2.split(lab)

show(img_rgb, title='RGB')
show(H, title='HSV: H channel')
show(S, title='HSV: S channel')
show(V, title='HSV: V channel')
show(L, title='LAB: L channel')

### Blending and masking

Blending combines two images of the same size:

`new_pixel = a * pixel_1 + b * pixel_2 + y`

In OpenCV, `cv2.addWeighted(...)` implements this efficiently.

Masking selects which pixels to keep:

- white/1 means keep
- black/0 means ignore

Masks are the basic tool behind many segmentation and compositing operations.


In [ ]:
import cv2
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
img1 = cv2.imread('data/dog_backpack.jpg')
img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
img2 = cv2.imread('data/watermark_no_copy.png')
img2 = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)

In [ ]:
plt.imshow(img1)

In [ ]:
plt.imshow(img2)

In [ ]:
img1.shape

In [ ]:
img2.shape

In [ ]:
# BLENDING IMAGE OF THE SAME SIZE

In [ ]:
img1 = cv2.resize(img1, (1200, 1200))
img2 = cv2.resize(img2, (1200, 1200))

In [ ]:
blended = cv2.addWeighted(src1=img1, alpha = 0.8, src2=img2, beta = 0.1, gamma=0)

> **🔀 Blending two images — `cv2.addWeighted`**
>
> `cv2.addWeighted(src1, alpha, src2, beta, gamma)` blends two images with the formula:
>
> ```
> output = alpha × src1 + beta × src2 + gamma
> ```
>
> | Parameter | Value used | Effect |
> |---|---|---|
> | `alpha=0.8` | 80% of img1 | img1 is dominant |
> | `beta=0.1` | 10% of img2 | img2 is subtle (watermark-like) |
> | `gamma=0` | No brightness shift | |
>
> ⚠️ Both images must be the **same size** — that's why cell 118 resizes them both to `1200×1200` first.
>
> **Expected result**: the dog image with a faint watermark overlaid on top.


In [ ]:
plt.imshow(blended)

In [ ]:
# OVERLAY SMALL IMAGE ON TOP OF A LARGE IMAGE ( NO BLENDING ) using numpy

In [ ]:
img1 = cv2.imread('data/dog_backpack.jpg')
img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
img2 = cv2.imread('data/watermark_no_copy.png')
img2 = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)

In [ ]:
img2 = cv2.resize(img2, (600,600))

In [ ]:
large_img = img1
small_img = img2

In [ ]:
# TOP LEFT CORNER
x_offset = 0
y_offset = 0

In [ ]:
x_end = x_offset + small_img.shape[1]
y_end = y_offset + small_img.shape[0]

In [ ]:
large_img[y_offset:y_end, x_offset:x_end] = small_img

In [ ]:
plt.imshow(large_img)

> **📌 Overlaying a small image on a large image (NumPy slicing)**
>
> Instead of blending, here we **directly replace** a region of the large image with the small image:
> ```python
> large_img[y_offset:y_end, x_offset:x_end] = small_img
> ```
>
> This is a hard paste — no transparency, no blending.  
> The watermark fully replaces the top-left corner of the dog image.
>
> **Expected result**: the small watermark image pasted sharply in the top-left corner of the dog photo.


In [ ]:
# BLEND DIFFERENT SIZE IMAGES

In [ ]:
img1 = cv2.imread('data/dog_backpack.jpg')
img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
img2 = cv2.imread('data/watermark_no_copy.png')
img2 = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)

In [ ]:
img2 = cv2.resize(img2, (600,600))

In [ ]:
x_offset = img1.shape[1] - img2.shape[1]
y_offset = img1.shape[0] - img2.shape[0]

In [ ]:
rows, cols, channels = img2.shape

In [ ]:
roi = img1[y_offset:img1.shape[0], x_offset:img1.shape[1]]

In [ ]:
plt.imshow(roi)

In [ ]:
img2gray = cv2.cvtColor(img2, cv2.COLOR_RGB2GRAY)

In [ ]:
plt.imshow(img2gray, cmap='gray')

In [ ]:
# Create a binary mask of the watermark
# new_pixel = 255 - original_pixel
mask_inv = cv2.bitwise_not(img2gray)

In [ ]:
plt.imshow(mask_inv, cmap='gray')

> **🔲 Creating a binary mask with bitwise NOT**
>
> `cv2.bitwise_not(img2gray)` inverts all pixel values: `new_pixel = 255 - original_pixel`.
>
> - Where the watermark text was **black** (0) → becomes **white** (255)
> - Where the background was **white** (255) → becomes **black** (0)
>
> This inverted mask (`mask_inv`) is used in subsequent cells to:
> 1. Extract just the foreground (watermark text) from the watermark image
> 2. Cut a matching hole in the background image
>
> **Expected result**: the mask with white text on a black background.


In [ ]:
import numpy as np

In [ ]:
white_background = np.full(img2.shape, 255, dtype=np.uint8)

In [ ]:
white_background.shape

In [ ]:
# this will put the white background where the watermark is not, and black where the watermark is
# copies pixels only where mask allows
bk = cv2.bitwise_or(src1=white_background, src2=white_background, mask=mask_inv)

# The mask is a single-channel 8-bit image (grayscale), where:

# Pixels = 0 → operation ignored, output stays black (0)

# Pixels ≠ 0 → operation applied, output gets src1 OR src2

# So in this line:

    # Only where mask_inv is white (255) will pixels from white_background appear in bk

    # Where mask_inv is black (0) → pixels in bk stay 0

In [ ]:
plt.imshow(bk)

> **🔧 Using `cv2.bitwise_or` with a mask**
>
> `cv2.bitwise_or(src1, src2, mask=mask_inv)` performs a bitwise OR between `src1` and `src2`, but **only where the mask is non-zero (white)**.
>
> - Where `mask_inv = 255` → the OR operation is applied → `white_background` pixels pass through to `bk`
> - Where `mask_inv = 0` → the output stays black (0)
>
> **Result `bk`**: a white image with black holes exactly where the watermark text is.  
> This will later be combined with the dog photo region to create a transparent-looking watermark effect.


In [ ]:
mask_inv

In [ ]:
fg = cv2.bitwise_or(img2, img2, mask=mask_inv)

In [ ]:
plt.imshow(fg)

In [ ]:
final_roi = cv2.bitwise_or(roi, fg)

In [ ]:
plt.imshow(final_roi)

In [ ]:
large_img = img1
small_img = final_roi

In [ ]:
large_img[y_offset:y_offset+small_img.shape[0], x_offset:x_offset+small_img.shape[1]] = small_img

In [ ]:
plt.imshow(large_img)

### When to threshold

Thresholding works best when the foreground/background have different intensity or color. If lighting varies across the image, adaptive thresholding or converting to another color space (like HSV) often helps.


### Thresholding, blurring, and smoothing

- Thresholding turns an image into a simpler (often binary) image to separate foreground/background.
- Smoothing/blur reduces noise and small details (often helpful before edge detection).
- Gamma correction can make an image appear brighter/darker in a non-linear way.

Kernels (small matrices) are used for many filters (blur, sharpen, edge detect).

Interactive kernel visualizer: http://setosa.io/ev/image-kernels/


In [ ]:
import cv2
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
img = cv2.imread('data/rainbow.jpg', 0)

In [ ]:
plt.imshow(img, cmap='gray')

In [ ]:
# ret is the cutoff value (127),  thresh1 is the image

# How does the thresholding work?

# if pixel_value > 127:
#     pixel_value = 255  # white
# else:
#     pixel_value = 0    # black

ret, thresh1 = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)

In [ ]:
plt.imshow(thresh1, cmap='gray')

> **🔳 Global thresholding**
>
> `cv2.threshold(img, thresh, maxVal, type)` converts a grayscale image to a **binary** image:
>
> ```
> if pixel > 127:  pixel = 255 (white)
> else:             pixel = 0   (black)
> ```
>
> The function returns two values:
> - `ret` — the threshold value used (127 here)
> - `thresh1` — the resulting binary image
>
> **When to use**: works well when the foreground and background have clearly different intensities across the whole image.  
> **Limitation**: a single global threshold fails when lighting is uneven.
>
> **Expected result**: a high-contrast black-and-white version of the rainbow image.


In [ ]:
img = cv2.imread('data/crossword.jpg', 0)

In [ ]:
plt.imshow(img, cmap='gray')

In [ ]:
def show_pic(img):
    fig = plt.figure(figsize=(15,15))
    ax = fig.add_subplot(111)
    ax.imshow(img, cmap='gray')

In [ ]:
show_pic(img)

In [ ]:
ret, th1 = cv2.threshold(img, 180, 255, cv2.THRESH_BINARY)
show_pic(th1)

In [ ]:
# Adaptive thresholding
# how does it work ?

# threshold = mean_of_neighbors - C (8 in our case)
# if pixel_value > threshold:
#     output = maxValue (255) # white
# else:
#     output = 0         # black

th2 = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 11, 8)
show_pic(th2)

> **🔳 Adaptive thresholding**
>
> Unlike global thresholding which uses one cutoff for the whole image, adaptive thresholding **computes a local threshold for each pixel** based on its neighborhood:
>
> ```
> local_threshold = mean(neighbors in 11×11 window) - C
> if pixel > local_threshold: white else: black
> ```
>
> **Parameters explained:**
> - `blockSize=11` — size of the neighborhood window (must be odd)
> - `C=8` — a constant subtracted from the mean (fine-tunes sensitivity)
>
> **Why it works better on uneven images**: even if one part of the image is brighter than another (e.g., a shadow), the local threshold adapts accordingly.
>
> **Expected result**: the crossword text is clearly separated from the background even where lighting varies.


Adaptive thresholding uses a local neighborhood instead of one global cutoff.

Example with a `11x11` neighborhood (`blockSize=11`):

- compute the mean of the local window around a pixel
- subtract `C` (e.g. `8`) to form a local threshold
- apply the binary rule (above -> white, below -> black)


In [ ]:
blended = cv2.addWeighted(src1=th1, alpha=0.6, src2=th2, beta=0.4, gamma=0)
show_pic(blended)

> **🔀 Combining threshold results**
>
> Here we blend the global threshold (`th1`) and adaptive threshold (`th2`) outputs using `cv2.addWeighted`.  
> This is not a standard processing step — it's a visualization technique to see both results side by side in a single image.
>
> **Expected result**: a mix showing both threshold approaches simultaneously — useful for comparison.


In [ ]:
# IAM HERE

# Finir cours
# Finir video utube / voir si ajouter des choses
# project

# machine learning from scratch
# explain theory of machine learning
# simple models from scratch
# project

# explain theory of deep learning
# explain theory of convolutional neural networks
# Convolutional Neural Networks from scratch
# projects

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def load_img():
    img = cv2.imread('data/bricks.jpg').astype(np.float32) / 255
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

In [ ]:
def display_img(img):
    fig = plt.figure(figsize=(12,10))
    ax = fig.add_subplot(111)
    ax.imshow(img)


In [ ]:
i = load_img()
display_img(i)

In [ ]:
# Values below 1 make it brigher, values greater make darker
gamma = 1/4

In [ ]:
result = np.power(i, gamma)

In [ ]:
display_img(result)

> **☀️ Gamma correction**
>
> Gamma correction adjusts the overall brightness of an image using a **power law**:
> ```
> output = pixel_value ^ gamma
> ```
> (applied on a normalized `[0.0, 1.0]` image)
>
> | Gamma | Effect |
> |---|---|
> | `< 1` (e.g. `1/4 = 0.25`) | Brightens the image |
> | `= 1` | No change |
> | `> 1` | Darkens the image |
>
> `np.power(i, gamma)` applies the power function element-wise to every pixel.
>
> **Why it matters**: human vision is non-linear. Gamma correction can compensate for monitor display curves and improve perceived contrast.
>
> **Expected result**: the brick image should look noticeably brighter.


In [ ]:
# LOW PASS FILTER WITH 2D CONVOLUTION

In [ ]:
img = load_img()
font = cv2.FONT_HERSHEY_COMPLEX
cv2.putText(img, text='bricks', org=(10,600), fontFace=font, fontScale=10, color=(255,0,0), thickness=4)

In [ ]:
display_img(img)

In [ ]:
kernel = np.ones(shape=(5,5), dtype=np.float32)/25

In [ ]:
kernel

In [ ]:
dst = cv2.filter2D(img, -1, kernel)
display_img(dst)

> **🔲 2D Convolution / Low-pass filter**
>
> `cv2.filter2D(img, -1, kernel)` convolves the image with a custom kernel.
>
> The kernel used here is a **5×5 box filter** (all values = `1/25`):
> ```
> kernel = np.ones((5,5)) / 25
> ```
> This computes the **average of each pixel and its 24 neighbors** — a simple blur.
>
> **Why blur?** Blurring is a *low-pass filter*: it smooths out high-frequency details (noise, sharp edges).  
> It is often the first step before edge detection or thresholding to reduce noise sensitivity.
>
> **Expected result**: the "bricks" image with text appears softer/blurrier.


In [ ]:
img = load_img()
font = cv2.FONT_HERSHEY_COMPLEX
cv2.putText(img, text='bricks', org=(10,600), fontFace=font, fontScale=10, color=(255,0,0), thickness=4)
print('reset')

In [ ]:
blurred = cv2.blur(img, ksize=(10,10))
display_img(blurred)

> **📦 Simple mean blur — `cv2.blur`**
>
> `cv2.blur(img, ksize=(10, 10))` is a convenient wrapper for the box filter (same idea as cell 181–183 but with a larger `10×10` kernel).  
> Larger kernel → stronger blur.
>
> **Expected result**: a heavily blurred version of the brick image.


In [ ]:
img = load_img()
font = cv2.FONT_HERSHEY_COMPLEX
cv2.putText(img, text='bricks', org=(10,600), fontFace=font, fontScale=10, color=(255,0,0), thickness=4)
print('reset')

In [ ]:
# Gaussian Blur
blurred_img = cv2.GaussianBlur(img, (5,5), 10)
display_img(blurred_img)

> **🌫️ Gaussian blur**
>
> `cv2.GaussianBlur(img, (5,5), sigmaX=10)` blurs using a **Gaussian (bell-shaped) kernel**.
>
> Unlike the box filter (where all neighbors contribute equally), the Gaussian filter gives **more weight to closer pixels** and less weight to distant ones.  
> This produces a smoother, more natural-looking blur.
>
> **Parameters:**
> - `(5, 5)` — kernel size (must be odd)
> - `10` — standard deviation (σ); larger σ = more blur
>
> **When to use**: Gaussian blur is the standard choice before edge detection (Canny, Sobel) because it removes noise without introducing artificial sharp artifacts.


In [ ]:
img = load_img()
font = cv2.FONT_HERSHEY_COMPLEX
cv2.putText(img, text='bricks', org=(10,600), fontFace=font, fontScale=10, color=(255,0,0), thickness=4)
print('reset')

In [ ]:
# Median Blur
# Median Blur is great for removing noise while retaining shape
median = cv2.medianBlur(img, 5)
display_img(median)

> **🔢 Median blur**
>
> `cv2.medianBlur(img, 5)` replaces each pixel with the **median value** of its 5×5 neighborhood.
>
> **Key advantage**: unlike mean/Gaussian blur, median blur is very effective at removing **salt-and-pepper noise** (random bright/dark isolated pixels) while preserving edges better.
>
> **Why?** Extreme noise values (like a single white pixel in a dark area) don't survive the median — they get replaced by the surrounding majority value.
>
> **Expected result**: cells 194 demonstrate this clearly — the noisy dog image is cleaned up while keeping sharp edges.


In [ ]:
img = cv2.imread('data/sammy.jpg')
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

In [ ]:
display_img(img)

In [ ]:
noise_img = cv2.imread('data/sammy_noise.jpg')

In [ ]:
display_img(noise_img)

In [ ]:
median = cv2.medianBlur(noise_img, 5)
display_img(median)

In [ ]:
# Bilateral Filtering
img = load_img()
font = cv2.FONT_HERSHEY_COMPLEX
cv2.putText(img, text='bricks', org=(10,600), fontFace=font, fontScale=10, color=(255,0,0), thickness=4)
print('reset')

In [ ]:
blur = cv2.bilateralFilter(img, 9, 75, 75)
display_img(blur)

> **🎨 Bilateral filter — edge-preserving blur**
>
> `cv2.bilateralFilter(img, d=9, sigmaColor=75, sigmaSpace=75)` blurs the image while **preserving edges**.
>
> It considers two factors for each neighbor pixel:
> - **Spatial distance** (`sigmaSpace`): how far away the neighbor is (like Gaussian blur)
> - **Intensity difference** (`sigmaColor`): how different the neighbor's color/intensity is
>
> Pixels across a strong edge have very different intensities → low weight → edge is preserved.
>
> **When to use**: ideal for skin smoothing, cartoon effects, or any case where you want to blur flat regions but keep sharp boundaries.
>
> **Expected result**: the bricks texture is smoothed but the "bricks" text edges remain sharp.


### Choosing a kernel

The kernel/structuring element controls what morphology removes or preserves. Common choices:

- small (3x3) kernels remove tiny noise
- larger kernels remove/bridge larger structures
- shapes (rect/ellipse) change how corners and thin lines are affected


### Morphological operators

Morphology uses a small binary pattern (a **kernel** / structuring element) to modify shapes in an image (usually a binary mask).

Common uses:

- remove small noise
- fill small gaps
- separate touching objects
- clean up segmentation masks

Core operations:

- erosion: shrinks foreground
- dilation: expands foreground
- opening: erosion then dilation (remove small foreground noise)
- closing: dilation then erosion (fill small holes)

Example:

```python
opening = cv2.morphologyEx(noise_img, cv2.MORPH_OPEN, kernel)
```

Gradient-based operators (e.g. Sobel) estimate changes in intensity and are often used for edge-like responses.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def load_img():
    blank_img = np.zeros((600,600))
    font = cv2.FONT_HERSHEY_SIMPLEX
    cv2.putText(blank_img, text='ABCDE', org=(50,300), fontFace=font, fontScale=5, color=(255,255,255), thickness = 25)
    return blank_img

In [ ]:
def display_img(img):
    fig = plt.figure(figsize=(12,10))
    ax = fig.add_subplot(111)
    ax.imshow(img, cmap='gray')

In [ ]:
img = load_img()
display_img(img)

In [ ]:
kernel = np.ones((5,5), dtype=np.uint8)

In [ ]:
result = cv2.erode(img, kernel, iterations=1)

In [ ]:
display_img(result)

> **🔴 Morphological Erosion**
>
> `cv2.erode(img, kernel, iterations=1)` **shrinks** white (foreground) regions by eroding their borders.
>
> **How it works**: for each pixel, look at its neighborhood defined by the kernel. If *all* neighboring pixels are white → keep it white. If *any* is black → set it to black.
>
> - Thin strokes become thinner
> - Small white regions may disappear entirely
>
> **Expected result**: the "ABCDE" text on the black canvas should appear slightly thinner.


In [ ]:
img = load_img()
display_img(img)

In [ ]:
result = cv2.erode(img, kernel, iterations=4)

In [ ]:
display_img(result)

> **🔴 Erosion with 4 iterations**
>
> Applying erosion 4 times exaggerates the effect.  
> With `iterations=4`, the operation is applied repeatedly → borders erode further with each pass.
>
> **Expected result**: the text is noticeably thinner; thin strokes may have partially disappeared.


In [ ]:
img = load_img()
display_img(img)

In [ ]:
white_noise = np.random.randint(low=0, high=2, size=(600,600))

In [ ]:
display_img(white_noise)

In [ ]:
white_noise = white_noise * 255

In [ ]:
noise_img = white_noise + img

In [ ]:
display_img(noise_img)

In [ ]:
opening = cv2.morphologyEx(noise_img, cv2.MORPH_OPEN, kernel)

In [ ]:
display_img(opening)

> **🧹 Morphological Opening — removing noise**
>
> `cv2.morphologyEx(noise_img, cv2.MORPH_OPEN, kernel)` performs **opening** = erosion followed by dilation.
>
> **Effect**: small isolated white noise pixels are removed, while larger structures (the text) are preserved.
>
> In cell 211–215, random binary noise was added to the image (white random pixels scattered everywhere).  
> Opening eliminates those small noise blobs because:
> - Erosion removes them first (they're too small to survive)
> - Dilation restores the original size of the larger text letters
>
> **Expected result**: the "ABCDE" text is restored cleanly with the noise removed.


In [ ]:
img = load_img()

In [ ]:
black_noise = np.random.randint(low=0, high=2, size=(600,600))

In [ ]:
black_noise = black_noise * -255

In [ ]:
black_noise_img = img+black_noise

In [ ]:
black_noise_img[black_noise_img==-255] = 0

In [ ]:
display_img(black_noise_img)

In [ ]:
closing = cv2.morphologyEx(black_noise_img, cv2.MORPH_CLOSE, kernel)

In [ ]:
display_img(closing)

> **🔒 Morphological Closing — filling holes**
>
> `cv2.morphologyEx(black_noise_img, cv2.MORPH_CLOSE, kernel)` performs **closing** = dilation followed by erosion.
>
> **Effect**: small black holes/gaps inside white foreground regions are filled.
>
> In cells 219–223, black random pixels were subtracted from the white text (creating holes inside the letters).  
> Closing fills those small holes because:
> - Dilation expands white regions, covering the holes
> - Erosion shrinks back to the original size but the holes are now filled
>
> **Expected result**: the "ABCDE" text is restored with holes filled in.


In [ ]:
img = load_img()

In [ ]:
display_img(img)

In [ ]:
gradient = cv2.morphologyEx(img, cv2.MORPH_GRADIENT, kernel)

In [ ]:
display_img(gradient)

> **📐 Morphological Gradient — edge detection**
>
> `cv2.morphologyEx(img, cv2.MORPH_GRADIENT, kernel)` computes the difference between dilation and erosion:
>
> ```
> gradient = dilation(img) - erosion(img)
> ```
>
> This highlights the **boundaries** of objects — the result is essentially an outline of the shapes.
>
> **Expected result**: only the outer edges of the "ABCDE" letters are visible — a clean edge/outline effect.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def display_img(img):
    fig = plt.figure(figsize=(12,10))
    ax = fig.add_subplot(111)
    ax.imshow(img, cmap='gray')

In [ ]:
img = cv2.imread('data/sudoku.jpg', 0)

In [ ]:
display_img(img)

In [ ]:
sobelx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=5)

In [ ]:
display_img(sobelx)

In [ ]:
sobely = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=5)

In [ ]:
display_img(sobely)

In [ ]:
laplacian = cv2.Laplacian(img, cv2.CV_64F)

In [ ]:
display_img(laplacian)

> **📐 Gradient-based edge detection: Sobel & Laplacian**
>
> These filters detect edges by looking at how quickly pixel intensity changes.
>
> **Sobel filter** computes the gradient in one direction:
> - `cv2.Sobel(..., dx=1, dy=0)` → horizontal edges (detects vertical intensity changes)
> - `cv2.Sobel(..., dx=0, dy=1)` → vertical edges (detects horizontal intensity changes)
>
> **Laplacian** computes the second derivative — sensitive to edges in all directions at once, but also more sensitive to noise.
>
> `cv2.CV_64F` stores the output as 64-bit float to capture both positive and negative gradients (negative values would be lost with `uint8`).
>
> **Expected results:**
> - `sobelx`: edges with strong horizontal gradients (vertical lines highlighted)
> - `sobely`: edges with strong vertical gradients (horizontal lines highlighted)
> - `laplacian`: all edges, but potentially noisy


In [ ]:
blended = cv2.addWeighted(src1=sobelx, alpha=0.5, src2=sobely, beta=0.5, gamma=0)

In [ ]:
display_img(blended)

> **🔀 Combining Sobel X and Y**
>
> Blending `sobelx` and `sobely` with equal weights (`alpha=0.5, beta=0.5`) approximates the full gradient magnitude:
> ```
> approx_magnitude ≈ 0.5 × |Gx| + 0.5 × |Gy|
> ```
> (A more accurate formula is `sqrt(Gx² + Gy²)`, but this blended version is a quick visual approximation.)
>
> **Expected result**: edges from both horizontal and vertical directions are visible together — a more complete edge map of the sudoku grid.


In [ ]:
ret, th1 = cv2.threshold(img, 100, 255, cv2.THRESH_BINARY)

In [ ]:
display_img(th1)

In [ ]:
kernel = np.ones((4,4), np.uint8)

In [ ]:
gradient = cv2.morphologyEx(blended, cv2.MORPH_GRADIENT, kernel)

In [ ]:
display_img(gradient)

### Histograms

A histogram shows how frequently values occur. For images, you can plot distributions per channel (0-255 for `uint8`).

Histogram equalization is a contrast-enhancement technique based on redistributing intensities using the histogram.


A histogram counts how many pixels have each intensity value.
- Dark images: histogram concentrated near 0
- Bright images: histogram concentrated near 255
- Low contrast: histogram narrow
- High contrast: histogram spread


In [ ]:
def plot_gray_hist(gray_img, title='Histogram'):
    hist = cv2.calcHist([gray_img], [0], None, [256], [0, 256]).ravel()
    plt.figure(figsize=(6, 3))
    plt.title(title)
    plt.plot(hist)
    plt.xlim(0, 255)
    plt.xlabel('intensity')
    plt.ylabel('count')
    plt.show()

plot_gray_hist(gray, title='Grayscale histogram')

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
dark_horse = cv2.imread('data/horse.jpg')
show_horse = cv2.cvtColor(dark_horse, cv2.COLOR_BGR2RGB)

rainbow = cv2.imread('data/rainbow.jpg')
show_rainbow = cv2.cvtColor(rainbow, cv2.COLOR_BGR2RGB)

blue_bricks = cv2.imread('data/bricks.jpg')
show_bricks = cv2.cvtColor(blue_bricks, cv2.COLOR_BGR2RGB)

In [ ]:
plt.imshow(show_bricks)

In [ ]:
hist_values = cv2.calcHist([blue_bricks], channels=[0], mask=None, histSize=[256], ranges=[0,256])

In [ ]:
hist_values.shape

In [ ]:
# HIST FOR BLUE COLORS
plt.plot(hist_values)

> **📊 Computing and plotting a histogram**
>
> `cv2.calcHist([img], channels, mask, histSize, ranges)` counts how many pixels have each intensity value:
>
> | Parameter | Meaning |
> |---|---|
> | `channels=[0]` | Which channel to analyze (0=Blue in BGR) |
> | `mask=None` | Analyze the full image (no mask) |
> | `histSize=[256]` | Number of bins (one per intensity value 0–255) |
> | `ranges=[0, 256]` | Intensity range to consider |
>
> **Expected result**: a line plot showing the distribution of blue-channel intensities.  
> The blue bricks image should show a peak toward the higher (brighter) blue values.


In [ ]:
hist_values = cv2.calcHist([dark_horse], channels=[0], mask=None, histSize=[256], ranges=[0,256])

In [ ]:
# HIST FOR BLUE COLORS
plt.plot(hist_values)

In [ ]:
img = blue_bricks

In [ ]:
color = ('b','g','r')

for i,col in enumerate(color):
    histr = cv2.calcHist([img], [i], None, [256], [0,256])
    plt.plot(histr, color=col)
    plt.xlim([0,256])
    
plt.title('HISTOGRAM FOR BLUE BRICKS')

> **🌈 Per-channel color histogram**
>
> By looping over all 3 channels and plotting each in its respective color (blue, green, red), you get a full picture of the image's color distribution.
>
> **How to read the plot:**
> - X-axis: intensity value (0 = dark, 255 = bright)
> - Y-axis: number of pixels with that intensity
> - A peak at 200+ in the blue channel confirms the image has many bright blue pixels
>
> The "dark horse" histogram should show most values clustered near 0 (very dark image).


In [ ]:
img = dark_horse
color = ('b','g','r')

for i,col in enumerate(color):
    histr = cv2.calcHist([img], [i], None, [256], [0,256])
    plt.plot(histr, color=col)
    plt.xlim([0, 50])
    plt.ylim([0,500000])
    
plt.title('HISTOGRAM FOR DARK HORSE')

In [ ]:
rainbow = cv2.imread('data/rainbow.jpg')
show_rainbow = cv2.cvtColor(rainbow, cv2.COLOR_BGR2RGB)

In [ ]:
img = rainbow
img.shape

In [ ]:
mask = np.zeros(img.shape[:2], np.uint8)
mask.shape

In [ ]:
plt.imshow(mask, cmap='gray')

In [ ]:
mask[300:400, 100:400] = 255

In [ ]:
plt.imshow(mask, cmap='gray')

In [ ]:
# FOR THE ACTUAL CALCULATION
masked_img = cv2.bitwise_and(img, img, mask=mask)

In [ ]:
# SHOW FOR VISUALIZATION
show_masked_img = cv2.bitwise_and(show_rainbow, show_rainbow, mask=mask)

In [ ]:
plt.imshow(show_masked_img)

> **🎭 Masked region selection**
>
> A **mask** is a grayscale image of the same size as the original where:
> - `255` (white) = include this pixel
> - `0` (black) = exclude this pixel
>
> `cv2.bitwise_and(img, img, mask=mask)` zeroes out all pixels where the mask is black, keeping only the masked region visible.
>
> In cells 265–268, a white rectangle was drawn on a black mask at `[300:400, 100:400]`.  
> **Expected result**: only the masked rectangular band of the rainbow is visible, the rest is black.


In [ ]:
# B G R
hist_mask_values_red = cv2.calcHist([rainbow], channels=[2], mask=mask, histSize =[256], ranges=[0,256])

In [ ]:
hist_values_red = cv2.calcHist([rainbow], channels=[2], mask=None, histSize =[256], ranges=[0,256])

In [ ]:
plt.plot(hist_mask_values_red)
plt.title('RED HISTOGRAM FOR MASKED RAINBOW')

In [ ]:
plt.plot(hist_values_red)
plt.title('RED HISTOGRAM FOR FULL RAINBOW')

In [ ]:
gorilla = cv2.imread('data/gorilla.jpg', 0)

In [ ]:
def display_img(img):
    fig = plt.figure(figsize=(12,10))
    ax = fig.add_subplot(111)
    ax.imshow(img, cmap='gray')

In [ ]:
display_img(gorilla)

In [ ]:
gorilla.shape

In [ ]:
hist_values = cv2.calcHist([gorilla], channels=[0], mask=None, histSize=[256], ranges=[0,256])

In [ ]:
plt.plot(hist_values)

In [ ]:
eq_gorilla = cv2.equalizeHist(gorilla)

In [ ]:
display_img(eq_gorilla)

> **📊 Histogram Equalization**
>
> `cv2.equalizeHist(gray_img)` redistributes pixel intensities so that the histogram becomes more uniform across the full `[0, 255]` range.
>
> **Effect**: increases contrast in dark or low-contrast images by stretching out the intensity distribution.
>
> **How it works**: it uses the cumulative distribution function (CDF) of the histogram to remap pixel values.
>
> **Expected result**: the gorilla image (which was likely dark/low-contrast) should appear with more visible detail and higher contrast after equalization.


In [ ]:
hist_values = cv2.calcHist([eq_gorilla], channels=[0], mask=None, histSize=[256], ranges=[0,256])

In [ ]:
plt.plot(hist_values)

In [ ]:
color_gorilla = cv2.imread('data/gorilla.jpg')

In [ ]:
show_gorilla = cv2.cvtColor(color_gorilla, cv2.COLOR_BGR2RGB)

In [ ]:
display_img(show_gorilla)

In [ ]:
hsv = cv2.cvtColor(color_gorilla, cv2.COLOR_BGR2HSV)

In [ ]:
hsv[:,:,2] = cv2.equalizeHist(hsv[:,:,2])

In [ ]:
eq_color_gorilla = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

In [ ]:
display_img(eq_color_gorilla)

> **🎨 Histogram equalization on color images**
>
> Directly applying `equalizeHist` to all 3 RGB channels distorts colors (each channel is equalized independently).
>
> The correct approach for color images:
> 1. Convert BGR → HSV
> 2. Equalize **only the V (Value/brightness) channel**: `cv2.equalizeHist(hsv[:,:,2])`
> 3. Convert back HSV → RGB
>
> This preserves the original hue and saturation (the "actual color") while only boosting brightness.
>
> **Expected result**: the gorilla image appears brighter and more detailed without unnatural color shifts.


## Contrast Enhancement: Histogram Equalization vs CLAHE

- Histogram equalization spreads intensities globally (can over-amplify noise)
- CLAHE spreads intensities locally with clipping (often better in practice)


In [ ]:
eq = cv2.equalizeHist(gray)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)

show(gray, title='Original gray')
show(eq, title='Equalized')
show(clahe, title='CLAHE')

plot_gray_hist(eq, title='Histogram after equalization')
plot_gray_hist(clahe, title='Histogram after CLAHE')

## Exercise 2 : Color Segmentation in HSV

Goal: create a mask that selects the yellow circle in the synthetic image (or anything yellow-ish in your own image).

Steps:
1) Convert RGB->BGR->HSV
2) Use `cv2.inRange(hsv, lower, upper)`
3) Visualize the mask and the segmented result

Tip: print pixel HSV values by clicking? (We won't use GUI). Instead, inspect a few pixels manually around the region or adjust ranges gradually.


In [ ]:
img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

# TODO: tune these ranges (H in [0,179] in OpenCV)
lower = np.array([20, 80, 80])
upper = np.array([40, 255, 255])
mask = cv2.inRange(hsv, lower, upper)

seg = cv2.bitwise_and(img_rgb, img_rgb, mask=mask)

show(mask, title='Mask')
show(seg, title='Segmented')
print('mask coverage:', mask.mean() / 255.0)

## Geometric Transforms

You will use these constantly in CV pipelines:
- Resize + preserve aspect ratio
- Translation, rotation, affine transform
- Perspective transform (homography) for document scanning / planar objects


In [ ]:
def resize_max_side(img, max_side=512):
    h, w = img.shape[:2]
    scale = max_side / max(h, w)
    if scale >= 1:
        return img.copy()
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))
    return cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

small = resize_max_side(img_rgb, 320)
show(small, title='Resized (keep aspect)')
print('old:', img_rgb.shape, 'new:', small.shape)

In [ ]:
def rotate_about_center(img, angle_deg, scale=1.0):
    h, w = img.shape[:2]
    center = (w / 2, h / 2)
    M = cv2.getRotationMatrix2D(center, angle_deg, scale)
    out = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    return out

rot = rotate_about_center(img_rgb, 25)
show(rot, title='Rotation (25 deg)')

## Perspective Transform (Practical Homography)

When an object is planar (paper, screen, book cover), you can map 4 corners to a rectangle.

We will demonstrate on a synthetic 'document' created by warping a rectangle into perspective, then unwarping it back.


In [ ]:
def draw_document(h=420, w=600):
    doc = np.full((h, w, 3), 245, dtype=np.uint8)
    cv2.rectangle(doc, (50, 60), (w - 50, h - 60), (30, 30, 30), 3)
    cv2.putText(doc, 'INVOICE #123', (80, 130), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (10, 10, 10), 2, cv2.LINE_AA)
    cv2.putText(doc, 'Total: $42.00', (80, 200), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (10, 10, 10), 2, cv2.LINE_AA)
    for i in range(6):
        y = 260 + i * 22
        cv2.line(doc, (80, y), (w - 80, y), (120, 120, 120), 1)
    return cv2.cvtColor(doc, cv2.COLOR_BGR2RGB)

doc_rgb = draw_document()
show(doc_rgb, title='Synthetic document (fronto-parallel)')

# Warp it into a perspective view
doc_bgr = cv2.cvtColor(doc_rgb, cv2.COLOR_RGB2BGR)
h, w = doc_bgr.shape[:2]
src = np.float32([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]])
dst = np.float32([[80, 40], [w - 120, 70], [w - 40, h - 60], [60, h - 20]])
H = cv2.getPerspectiveTransform(src, dst)
warped = cv2.warpPerspective(doc_bgr, H, (w, h), borderMode=cv2.BORDER_CONSTANT, borderValue=(220, 220, 220))
warped_rgb = cv2.cvtColor(warped, cv2.COLOR_BGR2RGB)
show(warped_rgb, title='Document with perspective distortion')

In [ ]:
# Rectify back using the inverse mapping (dst -> src)
H_inv = cv2.getPerspectiveTransform(dst, src)
rectified = cv2.warpPerspective(warped, H_inv, (w, h))
rectified_rgb = cv2.cvtColor(rectified, cv2.COLOR_BGR2RGB)
show(rectified_rgb, title='Rectified back (should look like original)')

## Exercise 3: Build a Simple Document Scanner (Manual Corners)

Use the `warped_rgb` image above. Your goal is to produce a top-down view (rectified) with a *tight crop* around the document.

Steps:
1) Choose 4 corner points on the perspective image (hardcode them)
2) Map to a rectangle of size `(out_w, out_h)`
3) Apply `cv2.getPerspectiveTransform` + `cv2.warpPerspective`

Stretch goal: compute `out_w/out_h` based on distances between corners.


In [ ]:
# TODO: pick points in the warped image coordinate system
# Example format: pts = np.float32([[x1,y1],[x2,y2],[x3,y3],[x4,y4]])
pts = None

# TODO: define output rectangle
out_w, out_h = 500, 360
dst_rect = np.float32([[0, 0], [out_w - 1, 0], [out_w - 1, out_h - 1], [0, out_h - 1]])

# TODO: compute and warp
# H = cv2.getPerspectiveTransform(pts, dst_rect)
# scan = cv2.warpPerspective(cv2.cvtColor(warped_rgb, cv2.COLOR_RGB2BGR), H, (out_w, out_h))
# show(cv2.cvtColor(scan, cv2.COLOR_BGR2RGB), title='Scanned (rectified)')

print('Fill TODOs to run this cell.')

## Filtering, Noise, and Edge Detection

Core idea: many CV algorithms start with smoothing (reduce noise) then compute derivatives (edges).

We will cover:
- Convolution intuition
- Mean / Gaussian / Median / Bilateral filtering
- Gradients: Sobel
- Canny edge detector


In [ ]:
rng = np.random.default_rng(0)
noisy = gray.copy().astype(np.float32)
noisy += rng.normal(0, 18, size=noisy.shape).astype(np.float32)
noisy = np.clip(noisy, 0, 255).astype(np.uint8)

mean_blur = cv2.blur(noisy, (5, 5))
gauss = cv2.GaussianBlur(noisy, (5, 5), 1.2)
median = cv2.medianBlur(noisy, 5)
bilateral = cv2.bilateralFilter(noisy, d=7, sigmaColor=30, sigmaSpace=30)

show(noisy, title='Noisy')
show(mean_blur, title='Mean blur')
show(gauss, title='Gaussian blur')
show(median, title='Median blur')
show(bilateral, title='Bilateral filter')

In [ ]:
g = gauss
sx = cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3)
sy = cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3)
mag = cv2.magnitude(sx, sy)
mag_u8 = np.clip(mag / (mag.max() + 1e-6) * 255, 0, 255).astype(np.uint8)
edges = cv2.Canny(g, threshold1=50, threshold2=120)

show(g, title='Smoothed gray')
show(mag_u8, title='Sobel magnitude')
show(edges, title='Canny edges')

## Thresholding (Binary Segmentation)

Thresholding converts grayscale to black/white.
- Global threshold: one value for the whole image
- Otsu: chooses a value automatically (works when histogram is bimodal)
- Adaptive threshold: different value per local neighborhood (good for uneven lighting)


In [ ]:
g = cv2.GaussianBlur(gray, (5, 5), 0)
_, th_fixed = cv2.threshold(g, 120, 255, cv2.THRESH_BINARY)
_, th_otsu = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
th_adapt = cv2.adaptiveThreshold(g, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 21, 5)

show(g, title='Gray (blurred)')
show(th_fixed, title='Fixed threshold')
show(th_otsu, title='Otsu threshold')
show(th_adapt, title='Adaptive threshold')

## Exercise 4: Canny Threshold Exploration

Try 3 different pairs of thresholds `(t1, t2)` and compare:
- Too low: lots of noise edges
- Too high: missing true edges

Rule of thumb: `t2` is often ~2x to 3x `t1`, but it depends on the image.


In [ ]:
for t1, t2 in [(30, 90), (60, 180), (100, 240)]:
    e = cv2.Canny(gauss, t1, t2)
    show(e, title=f'Canny t1={t1}, t2={t2}', figsize=(5, 3))

## Morphology, Components, and Contours

Binary images become useful when you can clean them and measure shapes.

You will learn:
- Erosion / dilation
- Opening / closing
- Connected components labeling
- Contours, bounding boxes, area, perimeter


In [ ]:
# Make a simple binary mask from the HSV segmentation earlier (or threshold)
img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
mask = cv2.inRange(hsv, np.array([20, 80, 80]), np.array([40, 255, 255]))

kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
opened = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel)

show(mask, title='Raw mask')
show(opened, title='Opening (remove small noise)')
show(closed, title='Closing (fill small holes)')

In [ ]:
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(closed)
print('num_labels (including background):', num_labels)

# Visualize labels (skip background=0)
vis = np.zeros((*labels.shape, 3), dtype=np.uint8)
colors = np.array([[0, 0, 0]] + [list(map(int, rng.integers(0, 255, size=3))) for _ in range(num_labels - 1)], dtype=np.uint8)
vis = colors[labels]
show(vis, title='Connected component labels')

# Print areas
for i in range(1, num_labels):
    x, y, w, h, area = stats[i]
    cx, cy = centroids[i]
    print(f'label {i}: area={area}, bbox=({x},{y},{w},{h}), centroid=({cx:.1f},{cy:.1f})')

In [ ]:
contours, hierarchy = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
overlay = img_rgb.copy()
overlay_bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)

for cnt in contours:
    area = cv2.contourArea(cnt)
    if area < 50:
        continue
    x, y, w, h = cv2.boundingRect(cnt)
    cv2.rectangle(overlay_bgr, (x, y), (x + w, y + h), (0, 255, 0), 2)

show(cv2.cvtColor(overlay_bgr, cv2.COLOR_BGR2RGB), title='Contours bounding boxes')
print('contours found:', len(contours))

## Exercise 5: Shape Features on Synthetic Shapes

We'll generate a binary image with circles and rectangles, then classify each contour by a simple heuristic using:
- Area
- Perimeter
- Circularity: `4*pi*area / perimeter^2`

Task: label each detected object as `circle` or `rectangle` and draw the label on the image.


In [ ]:
canvas = np.zeros((420, 620), dtype=np.uint8)
cv2.circle(canvas, (140, 140), 60, 255, -1)
cv2.circle(canvas, (460, 120), 40, 255, -1)
cv2.rectangle(canvas, (80, 260), (220, 380), 255, -1)
cv2.rectangle(canvas, (360, 260), (560, 360), 255, -1)

# Add small noise
noise = (rng.random(canvas.shape) > 0.995).astype(np.uint8) * 255
canvas_noisy = cv2.bitwise_or(canvas, noise)

kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
clean = cv2.morphologyEx(canvas_noisy, cv2.MORPH_OPEN, kernel)

contours, _ = cv2.findContours(clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
out = cv2.cvtColor(clean, cv2.COLOR_GRAY2BGR)

import math

for cnt in contours:
    area = cv2.contourArea(cnt)
    if area < 200:
        continue
    peri = cv2.arcLength(cnt, True)
    circ = 4 * math.pi * area / (peri * peri + 1e-6)
    x, y, w, h = cv2.boundingRect(cnt)
    label = 'circle' if circ > 0.78 else 'rectangle'
    cv2.rectangle(out, (x, y), (x + w, y + h), (0, 255, 0), 2)
    cv2.putText(out, f'{label} ({circ:.2f})', (x, y - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA)

show(clean, title='Clean binary')
show(cv2.cvtColor(out, cv2.COLOR_BGR2RGB), title='Labeled shapes')

## Template Matching, Keypoints, and Homography

Two common ways to locate an object:
- Template matching (works for same scale/rotation)
- Feature matching (ORB/SIFT-like) + RANSAC homography (works under viewpoint changes for planar objects)

Note: SIFT may not be available in all OpenCV builds. We'll use ORB (free, fast).


In [ ]:
# Build a simple template matching demo using the synthetic image
scene = img_rgb.copy()
scene_gray = cv2.cvtColor(cv2.cvtColor(scene, cv2.COLOR_RGB2BGR), cv2.COLOR_BGR2GRAY)

# Template: a crop around the yellow circle area (adjust if you use your own image)
tmpl = scene_gray[70:190, 40:170]
res = cv2.matchTemplate(scene_gray, tmpl, cv2.TM_CCOEFF_NORMED)
min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(res)
top_left = max_loc
h, w = tmpl.shape

vis = cv2.cvtColor(scene_gray, cv2.COLOR_GRAY2BGR)
cv2.rectangle(vis, top_left, (top_left[0] + w, top_left[1] + h), (0, 255, 0), 2)

show(tmpl, title='Template')
show(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB), title=f'Match (score={max_val:.3f})')
print('best match:', top_left, 'score:', max_val)

In [ ]:
# ORB matching demo (scene vs rotated version)
img1 = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
img2 = cv2.cvtColor(rotate_about_center(img_rgb, 18), cv2.COLOR_RGB2GRAY)

orb = cv2.ORB_create(nfeatures=800)
kp1, des1 = orb.detectAndCompute(img1, None)
kp2, des2 = orb.detectAndCompute(img2, None)

bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = bf.match(des1, des2)
matches = sorted(matches, key=lambda m: m.distance)

vis = cv2.drawMatches(cv2.cvtColor(img1, cv2.COLOR_GRAY2BGR), kp1, cv2.cvtColor(img2, cv2.COLOR_GRAY2BGR), kp2, matches[:40], None, flags=2)
show(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB), title=f'ORB matches: {len(matches)} (showing 40)', figsize=(10, 5))
print('kp1:', len(kp1), 'kp2:', len(kp2), 'matches:', len(matches))

## Motion and Tracking (Intro)

We will simulate a simple video (moving dot) so you can learn motion methods without downloading videos:
- Background subtraction (conceptual)
- Optical flow (Farneback)
- Tracking by detecting the moving object per frame


In [ ]:
def make_moving_dot_video(num_frames=40, h=240, w=320):
    frames = []
    for t in range(num_frames):
        img = np.zeros((h, w), dtype=np.uint8)
        x = int(30 + t * (w - 60) / (num_frames - 1))
        y = int(h * 0.5 + 40 * np.sin(t * 0.25))
        cv2.circle(img, (x, y), 10, 255, -1)
        frames.append(img)
    return frames

frames = make_moving_dot_video()
show(frames[0], title='Frame 0')
show(frames[len(frames)//2], title='Middle frame')
show(frames[-1], title='Last frame')

In [ ]:
prev = frames[10]
nxt = frames[11]
flow = cv2.calcOpticalFlowFarneback(prev, nxt, None, 0.5, 3, 15, 3, 5, 1.2, 0)
fx, fy = flow[..., 0], flow[..., 1]
mag = np.sqrt(fx * fx + fy * fy)
mag_u8 = np.clip(mag / (mag.max() + 1e-6) * 255, 0, 255).astype(np.uint8)
show(prev, title='Prev')
show(nxt, title='Next')
show(mag_u8, title='Flow magnitude')
print('mean flow magnitude:', float(mag.mean()))

## Exercise 6: Track the Moving Dot

For each frame:
1) Threshold to get a binary mask
2) Find the centroid (mean x/y of white pixels)
3) Store the trajectory and plot it

Optional: draw the trajectory on the last frame.


### Video loop checklist

- Always call `cap.release()` when done
- Use `cv2.waitKey(...)` to process UI events
- If the camera stays busy, restart the kernel and close other apps using the webcam


## Video basics

Working with video means reading frames in a loop and responding to keys/events in real time.


Important: avoid having multiple notebooks/kernels trying to access the same webcam at the same time. Close any other camera-using process first.


In [ ]:
# check the main python file :
# python3 main.py --type="video"
# testVideo.mp4

In [ ]:
# check the main python file :
# python3 main.py --type="interactive_draw"

## Object detection

Here we'll look at classical (non-deep-learning) ways to locate objects or patterns in an image.


### Template matching

Template matching looks for a small template inside a larger image by sliding it over every possible position and scoring how well it matches.

Notes:

- it works best when the object appears at the same scale/rotation as the template
- the main choice is the matching method (a correlation-like score)


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
full = cv2.imread('data/sammy.jpg')
full = cv2.cvtColor(full, cv2.COLOR_BGR2RGB)

In [ ]:
plt.imshow(full)

In [ ]:
face = cv2.imread('data/sammy_face.jpg')
face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)

In [ ]:
plt.imshow(face)

In [ ]:
# All the 6 methods for comparison in a list
# Note how we are using strings, later on we'll use the eval() function to convert to function
methods = ['cv2.TM_CCOEFF', 'cv2.TM_CCOEFF_NORMED', 'cv2.TM_CCORR','cv2.TM_CCORR_NORMED', 'cv2.TM_SQDIFF', 'cv2.TM_SQDIFF_NORMED']

In [ ]:
# Here we are going to find the face in the full image using template matching with the 6 different methods, and show the results for each method

# How It Works ?
# 1 - Place the template at top-left of the image
# 2 - Compute similarity score
# 3 - Move template one pixel to the right
# 4 - Repeat for entire image

# It’s like sliding a small window across the image.

for m in methods:
    
    # CREATE A COPY
    full_copy = full.copy()
    method = eval(m)
    
    # TEMPLATE MATCHING
    res = cv2.matchTemplate(full_copy, face, method)
    
    min_val, max_value, min_loc, max_loc = cv2.minMaxLoc(res)
    
    if method in [cv2.TM_SQDIFF, cv2.TM_SQDIFF_NORMED]:
        top_left = min_loc
    else:
        top_left = max_loc

    height, width, channels = face.shape    
    
    bottom_right = (top_left[0]+width, top_left[1] + height)
    
    cv2.rectangle(full_copy, top_left, bottom_right, (255,0,0), 10)
    
    # PLOT AND SHOW THE IMAGES
    plt.subplot(121)
    plt.imshow(res)
    plt.title('HEATMAP OF TEMPLATE MATCHING')
    
    plt.subplot(122)
    plt.imshow(full_copy)
    plt.title('DETECTION OF TEMPLATE')
    plt.suptitle(m)
    
    plt.show()
    
    print('\n')
    print('\n')

> **🔍 Template Matching — how it works**
>
> `cv2.matchTemplate(image, template, method)` slides the template over the image and computes a similarity score at each position.
>
> The result `res` is a **2D score map** — each pixel represents how well the template matches at that location.
>
> **The 6 methods differ in how similarity is computed:**
>
> | Method | Best match |
> |---|---|
> | `TM_CCOEFF` / `TM_CCOEFF_NORMED` | Maximum value (`max_loc`) |
> | `TM_CCORR` / `TM_CCORR_NORMED` | Maximum value (`max_loc`) |
> | `TM_SQDIFF` / `TM_SQDIFF_NORMED` | Minimum value (`min_loc`) |
>
> `NORMED` variants normalize the score to `[0, 1]` — more robust across different image brightnesses.
>
> **Expected result**: a rectangle is drawn on the full image at the location where the face template best matches.


Example (choose a template matching method and run the match):

```python
myMethod = eval('cv2.TM_CCOEFF')
res = cv2.matchTemplate(full_copy, face, myMethod)
```

`res` is a 2D score map: each location contains the match score for placing the template there.


In [ ]:
plt.imshow(res)

### Corner detection

A corner is a point where intensity changes strongly in (at least) two different directions (think: a junction of two edges).

Two common methods:

- Harris: classic detector based on local intensity changes
- Shi-Tomasi: a refinement of Harris that often gives more stable points


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
flat_chess = cv2.imread('data/flat_chessboard.png')
flat_chess = cv2.cvtColor(flat_chess, cv2.COLOR_BGR2RGB)

In [ ]:
plt.imshow(flat_chess)

In [ ]:
gray_flat_chess = cv2.cvtColor(flat_chess, cv2.COLOR_BGR2GRAY)

In [ ]:
plt.imshow(gray_flat_chess, cmap='gray')

In [ ]:
real_chess = cv2.imread('data/real_chessboard.jpg')
real_chess = cv2.cvtColor(real_chess, cv2.COLOR_BGR2RGB)

In [ ]:
plt.imshow(real_chess)

In [ ]:
gray_real_chess = cv2.cvtColor(real_chess, cv2.COLOR_BGR2GRAY)

In [ ]:
plt.imshow(gray_real_chess, cmap='gray')

In [ ]:
gray = np.float32(gray_flat_chess)

In [ ]:
# Harris Corner Detection Algorithm
dst = cv2.cornerHarris(src=gray, blockSize=2, ksize=3, k=0.04)

In [ ]:
# Needed to show the results
dst = cv2.dilate(dst, None)

In [ ]:
# Whenever our HCA is greater than 10 % of our max value, set it to red (for visualization)
flat_chess[dst>0.01 * dst.max()] = [255,0,0] #RGB

In [ ]:
# Can't detect on edge because there is nothing on the edge to see to
plt.imshow(flat_chess)

> **🔵 Harris Corner Detection**
>
> `cv2.cornerHarris(src, blockSize, ksize, k)` detects corners by measuring how much intensity changes in multiple directions.
>
> **Parameters:**
> - `blockSize=2` — neighborhood size considered around each pixel
> - `ksize=3` — Sobel kernel size used internally for derivatives
> - `k=0.04` — Harris detector free parameter (typically 0.04–0.06)
>
> A **corner** is where intensity changes strongly in at least 2 different directions (unlike an edge, which only changes in 1 direction).
>
> `cv2.dilate(dst, None)` enlarges the detected corner spots so they are visible on the image.
>
> `flat_chess[dst > 0.01 * dst.max()] = [255,0,0]` colors all pixels that are corners with red.
>
> **Expected result**: red dots at the internal corners of the chessboard. Note that border corners may be missed (no neighborhood on one side).


In [ ]:
gray = np.float32(gray_real_chess)

dst = cv2.cornerHarris(src=gray, blockSize=2, ksize=3, k=0.04)

In [ ]:
dst = cv2.dilate(dst, None)

In [ ]:
real_chess[dst>0.01 * dst.max()] = [255,0,0] #RGB

In [ ]:
plt.imshow(real_chess)

In [ ]:
# Shi-Tomasi Corner Detection Algorithm

In [ ]:
flat_chess = cv2.imread('data/flat_chessboard.png')
flat_chess = cv2.cvtColor(flat_chess, cv2.COLOR_BGR2RGB)
gray_flat_chess = cv2.cvtColor(flat_chess, cv2.COLOR_BGR2GRAY)

real_chess = cv2.imread('data/real_chessboard.jpg')
real_chess = cv2.cvtColor(real_chess, cv2.COLOR_BGR2RGB)
gray_real_chess = cv2.cvtColor(real_chess, cv2.COLOR_BGR2GRAY)

In [ ]:
# 5 - how many corners to detect
# downside to this method is that it doesn't automatically mark corners
corners = cv2.goodFeaturesToTrack(gray_flat_chess, 64, 0.01, 10)

In [ ]:
corners = np.int32(corners)

In [ ]:
for i in corners:
    x,y = i.ravel()
    cv2.circle(flat_chess, (x,y), 3, (255,0,0), -1)

In [ ]:
plt.imshow(flat_chess)

> **🔵 Shi-Tomasi Corner Detection**
>
> `cv2.goodFeaturesToTrack(gray, maxCorners, qualityLevel, minDistance)` is an improved version of Harris corners.
>
> **Parameters:**
> - `maxCorners=64` — maximum number of corners to return
> - `qualityLevel=0.01` — minimum quality relative to the best corner (filters out weak corners)
> - `minDistance=10` — minimum pixel distance between returned corners (avoids clusters)
>
> **Advantage over Harris**: automatically selects the N best corners; easier to use for tracking applications.
>
> The detected corners are then drawn as red circles using a loop.
>
> **Expected result**: red dots on the flat chessboard at the grid intersections.


In [ ]:
corners = cv2.goodFeaturesToTrack(gray_real_chess, 100, 0.01, 10)

In [ ]:
corners = np.int32(corners)

In [ ]:
for i in corners:
    x,y = i.ravel()
    cv2.circle(real_chess, (x,y), 3, (255,0,0), -1)

In [ ]:
plt.imshow(real_chess)

### Edge detection

Edges are locations with strong intensity change. A popular detector is Canny, which is a multi-stage process:

1) blur (often Gaussian) to reduce noise
2) compute image gradients
3) non-maximum suppression (thin edges)
4) double threshold (strong vs weak edges)
5) hysteresis (keep weak edges connected to strong ones)

Practical tips:

- for high-resolution images, applying a custom blur first can help
- Canny requires choosing low/high thresholds; the right values depend on contrast and noise


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
img = cv2.imread('data/sammy_face.jpg')

In [ ]:
plt.imshow(img)

In [ ]:
edges = cv2.Canny(image=img, threshold1=127, threshold2=127)

In [ ]:
plt.imshow(edges)

> **🔍 Canny Edge Detection**
>
> `cv2.Canny(image, threshold1, threshold2)` is a multi-stage edge detector:
>
> 1. **Gaussian blur** (internal) — reduces noise
> 2. **Gradient computation** — finds intensity changes
> 3. **Non-maximum suppression** — thins edges to 1 pixel
> 4. **Double thresholding**:
>    - Pixels above `threshold2` → definite edges (strong)
>    - Pixels between `threshold1` and `threshold2` → weak edges
>    - Pixels below `threshold1` → discarded
> 5. **Hysteresis** — weak edges are kept only if connected to strong edges
>
> **Effect of threshold values:**
> - Both set to `127` → balanced
> - `threshold1=0, threshold2=255` → very few edges (only very strong ones pass)
> - Low values → many edges including noise


In [ ]:
edges = cv2.Canny(image=img, threshold1=0, threshold2=255)

In [ ]:
plt.imshow(edges)

In [ ]:
# Median pixel value
med_val = np.median(img)
med_val

In [ ]:
# Set the lower threshold to 0 or 70% of the median value, whichever is greater
lower = int(max(0, 0.7*med_val))
# Set the upper threshold to max of 255 or 130% of the median value, whichever is smaller
upper = int(min(255, 1.3*med_val))

In [ ]:
edges = cv2.Canny(image=img, threshold1=lower, threshold2=upper+100)

In [ ]:
plt.imshow(edges)

> **🎯 Automatic threshold selection for Canny**
>
> Instead of guessing thresholds, we compute them from the **median pixel intensity** of the image:
>
> ```python
> lower = max(0, 0.7 × median)
> upper = min(255, 1.3 × median)
> ```
>
> This heuristic adapts to the overall brightness of the image:
> - Dark images → lower thresholds (needed to detect subtle edges)
> - Bright images → higher thresholds
>
> The `+100` on the upper threshold in cell 388 makes it more selective (only catches strong edges).
>
> **Best practice**: always blur slightly before Canny (cell 390) to reduce noise-induced false edges.


In [ ]:
blurred_img = cv2.blur(img, ksize=(5,5))
edges = cv2.Canny(image=blurred_img, threshold1=lower, threshold2=upper+50)

In [ ]:
plt.imshow(edges)

### Grid detection

Grid patterns (checkerboards, dot grids) are commonly used for camera calibration and tracking because they provide many reliable corners/points.

Calibration helps correct lens distortion (radial/tangential), which can improve downstream tasks like measurement and tracking.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
flat_chess = cv2.imread('data/flat_chessboard.png')

In [ ]:
plt.imshow(flat_chess)

In [ ]:
# found is if it was able to find a chessboard
# corners is a list of coordinates that it found a chess board
found, corners = cv2.findChessboardCorners(flat_chess, (7,7))

In [ ]:
cv2.drawChessboardCorners(flat_chess, (7,7), corners, found)
print ('suppressed')

In [ ]:
plt.imshow(flat_chess)

> **♟️ Chessboard corner detection**
>
> `cv2.findChessboardCorners(img, (7,7))` automatically detects a **7×7 grid** of internal corners on a chessboard pattern.
>
> Note: `(7,7)` refers to the number of **inner corners per row/column** — an 8×8 chessboard has 7×7 internal corners.
>
> `cv2.drawChessboardCorners(img, (7,7), corners, found)` draws the detected corners with connecting lines.
>
> **Why is this useful?**  
> Chessboard patterns are the standard tool for **camera calibration** — knowing where these corners are in the image vs. in 3D space lets you estimate lens distortion and camera parameters.


In [ ]:
dots = cv2.imread('data/dot_grid.png')

In [ ]:
plt.imshow(dots)

In [ ]:
found, corners = cv2.findCirclesGrid(dots, (10,10), cv2.CALIB_CB_SYMMETRIC_GRID)
found

In [ ]:
cv2.drawChessboardCorners(dots, (10,10), corners, found)
print('suppressed')

In [ ]:
plt.imshow(dots)

> **🔵 Circular grid detection**
>
> `cv2.findCirclesGrid(img, (10,10), cv2.CALIB_CB_SYMMETRIC_GRID)` detects a 10×10 grid of circles.
>
> Like chessboard corners, circular grids are used for camera calibration — they are often preferred because circle centers can be localized more precisely than corner intersections.
>
> **Expected result**: the 100 detected circle centers are highlighted and connected on the dot grid image.


### Contour detection

Contours represent curves along object boundaries (typically extracted from a binary image).

They are useful for:

- shape analysis (area, perimeter, bounding boxes)
- filtering by size/shape
- separating external vs internal boundaries

Terminology:

- external contours: outer boundary
- internal contours: holes inside an object


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
img = cv2.imread('data/internal_external.png', 0)

In [ ]:
plt.imshow(img, cmap='gray')

In [ ]:
image = img.copy()

In [ ]:
contours, hierarchy = cv2.findContours(img, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)

In [ ]:
contours

In [ ]:
hierarchy

In [ ]:
external_contours = np.zeros(image.shape)

In [ ]:
for i in range(len(contours)):
    # External Contour
    if hierarchy[0][i][3] == -1:
        cv2.drawContours(external_contours, contours, i, 255, -1)

In [ ]:
plt.imshow(external_contours, cmap='gray')

In [ ]:
internal_contours = np.zeros(image.shape)

for i in range(len(contours)):
    # External Contour
    if hierarchy[0][i][3] != -1:
        cv2.drawContours(internal_contours, contours, i, 255, -1)

In [ ]:
plt.imshow(internal_contours, cmap='gray')

> **📐 External vs Internal contours**
>
> `cv2.findContours(img, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)` finds all contours and organizes them into a 2-level hierarchy:
>
> - **Level 0** (top) = external contours (outer boundaries)
> - **Level 1** = internal contours (holes inside objects)
>
> **Understanding `hierarchy[0][i]`**: for each contour `i`, it stores `[next, prev, first_child, parent]`.  
> - `hierarchy[0][i][3] == -1` → no parent → **external contour**
> - `hierarchy[0][i][3] != -1` → has a parent → **internal contour (hole)**
>
> **Expected results:**
> - Cell 416: external boundaries filled in white
> - Cell 418: holes/internal boundaries filled in white


### Feature matching

Feature matching finds correspondences between two images even when the object is not an exact pixel copy (changes in viewpoint, lighting, partial occlusion).

Typical pipeline:

1) detect keypoints
2) compute descriptors
3) match descriptors using a distance metric
4) optionally filter matches (ratio test, RANSAC)


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def display(img, cmap='gray'):
    fig = plt.figure(figsize=(12,10))
    ax = fig.add_subplot(111)
    ax.imshow(img, cmap='gray')

In [ ]:
reeses = cv2.imread('data/reeses_puffs.png', 0)

In [ ]:
display(reeses)

In [ ]:
cereals = cv2.imread('data/many_cereals.jpg', 0)

In [ ]:
display(cereals)

In [ ]:
# ORB Descriptors
orb = cv2.ORB_create()

In [ ]:
# Get keypoints and descriptors
kp1, des1 = orb.detectAndCompute(reeses, None)
kp2, des2 = orb.detectAndCompute(cereals, None)

In [ ]:
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)

In [ ]:
matches = bf.match(des1, des2)

In [ ]:
# Sort in order of distance
matches = sorted(matches, key=lambda x:x.distance)

In [ ]:
reeses_matches = cv2.drawMatches(reeses, kp1, cereals, kp2, matches[:25], None, flags=2)

In [ ]:
display(reeses_matches)

> **🔑 Feature matching with ORB + Brute-Force Matcher**
>
> **ORB (Oriented FAST and Rotated BRIEF)** is a fast, patent-free feature detector and descriptor:
> 1. `orb.detectAndCompute(img, None)` → finds keypoints (interesting points) + computes binary descriptors
> 2. `cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)` → matches descriptors using Hamming distance (counts different bits)
> 3. `bf.match(des1, des2)` → finds the best match for each descriptor
> 4. Sorting by `distance` → best matches first
>
> The `crossCheck=True` option ensures a match is only accepted if descriptor A's best match is B, **and** B's best match is also A.
>
> **Expected result**: lines connecting matching keypoints between the Reese's Puffs box and the cereals shelf image.  
> Some matches may be wrong (outliers) — this is normal with brute-force matching.


In [ ]:
# Scale Invariant Feature Transfer (SIFT) Descriptors
sift = cv2.xfeatures2d.SIFT_create()

In [ ]:
kp1, des1 = sift.detectAndCompute(reeses, None)
kp2, des2 = sift.detectAndCompute(cereals, None)

In [ ]:
bf = cv2.BFMatcher()

In [ ]:
matches = bf.knnMatch(des1, des2, k=2)

In [ ]:
# Ratio Test
# Less distance == better match
good = []

for match1, match2 in matches:
    # If match 1 distance is less than 75% of match 2 distance
    # then descriptor for that row is a good match, keep it
    if match1.distance < 0.75*match2.distance:
        good.append([match1])

In [ ]:
sift_matches = cv2.drawMatchesKnn(reeses, kp1, cereals, kp2, good, None, flags=2)

In [ ]:
display(sift_matches)

> **🎯 SIFT + Lowe's Ratio Test**
>
> **SIFT (Scale-Invariant Feature Transform)** produces more distinctive (but floating-point, larger) descriptors than ORB.  
> Note: SIFT requires `opencv-contrib-python`.
>
> **knnMatch(des1, des2, k=2)** returns the **2 best matches** for each descriptor.
>
> **Lowe's Ratio Test** (the filter in cell 438):
> ```python
> if match1.distance < 0.75 × match2.distance:
>     keep match1  # it's clearly better than the 2nd best
> ```
>
> If the best match is much better than the second best, it's likely a **true match**.  
> If they're similar in distance, the match is ambiguous → discard it.
>
> **Expected result**: fewer but more reliable matches compared to the brute-force ORB approach.


In [ ]:
# FLANN (Fast Library For Approximate Near Neighbors) Based Measure

In [ ]:
sift = cv2.xfeatures2d.SIFT_create()

In [ ]:
kp1, des1 = sift.detectAndCompute(reeses, None)
kp2, des2 = sift.detectAndCompute(cereals, None)

In [ ]:
FLANN_INDEX_KDTREE = 0
index_params = dict(algorithm=FLANN_INDEX_KDTREE,trees=5)
search_params = dict(checks=50)

In [ ]:
flann = cv2.FlannBasedMatcher(index_params, search_params)

In [ ]:
matches = flann.knnMatch(des1, des2, k=2)

In [ ]:
matchesMask = [[0,0] for i in range(len(matches))]

In [ ]:
# Ratio Test
# Less distance == better match
for i,(match1, match2) in enumerate(matches):
    # If match 1 distance is less than 70% of match 2 distance
    # then descriptor for that row is a good match, keep it
    if match1.distance < 0.7*match2.distance:
        matchesMask[i] = [1,0]

In [ ]:
draw_params = dict(matchColor=(0,255,0), 
                   singlePointColor=(255,0,0),
                   matchesMask=matchesMask,
                   flags=0)

In [ ]:
flann_matches = cv2.drawMatchesKnn(reeses, kp1, cereals, kp2, matches, None, **draw_params)

In [ ]:
display(flann_matches)

> **⚡ FLANN-based matching**
>
> **FLANN (Fast Library for Approximate Nearest Neighbors)** is faster than brute-force matching for large descriptor sets.
>
> - `FLANN_INDEX_KDTREE=0` with `trees=5` → builds a KD-tree index for efficient search
> - `checks=50` → number of tree traversals per query (higher = more accurate but slower)
>
> Same ratio test as before (`0.7` threshold), but results are stored in `matchesMask` (list of `[1,0]` / `[0,0]`) to control which matches are drawn.
>
> **Expected result**: matches visualized with green lines (good matches) and single red dots (unmatched keypoints) — similar to SIFT but computed faster for large datasets.


### Watershed algorithm

Watershed is a segmentation method that treats a grayscale image like a topographic surface:

- high intensity = peaks
- low intensity = valleys

If you place seed markers for different regions, the algorithm "floods" the valleys outward until different floods meet. The meeting lines become the region boundaries.


Analogy:

Imagine water rising in a landscape. Basins fill first (local minima). When water from two basins would merge, a ridge line is created to keep them separate. Those ridge lines are the segmentation boundaries.


In [ ]:
import cv2 
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def display(img, cmap='gray'):
    fig = plt.figure(figsize=(12,10))
    ax = fig.add_subplot(111)
    ax.imshow(img, cmap='gray')

In [ ]:
sep_coins = cv2.imread('data/pennies.jpg')

In [ ]:
plt.imshow(sep_coins)

In [ ]:
# Median Blur
sep_blur = cv2.medianBlur(sep_coins, 25)

In [ ]:
display(sep_blur)

In [ ]:
# Grayscale
gray_sep_coins = cv2.cvtColor(sep_blur, cv2.COLOR_BGR2GRAY)

In [ ]:
display(gray_sep_coins)

In [ ]:
# Binary Threshold
ret, sep_thresh = cv2.threshold(gray_sep_coins, 160, 255, cv2.THRESH_BINARY_INV)

In [ ]:
display(sep_thresh)

In [ ]:
# Find Contours
image = sep_thresh.copy()
contours, hierarchy = cv2.findContours(image, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)

In [ ]:
for i in range(len(contours)):
    if hierarchy[0][i][3] == -1:
        cv2.drawContours(sep_coins, contours, i, (255,0,0), 10)

In [ ]:
display(sep_coins)

> **🪙 Simple coin detection with thresholding + contours**
>
> This is a first attempt at separating touching coins:
> 1. **Median blur** (cell 459) — smooths the image while preserving coin boundaries
> 2. **Grayscale** (cell 461)
> 3. **Binary inverse threshold** (cell 463) — coins become white, background becomes black
> 4. **Find + draw contours** (cells 465–466)
>
> **Limitation**: if coins are touching, `findContours` may detect them as a single blob.  
> The Watershed algorithm (next section) solves this problem.


In [ ]:
# WATERSHED ALGORITHM
img = cv2.imread('data/pennies.jpg')

In [ ]:
img = cv2.medianBlur(img, 35)

In [ ]:
display(img)

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

In [ ]:
# Otsu's Method
ret, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)

In [ ]:
display(thresh)

In [ ]:
# NOISE REMOVAL (OPTIONAL)  
kernel = np.ones((3,3), np.uint8)

In [ ]:
opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)

In [ ]:
display(opening)

> **🧹 Watershed step 1 — Noise removal with Opening**
>
> `cv2.THRESH_OTSU` automatically determines the best global threshold using Otsu's method:  
> it finds the value that minimizes the within-class intensity variance (works best for bimodal histograms).
>
> `cv2.THRESH_BINARY_INV` inverts the result: coins become white, background becomes black.
>
> `cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)` removes small noise blobs that could confuse the Watershed algorithm.


In [ ]:
sure_bg = cv2.dilate(opening, kernel, iterations=3)

In [ ]:
display(sure_bg, cmap='gray')

In [ ]:
# DISTANCE TRANSFORM
dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)

In [ ]:
display(dist_transform)

In [ ]:
ret, sure_fg = cv2.threshold(dist_transform, 0.7*dist_transform.max(), 255,0)

In [ ]:
display(sure_fg)

In [ ]:
sure_fg = np.uint8(sure_fg)

In [ ]:
unknown = cv2.subtract(sure_bg, sure_fg)

In [ ]:
display(unknown)

> **📍 Watershed step 2 — Distance Transform & Sure Foreground/Background**
>
> **Distance Transform** (`cv2.distanceTransform`): each white pixel is replaced by its distance to the nearest black pixel.  
> → Centers of coins appear bright (far from background); coin edges appear dark.
>
> **Sure foreground** (cell 481): threshold at `0.7 × max(dist_transform)` → only keep the coin centers that are definitely foreground.
>
> **Sure background** (cell 477): dilate the opened mask → expand to get pixels that are definitely background.
>
> **Unknown region** (cell 484): `sure_bg - sure_fg` → the border zone between coins (where we're unsure) — this is exactly where Watershed will place its boundaries.


In [ ]:
ret, markers = cv2.connectedComponents(sure_fg)

In [ ]:
markers = markers + 1

In [ ]:
markers[unknown==255] = 0

In [ ]:
display(markers)

In [ ]:
markers = cv2.watershed(img, markers)

In [ ]:
display(markers)

> **🌊 Watershed step 3 — Running the algorithm**
>
> `cv2.watershed(img, markers)` floods from the seed markers:
> - Regions marked `≥ 2` → known coin regions (seeds)
> - Region marked `1` → known background
> - Region marked `0` → unknown (to be determined)
>
> After running, **boundary pixels are marked with `-1`** — these are the watershed lines separating touching coins.
>
> **Expected result in cell 491**: a marker image where each coin has a unique label ID and boundaries are -1.  
> The final cell 494 draws the detected contours in blue on the original coin image — each touching coin should be separated.


In [ ]:
image = markers.copy()
contours, hierarchy = cv2.findContours(image, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)

In [ ]:
for i in range(len(contours)):
    if hierarchy[0][i][3] == -1:
        cv2.drawContours(sep_coins, contours, i, (255,0,0), 10)

In [ ]:
display(sep_coins)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
road = cv2.imread('data/road_image.jpg')

In [ ]:
road_copy = np.copy(road)

In [ ]:
plt.imshow(road)

In [ ]:
road.shape

In [ ]:
marker_image = np.zeros(road.shape[:2], dtype=np.int32)

In [ ]:
segments = np.zeros(road.shape, dtype=np.uint8)

In [ ]:
from matplotlib import cm

In [ ]:
cm.tab10(0)

In [ ]:
def create_rgb(i):
    return tuple(np.array(cm.tab10(i)[:3]) * 255)

In [ ]:
colors = []
for i in range(10):
    colors.append(create_rgb(i))

In [ ]:
colors

In [ ]:
# check the main python file :
# python3 main.py --type="watershed"

In this demo you:

- load a road image
- mark regions with the mouse (seed labels)
- run watershed using those markers
- visualize each region with a different color
- optionally clear markers or switch label IDs interactively


### Haar cascades and face detection

Haar cascades (Viola-Jones) can detect faces (or other objects) by scanning an image with many simple rectangular features.

Important distinction:

- detection: "is there a face and where is it?"
- recognition: "whose face is it?" (this requires a different approach, often deep learning)

Why it can be fast:

- integral images allow fast feature sums
- a cascade of classifiers quickly rejects most non-face windows

Limitations:

- sensitive to pose/lighting/occlusion
- training your own cascades is data- and time-intensive


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
nadia = cv2.imread('data/Nadia_Murad.jpg', 0)
dennis = cv2.imread('data/Denis_Mukwege.jpg', 0)
solvay = cv2.imread('data/solvay_conference.jpg', 0)

In [ ]:
face_cascade = cv2.CascadeClassifier('data/haarcascades/haarcascade_frontalface_default.xml')

In [ ]:
def detect_face(img):
    face_img = img.copy()
    
    face_rects = face_cascade.detectMultiScale(face_img)
    
    for (x,y,w,h) in face_rects:
        cv2.rectangle(face_img, (x,y), (x+w, y+h), (255,255,255), 10)
        
    return face_img

In [ ]:
result = detect_face(dennis)

In [ ]:
plt.imshow(result, cmap='gray')

> **👤 Face detection with Haar Cascades**
>
> `cv2.CascadeClassifier` loads a pre-trained cascade model from an XML file.  
> `face_cascade.detectMultiScale(img)` scans the image at multiple scales and returns a list of rectangles `(x, y, w, h)` where faces were detected.
>
> **How it works:**
> 1. The image is scanned with a sliding window at multiple sizes
> 2. At each position, ~6000 simple rectangular features are evaluated quickly using **integral images**
> 3. The cascade structure rejects most non-face windows after just a few features
> 4. Only windows that pass all stages are reported as faces
>
> **Expected result**: white rectangle(s) drawn around detected faces.


In [ ]:
result = detect_face(solvay)

In [ ]:
plt.imshow(result, cmap='gray')

In [ ]:
def adj_detect_face(img):
    face_img = img.copy()
    
    face_rects = face_cascade.detectMultiScale(face_img, scaleFactor=1.2, minNeighbors=5)
    
    for (x,y,w,h) in face_rects:
        cv2.rectangle(face_img, (x,y), (x+w, y+h), (255,255,255), 10)
        
    return face_img

In [ ]:
result = adj_detect_face(solvay)

In [ ]:
plt.imshow(result, cmap='gray')

> **⚙️ Tuning detection parameters**
>
> `detectMultiScale(img, scaleFactor=1.2, minNeighbors=5)`:
>
> | Parameter | Effect |
> |---|---|
> | `scaleFactor=1.2` | Image is shrunk by 20% at each scale step; smaller value → more scales checked → slower but catches more sizes |
> | `minNeighbors=5` | A detection is kept only if it has at least 5 overlapping neighbor detections; higher → fewer but more reliable detections |
>
> **When to tune**: if you're getting too many false positives (random things detected as faces), increase `minNeighbors`. If real faces are being missed, decrease it.


In [ ]:
eye_cascade = cv2.CascadeClassifier('data/haarcascades/haarcascade_eye.xml')

In [ ]:
def detect_eyes(img):
    face_img = img.copy()
    
    eye_rects = eye_cascade.detectMultiScale(face_img, scaleFactor=1.2, minNeighbors=5)
    
    for (x,y,w,h) in eye_rects:
        cv2.rectangle(face_img, (x,y), (x+w, y+h), (255,255,255), 10)
        
    return face_img

In [ ]:
result = detect_eyes(nadia)

In [ ]:
plt.imshow(result, cmap='gray')

In [ ]:
result = detect_eyes(dennis)
plt.imshow(result, cmap='gray')

In [ ]:
# check the main python file :
# python3 main.py --type="detect_face"

## Object tracking

Tracking follows an object through a video over time once you have an initial location.


### Optical flow

Optical flow estimates motion between two consecutive frames.

Common assumptions:

1) pixel intensities of a moving point do not change much between frames
2) neighboring pixels move similarly

In OpenCV:

- Lucas-Kanade computes **sparse** optical flow (for selected points)
- Farneback computes **dense** optical flow (for many/all pixels)

A common visualization converts flow vectors to HSV: hue encodes direction, value encodes magnitude.

Image pyramids can improve tracking across scales: https://en.wikipedia.org/wiki/Pyramid_(image_processing)


In [ ]:
import cv2
import numpy as np

In [ ]:
# SPARSE OPTICAL FLOW (LUCAS KANADE)
corner_track_params = dict(maxCorners=25, qualityLevel=0.3, minDistance=7, blockSize=7)

In [ ]:
corner_track_params

In [ ]:
lk_params = dict(winSize=(200,200), maxLevel=2,  criteria =(cv2.TERM_CRITERIA_EPS | cv2.TermCriteria_COUNT, 10, 0.03))

> **🏃 Sparse Optical Flow — Lucas-Kanade parameters**
>
> **Shi-Tomasi params** (`corner_track_params`):
> - `maxCorners=25` — track at most 25 points
> - `qualityLevel=0.3` — only keep corners with at least 30% of the best corner quality
> - `minDistance=7` — minimum pixel distance between tracked points
>
> **Lucas-Kanade params** (`lk_params`):
> - `winSize=(200, 200)` — search window around each point (large = can handle fast motion)
> - `maxLevel=2` — use 3-level image pyramid (handles motion at different scales)
> - `criteria` — stopping condition (stop when moving < 0.03 px OR after 10 iterations)
>
> **What it does**: for each selected corner, LK estimates where it moved in the next frame by minimizing intensity difference inside the window.


In [ ]:
# Lucas-Kanade Sparse Optical Flow
# check the main python file :
# python3 main.py --type="sparse_optical_flow"

In [ ]:
# DENSE OPTICAL FLOW (GUNNER FARNEBACK)
# check the main python file :
# python3 main.py --type="dense_optical_flow2"

### MeanShift and CamShift

MeanShift iteratively moves a window toward the region with the highest density (in this context, the best match to a color distribution).

Typical tracking idea:

- select a region of interest (ROI)
- build a color histogram for that ROI
- compute back-projection on each frame
- shift the window toward the peak response until convergence

CamShift (Continuously Adaptive MeanShift) extends this by adapting the window size (and rotation in some variants) as the target changes scale.


In [ ]:
import cv2
import numpy as np

This function detects a face once, then tracks it in real time using MeanShift (optionally CamShift).

Tracking is based on a color-histogram back-projection of the detected face region.


In [ ]:
# check the main python file :
# python3 main.py --type="meanshift"

> **🎯 MeanShift and CamShift tracking**
>
> **MeanShift tracking** workflow:
> 1. Select a ROI (region of interest) around the object to track (e.g., a face)
> 2. Compute a **color histogram** (usually in Hue channel of HSV) for that ROI
> 3. For each new frame, compute the **back-projection** — a map showing how likely each pixel belongs to the target color distribution
> 4. **MeanShift** iteratively moves the search window toward the peak of the back-projection
>
> **CamShift** extends MeanShift by also **adapting the window size** as the object gets closer/farther.
>
> **Expected behavior**: the window follows the face across frames even as it moves.  
> Run via `python3 main.py --type="meanshift"` as it requires a live video window.


### Note on OpenCV versions

Some trackers live in `opencv-contrib-python` and APIs differ across OpenCV versions. If a tracker constructor is missing, check your OpenCV version and whether contrib modules are installed.


### Built-in Tracking APIs (OpenCV)

OpenCV provides several **ready-to-use object tracking algorithms**, accessible through simple API calls:

- **Boosting Tracker**  
  - Based on the **AdaBoost algorithm** (same principle as Haar Cascade face detection)  
  - Evaluates the object across **multiple frames**  

- **MIL Tracker (Multiple Instance Learning)**  
  - Extends Boosting by considering a **neighborhood of points** around the current location  
  - Creates **multiple instances** for more robust tracking  

- **KCF Tracker (Kernelized Correlation Filters)**  
  - Builds on MIL principles and exploits overlapping data points  
  - **Faster and more accurate**, often a **good first choice** for tracking  

- **TLD Tracker (Tracking, Learning, Detection)**  
  - Handles **occlusions** and **large scale changes** well  
  - Can produce **false positives**, but robust under challenging conditions  

- **MedianFlow Tracker**  
  - Excellent at **detecting tracking failures**  
  - Works well with **predictable motion**, but fails with **fast-moving objects**


# Capstone Projects (Pick 1)

Deliverable format (recommended):
- A single notebook section with: problem statement, approach, results, and failure cases
- Clear visualizations + a few quantitative metrics

Project A: Document Scanner
- Input: photo of a document
- Output: rectified scan + optional contrast enhancement (CLAHE)
- Bonus: automatically find corners using edges + contours

Project B: Color-Based Object Counter
- Input: image with objects of a specific color (e.g., candies, LEGO, tokens)
- Output: mask cleaning (morphology) + connected components count + bounding boxes
- Bonus: report size distribution (areas histogram)

Project C: Classical 'Find My Logo'
- Input: a template image + a larger scene image
- Output: detected location using ORB matching + RANSAC homography
- Bonus: compare to template matching and explain when each fails


## Install libraries


We'll use:

- OpenCV (`opencv-python`)
- Matplotlib (`matplotlib`)


Install the packages (inside the activated environment):

```bash
pip install opencv-python matplotlib
```

If you install packages from inside Jupyter, restart the kernel so the imports pick up the new environment.


In [ ]:
!pip install opencv-python matplotlib

> **📦 What just happened?**
>
> `pip install opencv-python matplotlib` downloads and installs:
> - **OpenCV** (`cv2`): the main computer vision library we'll use throughout this course.
> - **Matplotlib** (`plt`): a plotting library to display images inside the notebook.
>
> The `!` prefix tells Jupyter to run the command in the system shell instead of Python.


In [ ]:
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
from PIL import Image

print('python:', sys.version.split()[0])
print('opencv:', cv2.__version__)
print('numpy :', np.__version__)
print('mpl   :', plt.matplotlib.__version__)

> **🔍 What this cell does — Library imports & version check**
>
> ```python
> import cv2        # OpenCV: image loading, processing, drawing
> import numpy as np  # NumPy: all images are NumPy arrays under the hood
> import matplotlib.pyplot as plt  # Matplotlib: for displaying images in the notebook
> from PIL import Image  # Pillow: an alternative image-loading library
> ```
>
> `%matplotlib inline` is a Jupyter magic command that makes plots appear directly below the cell.
>
> **Expected output**: three version strings, e.g. `opencv: 4.x.x`, `numpy: 1.x.x`, `mpl: 3.x.x`.  
> If you see an `ImportError`, the library was not installed correctly — re-run cell 7.


## NumPy and Matplotlib recap

NumPy is the core library for arrays (including images).

```python
import numpy as np
```

A few useful NumPy functions you will see often:

- `np.arange` create a range of values
- `np.zeros` / `np.ones` create arrays filled with 0/1
- `arr.shape` read the array shape
- `arr.reshape(...)` change the view/shape
- `arr.max()` / `arr.min()`
- `arr.argmax()` / `arr.argmin()`
- `arr.mean()`
- `arr.copy()`
- `np.asarray(obj)` convert to an array

Slicing means selecting parts of an array, e.g. `arr[:10, :10]`.

### How images look as arrays

Most color images are stored as a 3D array with shape `(H, W, 3)`:

- `H` = height (rows)
- `W` = width (columns)
- `3` = color channels

Common value ranges:

- `uint8` images: integers in `[0, 255]`
- normalized floats: floats in `[0.0, 1.0]`

Matplotlib can display images inside the notebook with `plt.imshow(...)`.

Tip: colormaps like `viridis`, `plasma`, `inferno`, and `magma` are designed to remain readable for many forms of color vision deficiency.

References:

- RGB model: https://en.wikipedia.org/wiki/RGB_color_model
- Matplotlib: https://matplotlib.org/


In [ ]:
pic = Image.open('data/00-puppy.jpg')

> **🖼️ Loading an image with Pillow**
>
> `Image.open(path)` reads the image file from disk and returns a **PIL Image object** — a high-level representation of the image.  
> At this point the image is *not yet* a NumPy array; it is a Pillow object with attributes like `.size` (width, height) and `.mode` (e.g. `RGB`).
>
> We use Pillow here to illustrate the difference between the two common image representations before switching to OpenCV.


In [ ]:
type(pic)

In [ ]:
pic_arr = np.asarray(pic)

In [ ]:
type(pic_arr)

In [ ]:
pic_arr.shape

> **📐 Images as NumPy arrays**
>
> `np.asarray(pic)` converts the Pillow image into a **NumPy array**.  
> Every color image becomes a 3-D array with shape `(height, width, channels)`.
>
> For example, a 400×600 RGB image will have shape `(400, 600, 3)`:
> - `400` rows (height)
> - `600` columns (width)
> - `3` channels: Red, Green, Blue
>
> Each element is an integer in `[0, 255]` (`uint8`).  
> `0` = no intensity for that channel, `255` = full intensity.
>
> **Expected output of `pic_arr.shape`**: something like `(H, W, 3)`.


In [ ]:
pic_arr

In [ ]:
plt.imshow(pic_arr)

> **📊 Displaying an image with Matplotlib**
>
> `plt.imshow(array)` renders the NumPy array as an image in the notebook.  
> Matplotlib expects the channel order to be **RGB** (Red, Green, Blue), which is exactly what Pillow gives us.
>
> **What you should see**: the puppy photo displayed inline in the notebook.


In [ ]:
pic_red = pic_arr.copy()

In [ ]:
pic_red

In [ ]:
pic_red[:,:,0].shape

In [ ]:
# R G B

# RED CHANNEL VALUES 0 no red, 255 full red
#It is showing red intensity as brightness.
plt.imshow(pic_red[:,:,0], cmap='gray') # show the red channel in grayscale -> the more red, the brighter the pixel

> **🔴 Visualizing individual color channels**
>
> A color image has **3 channels stacked together**: `[:,:,0]` = Red, `[:,:,1]` = Green, `[:,:,2]` = Blue.
>
> When we display a single channel with `cmap='gray'`, each pixel's value (0–255) is shown as brightness:
> - **Bright (white)** → high value in that channel → that area has a lot of that color.
> - **Dark (black)** → low value → little or none of that color.
>
> For example, a red object will appear **bright** in the Red channel and **dark** in Green & Blue channels.


In [ ]:
# GREEN CHANNEL VALUES 0 no green, 255 full green
#It is showing green intensity as brightness.
plt.imshow(pic_red[:,:,1], cmap='gray') # show the green channel in grayscale -> the more green, the brighter the pixel

In [ ]:
# BLUE CHANNEL VALUES 0 no blue, 255 full blue
#It is showing blue intensity as brightness.
plt.imshow(pic_red[:,:,2], cmap='gray') # show the blue channel in grayscale -> the more blue, the brighter the pixel

In [ ]:
# GREEN CHANNEL
pic_red[:,:,1] = 0

In [ ]:
# BLUE CHANNEL
pic_red[:,:,2] = 0

In [ ]:
plt.imshow(pic_red)

> **🔴 Result — Red-only image**
>
> Cells 23 and 24 set the Green and Blue channels entirely to `0`, leaving only the Red channel intact.  
> The result is the image displayed with **only red tones** — all greens and blues have been removed.
>
> This demonstrates how color images are just 3 stacked 2D arrays that you can manipulate independently.


### Image conventions to remember

- OpenCV images are NumPy arrays
- shapes are typically `(H, W)` for grayscale and `(H, W, 3)` for color
- OpenCV loads color as **BGR**; Matplotlib displays as **RGB**
- `dtype` and value ranges matter (`uint8` in `[0,255]` is common)


## OpenCV basics

OpenCV is a computer vision library (originally C++, with Python bindings).

A few practical notes:

- `cv2.imread(...)` returns a NumPy array (or `None` if the path is wrong).
- OpenCV loads color images as **BGR** by default, while Matplotlib expects **RGB**.
- Use `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` before `plt.imshow(...)` when displaying OpenCV images with Matplotlib.

Common functions:

- `cv2.cvtColor` convert between color spaces
- `cv2.resize` change image size (either explicit size or scale factors)
- `cv2.flip` flip horizontally/vertically
- `cv2.imwrite` save an image to disk

Docs: https://opencv.org/


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import cv2

In [ ]:
img = cv2.imread('data/00-puppy.jpg')

In [ ]:
img.shape

In [ ]:
plt.imshow(img)

In [ ]:
# OpenCV reads images in BGR format, so we need to convert it to RGB for correct display with Matplotlib
# MATPLOTLIB ---->  RED,GREEN,BLUE
# OPENCV -------->  BLUE,GREEN,RED

> **⚠️ Important: OpenCV uses BGR, Matplotlib uses RGB**
>
> When you load an image with OpenCV (`cv2.imread`), the color channels are stored in **BGR** order (Blue, Green, Red) — the *opposite* of RGB.
>
> Matplotlib, on the other hand, expects **RGB** order.
>
> **Consequence**: if you display an OpenCV image directly with `plt.imshow`, the red and blue channels are swapped — the image will look color-distorted (reddish objects appear blue, and vice versa).
>
> **Fix**: always convert before displaying:
> ```python
> fix_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
> plt.imshow(fix_img)
> ```


In [ ]:
fix_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

In [ ]:
plt.imshow(fix_img)

> **✅ Result — Correct RGB display**
>
> After `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)`, the channels are reordered from BGR → RGB.  
> The image now displays with its natural colors in Matplotlib. Compare this with cell 32 to see the difference.


In [ ]:
# Read the image in grayscale
img_gray = cv2.imread('data/00-puppy.jpg', cv2.IMREAD_GRAYSCALE)

In [ ]:
plt.imshow(img_gray, cmap='gray')

> **⬛ Grayscale images**
>
> `cv2.IMREAD_GRAYSCALE` tells OpenCV to load the image directly as a **grayscale** (single-channel) image.  
> The array shape becomes `(H, W)` instead of `(H, W, 3)` — no color channels.
>
> Each pixel holds a single intensity value in `[0, 255]`:  
> - `0` = black  
> - `255` = white  
> - values in between = shades of gray
>
> We must pass `cmap='gray'` to Matplotlib; otherwise it applies a default color map and the image looks falsely colorized.


In [ ]:
plt.imshow(fix_img)

In [ ]:
fix_img.shape

## Resize

Resizing changes the image resolution. You can specify a target `(width, height)` or scale factors.

Tip: for shrinking images, interpolation like `cv2.INTER_AREA` is often a good default; for enlarging, `cv2.INTER_LINEAR` is common.


In [ ]:
new_img = cv2.resize(fix_img, (1000, 400))

In [ ]:
plt.imshow(new_img)

In [ ]:
w_ratio = 0.8
h_ratio = 0.2 # 20% of original height

In [ ]:
ratio_img = cv2.resize(fix_img, (0,0), fix_img, w_ratio, h_ratio)

In [ ]:
plt.imshow(ratio_img)

> **📏 Resizing images**
>
> `cv2.resize(src, (width, height))` changes the image dimensions.  
> Note: OpenCV uses **(width, height)** order, while NumPy shapes are **(height, width)** — easy to mix up!
>
> - **Cell 41**: resized to a fixed `1000×400` pixels (may distort the aspect ratio).
> - **Cells 43–45**: resized using ratio factors (`w_ratio=0.8`, `h_ratio=0.2`). Passing `(0,0)` as the target size tells OpenCV to use the `fx` and `fy` scale factors instead.  
>   Here `h_ratio=0.2` means the new height is only 20% of the original → the image looks squashed vertically.
>
> **Common interpolation choices**:
> - `cv2.INTER_AREA` — good for shrinking (less aliasing)
> - `cv2.INTER_LINEAR` — default, good for enlarging


In [ ]:
ratio_img.shape

## Flip image

Flipping mirrors the image:

- `flipCode = 0` vertical flip
- `flipCode = 1` horizontal flip
- `flipCode = -1` both axes


In [ ]:
flip_img = cv2.flip(fix_img, 1)

In [ ]:
plt.imshow(flip_img)

> **🔄 Flipping images**
>
> `cv2.flip(img, flipCode)` mirrors the image:
>
> | `flipCode` | Effect |
> |---|---|
> | `0` | Vertical flip (upside down) |
> | `1` | Horizontal flip (left ↔ right) |
> | `-1` | Both axes |
>
> **Cell 48** uses `flipCode=1` → horizontal mirror.  
> **Expected result**: the puppy is now facing the opposite direction.


In [ ]:
type(fix_img)

In [ ]:
# Save the modified image
cv2.imwrite('data/totally_new.jpg', fix_img)

> **💾 Saving images**
>
> `cv2.imwrite(filename, img)` saves the NumPy array to disk as an image file.  
> The format is inferred from the extension (`.jpg`, `.png`, etc.).
>
> ⚠️ Remember: OpenCV saves images in **BGR** format.  
> Here `fix_img` was already converted to RGB, so if you reload it with OpenCV it would look color-swapped.  
> For saving, it is best practice to keep the image in BGR (the OpenCV native format) before writing.


In [ ]:
fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot(111)
ax.imshow(fix_img)

## Helpers

Many beginner notebooks break because `sample.jpg` is missing. We'll add small helper utilities:
- Load from local path if it exists
- Otherwise generate a synthetic image (shapes + gradients)

This keeps every later section runnable without external files.


In [ ]:
from pathlib import Path

def bgr_to_rgb(img_bgr):
    if img_bgr is None:
        return None
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

def show(img, title=None, cmap=None, figsize=(6, 4)):
    plt.figure(figsize=figsize)
    if title:
        plt.title(title)
    if img.ndim == 2:
        plt.imshow(img, cmap=cmap or 'gray')
    else:
        # Expect RGB for display
        plt.imshow(img)
    plt.axis('off')
    plt.show()

def make_synthetic_image(h=360, w=540):
    y = np.linspace(0, 1, h, dtype=np.float32)[:, None]
    x = np.linspace(0, 1, w, dtype=np.float32)[None, :]

    r = (255 * np.tile(x, (h, 1))).astype(np.uint8)
    g = (255 * np.tile(y, (1, w))).astype(np.uint8)
    b = (255 * (1 - x * y)).astype(np.uint8)

    img = np.dstack([r, g, b])

    img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    cv2.circle(img_bgr, (int(w * 0.2), int(h * 0.35)), 55, (0, 255, 255), -1) # -1 means filled circle
    cv2.rectangle(img_bgr, (int(w * 0.55), int(h * 0.2)),
                  (int(w * 0.9), int(h * 0.45)), (255, 0, 0), -1)
    cv2.line(img_bgr, (20, h - 30), (w - 20, h - 30), (255, 255, 255), 3)
    cv2.putText(img_bgr, 'CV COURSE', (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2,
                (20, 20, 20), 3, cv2.LINE_AA)

    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

def load_image_rgb(path='sample.jpg'):
    p = Path(path)
    if p.exists():
        img_bgr = cv2.imread(str(p), cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise ValueError(f'Failed to read image: {p}')
        return bgr_to_rgb(img_bgr)
    return make_synthetic_image()


> **🛠️ Helper utilities**
>
> These three utility functions will be reused throughout the notebook:
>
> - **`bgr_to_rgb(img_bgr)`** — converts an OpenCV BGR image to RGB so Matplotlib can display it correctly.
> - **`show(img, title, ...)`** — a wrapper around `plt.imshow` that handles both grayscale and color images automatically, and hides the axis ticks for cleaner display.
> - **`make_synthetic_image(h, w)`** — generates a synthetic gradient image when no real image file is available, so that the notebook remains runnable even without external data files.


In [ ]:
img_path = "images/sample.png"
img_rgb = load_image_rgb(img_path)
show(img_rgb, title='Image (RGB)')
print('shape:', img_rgb.shape, 'dtype:', img_rgb.dtype, 'min/max:', img_rgb.min(), img_rgb.max())

## Crop

Key concepts:
- Shape conventions: `(H, W)` for grayscale, `(H, W, 3)` for color
- `dtype` and value range: `uint8` in `[0,255]` is common
- Coordinates: `(row, col)` = `(y, x)`
- Copy vs view (when slicing)


In [ ]:
# Grayscale conversion
gray = cv2.cvtColor(cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR), cv2.COLOR_BGR2GRAY)
show(gray, title='Grayscale')

print('gray shape:', gray.shape)
print('top-left 5x5:', gray[:5, :5])

# Crop (y1:y2, x1:x2)
crop = img_rgb[60:220, 80:280]
show(crop, title='Crop')

# IMPORTANT: crop is a view in NumPy; modifying it can modify the original
crop_copy = crop.copy()
crop_copy[:] = 0
print('Original min after editing crop_copy:', img_rgb.min())

> **✂️ Cropping and grayscale conversion**
>
> **Cropping** in NumPy is just **array slicing**:
> ```python
> crop = img_rgb[60:220, 80:280]   # rows 60–219, columns 80–279
> ```
> The slice notation is `[y_start:y_end, x_start:x_end]`.
>
> ⚠️ **View vs Copy**: NumPy slices are *views*, not copies.  
> Modifying `crop` would also modify `img_rgb`!  
> Use `.copy()` when you need an independent copy (as shown with `crop_copy`).
>
> **Grayscale conversion** uses:
> ```python
> gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
> ```
> The image goes from shape `(H, W, 3)` → `(H, W)`.  
> The output `gray[:5, :5]` prints the raw pixel intensity values of the top-left 5×5 region.


## Exercise

1) Create a function `center_crop(img, size)` that returns a square crop of shape `(size, size, 3)` from the center.
2) Display the crop.
3) Print its shape and dtype.

Hint: compute the center `(cy, cx)` then slice.


### Solution

This is a reference solution for `center_crop(img, size)`. It includes a small safety check (so it still works if `size` is larger than the image).


In [ ]:
def center_crop_solution(img, size):
    h, w = img.shape[:2]
    size = int(size)
    size = max(1, min(size, h, w))
    top = (h - size) // 2
    left = (w - size) // 2
    return img[top:top + size, left:left + size].copy()

cropped = center_crop_solution(img_rgb, 200)
show(cropped, title='Center crop (solution)')
print(cropped.shape, cropped.dtype)

> **🎯 Exercise solution — `center_crop`**
>
> The function computes the center of the image and slices a square region of size `size×size` around it:
> ```
> top  = (height - size) // 2
> left = (width  - size) // 2
> crop = img[top : top+size, left : left+size]
> ```
>
> The safety check `size = max(1, min(size, h, w))` prevents requesting a crop larger than the image.  
> `.copy()` ensures the returned crop is independent from the original array.  
> **Expected output**: a `(200, 200, 3)` array of `uint8`.


## Quiz

1. What is a pixel?
2. What is the difference between RGB and grayscale?
3. Why do we preprocess images?

## Exercises

- Load another image and display it
- Convert it to grayscale
- Print its shape
- Resize it to 200×200

## Mini Project – Image Explorer

Create a notebook that:
- Loads an image
- Displays original, grayscale, and resized versions
- Prints pixel statistics (min, max, mean)

### Drawing on images

These functions draw directly onto the image array (often in-place):

- `cv2.rectangle` draw rectangles
- `cv2.circle` draw circles (use `thickness=-1` to fill)
- `cv2.line` draw line segments
- `cv2.putText` draw text
- `cv2.polylines` draw polygons from a list of vertices

Notes:

- OpenCV colors are typically **BGR** (e.g. green is `(0, 255, 0)`).
- If you need to keep the original image unchanged, draw on a copy (`img.copy()`).
- Callback functions let you react to mouse/keyboard events in an OpenCV window.


This function is a mouse callback used with OpenCV to draw a rectangle interactively.

You click to set a start point, drag to preview the rectangle, and release to finalize it.


In [ ]:
import cv2
import numpy as np

import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
blank_img = np.zeros(shape=(512, 512, 3), dtype=np.int16)

In [ ]:
blank_img.shape

In [ ]:
blank_img

In [ ]:
plt.imshow(blank_img)

> **⬛ Creating a blank canvas**
>
> `np.zeros(shape=(512, 512, 3), dtype=np.int16)` creates a 512×512 black image with 3 channels.  
> All pixel values are `0` (black).
>
> We use `dtype=np.int16` here to allow for negative intermediate values during drawing.  
> In practice most OpenCV drawing functions work with `uint8` — be mindful of the dtype.
>
> **Expected output**: a completely black square image.


In [ ]:
cv2.rectangle(blank_img, pt1=(384,0), pt2=(500,150),color=(0,255,0), thickness=10)

In [ ]:
plt.imshow(blank_img)

> **🟩 Drawing a rectangle**
>
> `cv2.rectangle(img, pt1, pt2, color, thickness)` draws a rectangle on the image **in place** (it modifies the array directly).
>
> | Parameter | Meaning |
> |---|---|
> | `pt1` | Top-left corner `(x, y)` |
> | `pt2` | Bottom-right corner `(x, y)` |
> | `color` | BGR color tuple `(B, G, R)` — here `(0, 255, 0)` = green |
> | `thickness` | Pixel width of the border; `-1` fills the shape |
>
> **Expected result**: a green rectangle outline on the black canvas.


In [ ]:
cv2.rectangle(blank_img, pt1=(200,200), pt2=(300,300), color=(0,0,255), thickness=10)

In [ ]:
plt.imshow(blank_img)

In [ ]:
cv2.circle(img=blank_img, center=(100,100), radius=50, color=(255,0,0), thickness=8)

In [ ]:
plt.imshow(blank_img)

> **🔵 Drawing a circle (outline)**
>
> `cv2.circle(img, center, radius, color, thickness)`:
>
> | Parameter | Meaning |
> |---|---|
> | `center` | `(x, y)` of the circle center |
> | `radius` | Circle radius in pixels |
> | `color` | BGR color — here `(255, 0, 0)` = blue in BGR |
> | `thickness` | Stroke width; use `-1` (next cell) to fill |
>
> **Expected result**: a blue hollow circle on the canvas.


In [ ]:
cv2.circle(img=blank_img, center=(400,400), radius=50, color=(255,0,0), thickness=-1)

In [ ]:
plt.imshow(blank_img)

> **🔵 Drawing a filled circle**
>
> `thickness=-1` fills the entire circle with the specified color.  
> Compare this with the outline circle in cell 82.
>
> **Expected result**: a solid blue filled circle in the bottom-right area of the canvas.


In [ ]:
cv2.line(blank_img, pt1=(0,0), pt2=(512,512), color=(102,255,255), thickness=5)

In [ ]:
plt.imshow(blank_img)

> **📏 Drawing a line**
>
> `cv2.line(img, pt1, pt2, color, thickness)` draws a straight line from `pt1` to `pt2`.  
> Here `(0,0) → (512,512)` draws a diagonal from the top-left to the bottom-right corner.
>
> **Expected result**: a cyan diagonal line across the canvas.


In [ ]:
font = cv2.FONT_HERSHEY_SIMPLEX
cv2.putText(blank_img, text='Hello', org=(10,500), fontFace=font, 
            fontScale=4, color=(255,255,255), thickness=3, lineType=cv2.LINE_AA)
plt.imshow(blank_img)

> **🔤 Drawing text on an image**
>
> `cv2.putText(img, text, org, fontFace, fontScale, color, thickness, lineType)`:
>
> | Parameter | Meaning |
> |---|---|
> | `org` | Bottom-left corner of the text string `(x, y)` |
> | `fontFace` | Font family (e.g. `cv2.FONT_HERSHEY_SIMPLEX`) |
> | `fontScale` | Font size multiplier |
> | `lineType=cv2.LINE_AA` | Anti-aliased edges — smoother text |
>
> **Expected result**: white text "Hello" at the bottom of the canvas.


In [ ]:
blank_img = np.zeros(shape=(512,512,3), dtype=np.int32)

In [ ]:
plt.imshow(blank_img)

In [ ]:
vertices = np.array([ [100,300], [200,200], [400,300], [200,400] ], dtype=np.int32)

In [ ]:
pts = vertices.reshape((-1,1,2))

In [ ]:
pts

In [ ]:
cv2.polylines(blank_img, [pts], isClosed=True, color=(255,0,0), thickness=5)
plt.imshow(blank_img)

> **🔷 Drawing polygons**
>
> `cv2.polylines(img, [pts], isClosed, color, thickness)` connects a sequence of points with line segments.
>
> The vertices array must be reshaped to `(-1, 1, 2)` — OpenCV requires this specific shape for contour/polygon functions.
>
> | Parameter | Meaning |
> |---|---|
> | `isClosed=True` | Connects the last point back to the first |
> | `isClosed=False` | Leaves the polygon open |
>
> **Expected result**: a blue quadrilateral (diamond shape) drawn on the canvas.


#### Connecting callback functions

OpenCV windows can trigger event callbacks (especially mouse events). A common pattern is:

1) create a window
2) register a callback with `cv2.setMouseCallback(...)`
3) run a loop that displays the image and listens for keys


This callback reacts to three mouse events:

- `cv2.EVENT_LBUTTONDOWN` (left button pressed)
  - start drawing and store the initial point `(ix, iy)`
- `cv2.EVENT_MOUSEMOVE` (mouse moves)
  - while drawing, update the preview rectangle from `(ix, iy)` to `(x, y)`
- `cv2.EVENT_LBUTTONUP` (left button released)
  - stop drawing and finalize the rectangle

Key variables:

- `ix, iy` initial click position
- `drawing` boolean flag while dragging
- colors are BGR in OpenCV (e.g. `(0, 255, 0)` is green)


In [ ]:
import cv2
import numpy as np

drawing = False
ix, iy = -1, -1


def draw_rectangle(event, x, y, flags, param):
    
    global ix, iy, drawing
    
    if event == cv2.EVENT_LBUTTONDOWN:
        
        drawing = True
        ix,iy = x,y
        
    elif event == cv2.EVENT_MOUSEMOVE:
        if drawing == True:
            cv2.rectangle(img, (ix, iy), (x,y), (0,255,0), -1)
            
    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False
        cv2.rectangle(img, (ix, iy), (x,y), (0,255,0), -1)
            

# SHOWING THE IMAGE
img = np.zeros((512, 512, 3))

cv2.namedWindow(winname='my_drawing')

cv2.setMouseCallback('my_drawing', draw_rectangle)

while True:
    
    cv2.imshow('my_drawing', img)
    
    # Press q to quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
        
cv2.destroyAllWindows()

> **🖱️ Interactive drawing with mouse callbacks (draw_rectangle)**
>
> This pattern requires a real OpenCV window (not available in Jupyter — run it as a standalone Python script).
>
> **How it works:**
> 1. `cv2.namedWindow(...)` creates a window.
> 2. `cv2.setMouseCallback(window, function)` registers `draw_rectangle` to be called on mouse events.
> 3. The callback receives `(event, x, y, flags, param)` on every mouse action.
>
> The three events handled:
> - `EVENT_LBUTTONDOWN` → start drawing, record start point `(ix, iy)`
> - `EVENT_MOUSEMOVE` → while dragging, update the rectangle preview
> - `EVENT_LBUTTONUP` → stop drawing, finalize the rectangle
>
> `drawing = True/False` is a flag that tracks whether the user is currently dragging.


The next callback is similar, but uses different mouse buttons to draw circles in different colors.


In [ ]:
import cv2
import numpy as np


def draw_circle(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        cv2.circle(img, (x,y), 100, (0,255,0), -1)
    elif event == cv2.EVENT_RBUTTONDOWN:
        cv2.circle(img, (x,y), 100, (255,0,0), -1)

cv2.namedWindow(winname='my_drawing')

cv2.setMouseCallback('my_drawing', draw_circle)


img = np.zeros((512,512,3))

while True:
    cv2.imshow('my_drawing', img)
    
    if cv2.waitKey(20) & 0xFF == 27:
        break
        
cv2.destroyAllWindows()

> **🖱️ Mouse callback — draw circles**
>
> Similar to the rectangle callback but simpler:
> - **Left click** → draw a green filled circle
> - **Right click** → draw a blue filled circle
>
> `cv2.waitKey(20) & 0xFF == 27` waits 20 ms per frame for a key press; `27` is the ASCII code for the **Escape** key, which exits the loop.  
> `cv2.destroyAllWindows()` closes all OpenCV windows cleanly.
>
> ⚠️ Run this as a standalone `.py` script — it will not work inside Jupyter without a display server.


## Image processing with OpenCV

This section covers the classic building blocks you will reuse in most CV pipelines.


### Color spaces

RGB is common for display, but other color spaces can make tasks easier. Two popular choices:

- HSL: hue, saturation, lightness
- HSV: hue, saturation, value

Why convert?

- separating brightness from color can simplify thresholding/segmentation
- some tasks are more stable in HSV/HSL than raw RGB

In OpenCV you typically use `cv2.cvtColor(img, ...)` (remember the input is usually BGR).


In [ ]:
import cv2
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
img = cv2.imread('data/00-puppy.jpg')

In [ ]:
plt.imshow(img)

In [ ]:
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
plt.imshow(img)

> **🎨 Converting BGR → RGB for correct Matplotlib display**
>
> `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` reorders the channels from `[B, G, R]` to `[R, G, B]`.  
> Without this conversion, reds and blues appear swapped in Matplotlib.
>
> **Expected result**: the puppy photo with natural colors.


In [ ]:
img = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
plt.imshow(img)

> **🌈 HSV color space**
>
> HSV stands for **Hue, Saturation, Value**:
>
> | Channel | Meaning | Range (OpenCV) |
> |---|---|---|
> | **H** — Hue | The "color" (red, green, blue…) | `[0, 179]` |
> | **S** — Saturation | Color purity/intensity | `[0, 255]` |
> | **V** — Value | Brightness | `[0, 255]` |
>
> **Why use HSV?**  
> In RGB, "green" is a combination of all three values. In HSV, green is just a range of Hue values.  
> This makes it much easier to segment objects by color using a simple `cv2.inRange(...)` threshold.
>
> **Expected result**: the image looks strange/psychedelic — that's expected! Matplotlib interprets the HSV values as RGB, which is why the colors look distorted. We're just inspecting the raw channel data here.


In [ ]:
img = cv2.imread('data/00-puppy.jpg')
img = cv2.cvtColor(img, cv2.COLOR_BGR2HLS)
plt.imshow(img)

> **💡 HLS color space**
>
> HLS stands for **Hue, Lightness, Saturation** — similar to HSV but uses *lightness* instead of *value*.
>
> Both HSV and HLS separate brightness from color information, but they define "brightness" slightly differently:
> - **HSV Value**: the maximum of R, G, B
> - **HLS Lightness**: the average of the max and min of R, G, B
>
> In practice, HSV is more commonly used for color-based segmentation tasks in OpenCV.


In [ ]:
img = cv2.imread('data/00-puppy.jpg')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)

H, S, V = cv2.split(hsv)
L, A, B = cv2.split(lab)

show(img_rgb, title='RGB')
show(H, title='HSV: H channel')
show(S, title='HSV: S channel')
show(V, title='HSV: V channel')
show(L, title='LAB: L channel')

### Blending and masking

Blending combines two images of the same size:

`new_pixel = a * pixel_1 + b * pixel_2 + y`

In OpenCV, `cv2.addWeighted(...)` implements this efficiently.

Masking selects which pixels to keep:

- white/1 means keep
- black/0 means ignore

Masks are the basic tool behind many segmentation and compositing operations.


In [ ]:
import cv2
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
img1 = cv2.imread('data/dog_backpack.jpg')
img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
img2 = cv2.imread('data/watermark_no_copy.png')
img2 = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)

In [ ]:
plt.imshow(img1)

In [ ]:
plt.imshow(img2)

In [ ]:
img1.shape

In [ ]:
img2.shape

In [ ]:
# BLENDING IMAGE OF THE SAME SIZE

In [ ]:
img1 = cv2.resize(img1, (1200, 1200))
img2 = cv2.resize(img2, (1200, 1200))

In [ ]:
blended = cv2.addWeighted(src1=img1, alpha = 0.8, src2=img2, beta = 0.1, gamma=0)

> **🔀 Blending two images — `cv2.addWeighted`**
>
> `cv2.addWeighted(src1, alpha, src2, beta, gamma)` blends two images with the formula:
>
> ```
> output = alpha × src1 + beta × src2 + gamma
> ```
>
> | Parameter | Value used | Effect |
> |---|---|---|
> | `alpha=0.8` | 80% of img1 | img1 is dominant |
> | `beta=0.1` | 10% of img2 | img2 is subtle (watermark-like) |
> | `gamma=0` | No brightness shift | |
>
> ⚠️ Both images must be the **same size** — that's why cell 118 resizes them both to `1200×1200` first.
>
> **Expected result**: the dog image with a faint watermark overlaid on top.


In [ ]:
plt.imshow(blended)

In [ ]:
# OVERLAY SMALL IMAGE ON TOP OF A LARGE IMAGE ( NO BLENDING ) using numpy

In [ ]:
img1 = cv2.imread('data/dog_backpack.jpg')
img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
img2 = cv2.imread('data/watermark_no_copy.png')
img2 = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)

In [ ]:
img2 = cv2.resize(img2, (600,600))

In [ ]:
large_img = img1
small_img = img2

In [ ]:
# TOP LEFT CORNER
x_offset = 0
y_offset = 0

In [ ]:
x_end = x_offset + small_img.shape[1]
y_end = y_offset + small_img.shape[0]

In [ ]:
large_img[y_offset:y_end, x_offset:x_end] = small_img

In [ ]:
plt.imshow(large_img)

> **📌 Overlaying a small image on a large image (NumPy slicing)**
>
> Instead of blending, here we **directly replace** a region of the large image with the small image:
> ```python
> large_img[y_offset:y_end, x_offset:x_end] = small_img
> ```
>
> This is a hard paste — no transparency, no blending.  
> The watermark fully replaces the top-left corner of the dog image.
>
> **Expected result**: the small watermark image pasted sharply in the top-left corner of the dog photo.


In [ ]:
# BLEND DIFFERENT SIZE IMAGES

In [ ]:
img1 = cv2.imread('data/dog_backpack.jpg')
img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
img2 = cv2.imread('data/watermark_no_copy.png')
img2 = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)

In [ ]:
img2 = cv2.resize(img2, (600,600))

In [ ]:
x_offset = img1.shape[1] - img2.shape[1]
y_offset = img1.shape[0] - img2.shape[0]

In [ ]:
rows, cols, channels = img2.shape

In [ ]:
roi = img1[y_offset:img1.shape[0], x_offset:img1.shape[1]]

In [ ]:
plt.imshow(roi)

In [ ]:
img2gray = cv2.cvtColor(img2, cv2.COLOR_RGB2GRAY)

In [ ]:
plt.imshow(img2gray, cmap='gray')

In [ ]:
# Create a binary mask of the watermark
# new_pixel = 255 - original_pixel
mask_inv = cv2.bitwise_not(img2gray)

In [ ]:
plt.imshow(mask_inv, cmap='gray')

> **🔲 Creating a binary mask with bitwise NOT**
>
> `cv2.bitwise_not(img2gray)` inverts all pixel values: `new_pixel = 255 - original_pixel`.
>
> - Where the watermark text was **black** (0) → becomes **white** (255)
> - Where the background was **white** (255) → becomes **black** (0)
>
> This inverted mask (`mask_inv`) is used in subsequent cells to:
> 1. Extract just the foreground (watermark text) from the watermark image
> 2. Cut a matching hole in the background image
>
> **Expected result**: the mask with white text on a black background.


In [ ]:
import numpy as np

In [ ]:
white_background = np.full(img2.shape, 255, dtype=np.uint8)

In [ ]:
white_background.shape

In [ ]:
# this will put the white background where the watermark is not, and black where the watermark is
# copies pixels only where mask allows
bk = cv2.bitwise_or(src1=white_background, src2=white_background, mask=mask_inv)

# The mask is a single-channel 8-bit image (grayscale), where:

# Pixels = 0 → operation ignored, output stays black (0)

# Pixels ≠ 0 → operation applied, output gets src1 OR src2

# So in this line:

    # Only where mask_inv is white (255) will pixels from white_background appear in bk

    # Where mask_inv is black (0) → pixels in bk stay 0

In [ ]:
plt.imshow(bk)

> **🔧 Using `cv2.bitwise_or` with a mask**
>
> `cv2.bitwise_or(src1, src2, mask=mask_inv)` performs a bitwise OR between `src1` and `src2`, but **only where the mask is non-zero (white)**.
>
> - Where `mask_inv = 255` → the OR operation is applied → `white_background` pixels pass through to `bk`
> - Where `mask_inv = 0` → the output stays black (0)
>
> **Result `bk`**: a white image with black holes exactly where the watermark text is.  
> This will later be combined with the dog photo region to create a transparent-looking watermark effect.


In [ ]:
mask_inv

In [ ]:
fg = cv2.bitwise_or(img2, img2, mask=mask_inv)

In [ ]:
plt.imshow(fg)

In [ ]:
final_roi = cv2.bitwise_or(roi, fg)

In [ ]:
plt.imshow(final_roi)

In [ ]:
large_img = img1
small_img = final_roi

In [ ]:
large_img[y_offset:y_offset+small_img.shape[0], x_offset:x_offset+small_img.shape[1]] = small_img

In [ ]:
plt.imshow(large_img)

### When to threshold

Thresholding works best when the foreground/background have different intensity or color. If lighting varies across the image, adaptive thresholding or converting to another color space (like HSV) often helps.


### Thresholding, blurring, and smoothing

- Thresholding turns an image into a simpler (often binary) image to separate foreground/background.
- Smoothing/blur reduces noise and small details (often helpful before edge detection).
- Gamma correction can make an image appear brighter/darker in a non-linear way.

Kernels (small matrices) are used for many filters (blur, sharpen, edge detect).

Interactive kernel visualizer: http://setosa.io/ev/image-kernels/


In [ ]:
import cv2
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
img = cv2.imread('data/rainbow.jpg', 0)

In [ ]:
plt.imshow(img, cmap='gray')

In [ ]:
# ret is the cutoff value (127),  thresh1 is the image

# How does the thresholding work?

# if pixel_value > 127:
#     pixel_value = 255  # white
# else:
#     pixel_value = 0    # black

ret, thresh1 = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)

In [ ]:
plt.imshow(thresh1, cmap='gray')

> **🔳 Global thresholding**
>
> `cv2.threshold(img, thresh, maxVal, type)` converts a grayscale image to a **binary** image:
>
> ```
> if pixel > 127:  pixel = 255 (white)
> else:             pixel = 0   (black)
> ```
>
> The function returns two values:
> - `ret` — the threshold value used (127 here)
> - `thresh1` — the resulting binary image
>
> **When to use**: works well when the foreground and background have clearly different intensities across the whole image.  
> **Limitation**: a single global threshold fails when lighting is uneven.
>
> **Expected result**: a high-contrast black-and-white version of the rainbow image.


In [ ]:
img = cv2.imread('data/crossword.jpg', 0)

In [ ]:
plt.imshow(img, cmap='gray')

In [ ]:
def show_pic(img):
    fig = plt.figure(figsize=(15,15))
    ax = fig.add_subplot(111)
    ax.imshow(img, cmap='gray')

In [ ]:
show_pic(img)

In [ ]:
ret, th1 = cv2.threshold(img, 180, 255, cv2.THRESH_BINARY)
show_pic(th1)

In [ ]:
# Adaptive thresholding
# how does it work ?

# threshold = mean_of_neighbors - C (8 in our case)
# if pixel_value > threshold:
#     output = maxValue (255) # white
# else:
#     output = 0         # black

th2 = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 11, 8)
show_pic(th2)

> **🔳 Adaptive thresholding**
>
> Unlike global thresholding which uses one cutoff for the whole image, adaptive thresholding **computes a local threshold for each pixel** based on its neighborhood:
>
> ```
> local_threshold = mean(neighbors in 11×11 window) - C
> if pixel > local_threshold: white else: black
> ```
>
> **Parameters explained:**
> - `blockSize=11` — size of the neighborhood window (must be odd)
> - `C=8` — a constant subtracted from the mean (fine-tunes sensitivity)
>
> **Why it works better on uneven images**: even if one part of the image is brighter than another (e.g., a shadow), the local threshold adapts accordingly.
>
> **Expected result**: the crossword text is clearly separated from the background even where lighting varies.


Adaptive thresholding uses a local neighborhood instead of one global cutoff.

Example with a `11x11` neighborhood (`blockSize=11`):

- compute the mean of the local window around a pixel
- subtract `C` (e.g. `8`) to form a local threshold
- apply the binary rule (above -> white, below -> black)


In [ ]:
blended = cv2.addWeighted(src1=th1, alpha=0.6, src2=th2, beta=0.4, gamma=0)
show_pic(blended)

> **🔀 Combining threshold results**
>
> Here we blend the global threshold (`th1`) and adaptive threshold (`th2`) outputs using `cv2.addWeighted`.  
> This is not a standard processing step — it's a visualization technique to see both results side by side in a single image.
>
> **Expected result**: a mix showing both threshold approaches simultaneously — useful for comparison.


In [ ]:
# IAM HERE

# Finir cours
# Finir video utube / voir si ajouter des choses
# project

# machine learning from scratch
# explain theory of machine learning
# simple models from scratch
# project

# explain theory of deep learning
# explain theory of convolutional neural networks
# Convolutional Neural Networks from scratch
# projects

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def load_img():
    img = cv2.imread('data/bricks.jpg').astype(np.float32) / 255
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

In [ ]:
def display_img(img):
    fig = plt.figure(figsize=(12,10))
    ax = fig.add_subplot(111)
    ax.imshow(img)


In [ ]:
i = load_img()
display_img(i)

In [ ]:
# Values below 1 make it brigher, values greater make darker
gamma = 1/4

In [ ]:
result = np.power(i, gamma)

In [ ]:
display_img(result)

> **☀️ Gamma correction**
>
> Gamma correction adjusts the overall brightness of an image using a **power law**:
> ```
> output = pixel_value ^ gamma
> ```
> (applied on a normalized `[0.0, 1.0]` image)
>
> | Gamma | Effect |
> |---|---|
> | `< 1` (e.g. `1/4 = 0.25`) | Brightens the image |
> | `= 1` | No change |
> | `> 1` | Darkens the image |
>
> `np.power(i, gamma)` applies the power function element-wise to every pixel.
>
> **Why it matters**: human vision is non-linear. Gamma correction can compensate for monitor display curves and improve perceived contrast.
>
> **Expected result**: the brick image should look noticeably brighter.


In [ ]:
# LOW PASS FILTER WITH 2D CONVOLUTION

In [ ]:
img = load_img()
font = cv2.FONT_HERSHEY_COMPLEX
cv2.putText(img, text='bricks', org=(10,600), fontFace=font, fontScale=10, color=(255,0,0), thickness=4)

In [ ]:
display_img(img)

In [ ]:
kernel = np.ones(shape=(5,5), dtype=np.float32)/25

In [ ]:
kernel

In [ ]:
dst = cv2.filter2D(img, -1, kernel)
display_img(dst)

> **🔲 2D Convolution / Low-pass filter**
>
> `cv2.filter2D(img, -1, kernel)` convolves the image with a custom kernel.
>
> The kernel used here is a **5×5 box filter** (all values = `1/25`):
> ```
> kernel = np.ones((5,5)) / 25
> ```
> This computes the **average of each pixel and its 24 neighbors** — a simple blur.
>
> **Why blur?** Blurring is a *low-pass filter*: it smooths out high-frequency details (noise, sharp edges).  
> It is often the first step before edge detection or thresholding to reduce noise sensitivity.
>
> **Expected result**: the "bricks" image with text appears softer/blurrier.


In [ ]:
img = load_img()
font = cv2.FONT_HERSHEY_COMPLEX
cv2.putText(img, text='bricks', org=(10,600), fontFace=font, fontScale=10, color=(255,0,0), thickness=4)
print('reset')

In [ ]:
blurred = cv2.blur(img, ksize=(10,10))
display_img(blurred)

> **📦 Simple mean blur — `cv2.blur`**
>
> `cv2.blur(img, ksize=(10, 10))` is a convenient wrapper for the box filter (same idea as cell 181–183 but with a larger `10×10` kernel).  
> Larger kernel → stronger blur.
>
> **Expected result**: a heavily blurred version of the brick image.


In [ ]:
img = load_img()
font = cv2.FONT_HERSHEY_COMPLEX
cv2.putText(img, text='bricks', org=(10,600), fontFace=font, fontScale=10, color=(255,0,0), thickness=4)
print('reset')

In [ ]:
# Gaussian Blur
blurred_img = cv2.GaussianBlur(img, (5,5), 10)
display_img(blurred_img)

> **🌫️ Gaussian blur**
>
> `cv2.GaussianBlur(img, (5,5), sigmaX=10)` blurs using a **Gaussian (bell-shaped) kernel**.
>
> Unlike the box filter (where all neighbors contribute equally), the Gaussian filter gives **more weight to closer pixels** and less weight to distant ones.  
> This produces a smoother, more natural-looking blur.
>
> **Parameters:**
> - `(5, 5)` — kernel size (must be odd)
> - `10` — standard deviation (σ); larger σ = more blur
>
> **When to use**: Gaussian blur is the standard choice before edge detection (Canny, Sobel) because it removes noise without introducing artificial sharp artifacts.


In [ ]:
img = load_img()
font = cv2.FONT_HERSHEY_COMPLEX
cv2.putText(img, text='bricks', org=(10,600), fontFace=font, fontScale=10, color=(255,0,0), thickness=4)
print('reset')

In [ ]:
# Median Blur
# Median Blur is great for removing noise while retaining shape
median = cv2.medianBlur(img, 5)
display_img(median)

> **🔢 Median blur**
>
> `cv2.medianBlur(img, 5)` replaces each pixel with the **median value** of its 5×5 neighborhood.
>
> **Key advantage**: unlike mean/Gaussian blur, median blur is very effective at removing **salt-and-pepper noise** (random bright/dark isolated pixels) while preserving edges better.
>
> **Why?** Extreme noise values (like a single white pixel in a dark area) don't survive the median — they get replaced by the surrounding majority value.
>
> **Expected result**: cells 194 demonstrate this clearly — the noisy dog image is cleaned up while keeping sharp edges.


In [ ]:
img = cv2.imread('data/sammy.jpg')
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

In [ ]:
display_img(img)

In [ ]:
noise_img = cv2.imread('data/sammy_noise.jpg')

In [ ]:
display_img(noise_img)

In [ ]:
median = cv2.medianBlur(noise_img, 5)
display_img(median)

In [ ]:
# Bilateral Filtering
img = load_img()
font = cv2.FONT_HERSHEY_COMPLEX
cv2.putText(img, text='bricks', org=(10,600), fontFace=font, fontScale=10, color=(255,0,0), thickness=4)
print('reset')

In [ ]:
blur = cv2.bilateralFilter(img, 9, 75, 75)
display_img(blur)

> **🎨 Bilateral filter — edge-preserving blur**
>
> `cv2.bilateralFilter(img, d=9, sigmaColor=75, sigmaSpace=75)` blurs the image while **preserving edges**.
>
> It considers two factors for each neighbor pixel:
> - **Spatial distance** (`sigmaSpace`): how far away the neighbor is (like Gaussian blur)
> - **Intensity difference** (`sigmaColor`): how different the neighbor's color/intensity is
>
> Pixels across a strong edge have very different intensities → low weight → edge is preserved.
>
> **When to use**: ideal for skin smoothing, cartoon effects, or any case where you want to blur flat regions but keep sharp boundaries.
>
> **Expected result**: the bricks texture is smoothed but the "bricks" text edges remain sharp.


### Choosing a kernel

The kernel/structuring element controls what morphology removes or preserves. Common choices:

- small (3x3) kernels remove tiny noise
- larger kernels remove/bridge larger structures
- shapes (rect/ellipse) change how corners and thin lines are affected


### Morphological operators

Morphology uses a small binary pattern (a **kernel** / structuring element) to modify shapes in an image (usually a binary mask).

Common uses:

- remove small noise
- fill small gaps
- separate touching objects
- clean up segmentation masks

Core operations:

- erosion: shrinks foreground
- dilation: expands foreground
- opening: erosion then dilation (remove small foreground noise)
- closing: dilation then erosion (fill small holes)

Example:

```python
opening = cv2.morphologyEx(noise_img, cv2.MORPH_OPEN, kernel)
```

Gradient-based operators (e.g. Sobel) estimate changes in intensity and are often used for edge-like responses.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def load_img():
    blank_img = np.zeros((600,600))
    font = cv2.FONT_HERSHEY_SIMPLEX
    cv2.putText(blank_img, text='ABCDE', org=(50,300), fontFace=font, fontScale=5, color=(255,255,255), thickness = 25)
    return blank_img

In [ ]:
def display_img(img):
    fig = plt.figure(figsize=(12,10))
    ax = fig.add_subplot(111)
    ax.imshow(img, cmap='gray')

In [ ]:
img = load_img()
display_img(img)

In [ ]:
kernel = np.ones((5,5), dtype=np.uint8)

In [ ]:
result = cv2.erode(img, kernel, iterations=1)

In [ ]:
display_img(result)

> **🔴 Morphological Erosion**
>
> `cv2.erode(img, kernel, iterations=1)` **shrinks** white (foreground) regions by eroding their borders.
>
> **How it works**: for each pixel, look at its neighborhood defined by the kernel. If *all* neighboring pixels are white → keep it white. If *any* is black → set it to black.
>
> - Thin strokes become thinner
> - Small white regions may disappear entirely
>
> **Expected result**: the "ABCDE" text on the black canvas should appear slightly thinner.


In [ ]:
img = load_img()
display_img(img)

In [ ]:
result = cv2.erode(img, kernel, iterations=4)

In [ ]:
display_img(result)

> **🔴 Erosion with 4 iterations**
>
> Applying erosion 4 times exaggerates the effect.  
> With `iterations=4`, the operation is applied repeatedly → borders erode further with each pass.
>
> **Expected result**: the text is noticeably thinner; thin strokes may have partially disappeared.


In [ ]:
img = load_img()
display_img(img)

In [ ]:
white_noise = np.random.randint(low=0, high=2, size=(600,600))

In [ ]:
display_img(white_noise)

In [ ]:
white_noise = white_noise * 255

In [ ]:
noise_img = white_noise + img

In [ ]:
display_img(noise_img)

In [ ]:
opening = cv2.morphologyEx(noise_img, cv2.MORPH_OPEN, kernel)

In [ ]:
display_img(opening)

> **🧹 Morphological Opening — removing noise**
>
> `cv2.morphologyEx(noise_img, cv2.MORPH_OPEN, kernel)` performs **opening** = erosion followed by dilation.
>
> **Effect**: small isolated white noise pixels are removed, while larger structures (the text) are preserved.
>
> In cell 211–215, random binary noise was added to the image (white random pixels scattered everywhere).  
> Opening eliminates those small noise blobs because:
> - Erosion removes them first (they're too small to survive)
> - Dilation restores the original size of the larger text letters
>
> **Expected result**: the "ABCDE" text is restored cleanly with the noise removed.


In [ ]:
img = load_img()

In [ ]:
black_noise = np.random.randint(low=0, high=2, size=(600,600))

In [ ]:
black_noise = black_noise * -255

In [ ]:
black_noise_img = img+black_noise

In [ ]:
black_noise_img[black_noise_img==-255] = 0

In [ ]:
display_img(black_noise_img)

In [ ]:
closing = cv2.morphologyEx(black_noise_img, cv2.MORPH_CLOSE, kernel)

In [ ]:
display_img(closing)

> **🔒 Morphological Closing — filling holes**
>
> `cv2.morphologyEx(black_noise_img, cv2.MORPH_CLOSE, kernel)` performs **closing** = dilation followed by erosion.
>
> **Effect**: small black holes/gaps inside white foreground regions are filled.
>
> In cells 219–223, black random pixels were subtracted from the white text (creating holes inside the letters).  
> Closing fills those small holes because:
> - Dilation expands white regions, covering the holes
> - Erosion shrinks back to the original size but the holes are now filled
>
> **Expected result**: the "ABCDE" text is restored with holes filled in.


In [ ]:
img = load_img()

In [ ]:
display_img(img)

In [ ]:
gradient = cv2.morphologyEx(img, cv2.MORPH_GRADIENT, kernel)

In [ ]:
display_img(gradient)

> **📐 Morphological Gradient — edge detection**
>
> `cv2.morphologyEx(img, cv2.MORPH_GRADIENT, kernel)` computes the difference between dilation and erosion:
>
> ```
> gradient = dilation(img) - erosion(img)
> ```
>
> This highlights the **boundaries** of objects — the result is essentially an outline of the shapes.
>
> **Expected result**: only the outer edges of the "ABCDE" letters are visible — a clean edge/outline effect.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def display_img(img):
    fig = plt.figure(figsize=(12,10))
    ax = fig.add_subplot(111)
    ax.imshow(img, cmap='gray')

In [ ]:
img = cv2.imread('data/sudoku.jpg', 0)

In [ ]:
display_img(img)

In [ ]:
sobelx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=5)

In [ ]:
display_img(sobelx)

In [ ]:
sobely = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=5)

In [ ]:
display_img(sobely)

In [ ]:
laplacian = cv2.Laplacian(img, cv2.CV_64F)

In [ ]:
display_img(laplacian)

> **📐 Gradient-based edge detection: Sobel & Laplacian**
>
> These filters detect edges by looking at how quickly pixel intensity changes.
>
> **Sobel filter** computes the gradient in one direction:
> - `cv2.Sobel(..., dx=1, dy=0)` → horizontal edges (detects vertical intensity changes)
> - `cv2.Sobel(..., dx=0, dy=1)` → vertical edges (detects horizontal intensity changes)
>
> **Laplacian** computes the second derivative — sensitive to edges in all directions at once, but also more sensitive to noise.
>
> `cv2.CV_64F` stores the output as 64-bit float to capture both positive and negative gradients (negative values would be lost with `uint8`).
>
> **Expected results:**
> - `sobelx`: edges with strong horizontal gradients (vertical lines highlighted)
> - `sobely`: edges with strong vertical gradients (horizontal lines highlighted)
> - `laplacian`: all edges, but potentially noisy


In [ ]:
blended = cv2.addWeighted(src1=sobelx, alpha=0.5, src2=sobely, beta=0.5, gamma=0)

In [ ]:
display_img(blended)

> **🔀 Combining Sobel X and Y**
>
> Blending `sobelx` and `sobely` with equal weights (`alpha=0.5, beta=0.5`) approximates the full gradient magnitude:
> ```
> approx_magnitude ≈ 0.5 × |Gx| + 0.5 × |Gy|
> ```
> (A more accurate formula is `sqrt(Gx² + Gy²)`, but this blended version is a quick visual approximation.)
>
> **Expected result**: edges from both horizontal and vertical directions are visible together — a more complete edge map of the sudoku grid.


In [ ]:
ret, th1 = cv2.threshold(img, 100, 255, cv2.THRESH_BINARY)

In [ ]:
display_img(th1)

In [ ]:
kernel = np.ones((4,4), np.uint8)

In [ ]:
gradient = cv2.morphologyEx(blended, cv2.MORPH_GRADIENT, kernel)

In [ ]:
display_img(gradient)

### Histograms

A histogram shows how frequently values occur. For images, you can plot distributions per channel (0-255 for `uint8`).

Histogram equalization is a contrast-enhancement technique based on redistributing intensities using the histogram.


A histogram counts how many pixels have each intensity value.
- Dark images: histogram concentrated near 0
- Bright images: histogram concentrated near 255
- Low contrast: histogram narrow
- High contrast: histogram spread


In [ ]:
def plot_gray_hist(gray_img, title='Histogram'):
    hist = cv2.calcHist([gray_img], [0], None, [256], [0, 256]).ravel()
    plt.figure(figsize=(6, 3))
    plt.title(title)
    plt.plot(hist)
    plt.xlim(0, 255)
    plt.xlabel('intensity')
    plt.ylabel('count')
    plt.show()

plot_gray_hist(gray, title='Grayscale histogram')

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
dark_horse = cv2.imread('data/horse.jpg')
show_horse = cv2.cvtColor(dark_horse, cv2.COLOR_BGR2RGB)

rainbow = cv2.imread('data/rainbow.jpg')
show_rainbow = cv2.cvtColor(rainbow, cv2.COLOR_BGR2RGB)

blue_bricks = cv2.imread('data/bricks.jpg')
show_bricks = cv2.cvtColor(blue_bricks, cv2.COLOR_BGR2RGB)

In [ ]:
plt.imshow(show_bricks)

In [ ]:
hist_values = cv2.calcHist([blue_bricks], channels=[0], mask=None, histSize=[256], ranges=[0,256])

In [ ]:
hist_values.shape

In [ ]:
# HIST FOR BLUE COLORS
plt.plot(hist_values)

> **📊 Computing and plotting a histogram**
>
> `cv2.calcHist([img], channels, mask, histSize, ranges)` counts how many pixels have each intensity value:
>
> | Parameter | Meaning |
> |---|---|
> | `channels=[0]` | Which channel to analyze (0=Blue in BGR) |
> | `mask=None` | Analyze the full image (no mask) |
> | `histSize=[256]` | Number of bins (one per intensity value 0–255) |
> | `ranges=[0, 256]` | Intensity range to consider |
>
> **Expected result**: a line plot showing the distribution of blue-channel intensities.  
> The blue bricks image should show a peak toward the higher (brighter) blue values.


In [ ]:
hist_values = cv2.calcHist([dark_horse], channels=[0], mask=None, histSize=[256], ranges=[0,256])

In [ ]:
# HIST FOR BLUE COLORS
plt.plot(hist_values)

In [ ]:
img = blue_bricks

In [ ]:
color = ('b','g','r')

for i,col in enumerate(color):
    histr = cv2.calcHist([img], [i], None, [256], [0,256])
    plt.plot(histr, color=col)
    plt.xlim([0,256])
    
plt.title('HISTOGRAM FOR BLUE BRICKS')

> **🌈 Per-channel color histogram**
>
> By looping over all 3 channels and plotting each in its respective color (blue, green, red), you get a full picture of the image's color distribution.
>
> **How to read the plot:**
> - X-axis: intensity value (0 = dark, 255 = bright)
> - Y-axis: number of pixels with that intensity
> - A peak at 200+ in the blue channel confirms the image has many bright blue pixels
>
> The "dark horse" histogram should show most values clustered near 0 (very dark image).


In [ ]:
img = dark_horse
color = ('b','g','r')

for i,col in enumerate(color):
    histr = cv2.calcHist([img], [i], None, [256], [0,256])
    plt.plot(histr, color=col)
    plt.xlim([0, 50])
    plt.ylim([0,500000])
    
plt.title('HISTOGRAM FOR DARK HORSE')

In [ ]:
rainbow = cv2.imread('data/rainbow.jpg')
show_rainbow = cv2.cvtColor(rainbow, cv2.COLOR_BGR2RGB)

In [ ]:
img = rainbow
img.shape

In [ ]:
mask = np.zeros(img.shape[:2], np.uint8)
mask.shape

In [ ]:
plt.imshow(mask, cmap='gray')

In [ ]:
mask[300:400, 100:400] = 255

In [ ]:
plt.imshow(mask, cmap='gray')

In [ ]:
# FOR THE ACTUAL CALCULATION
masked_img = cv2.bitwise_and(img, img, mask=mask)

In [ ]:
# SHOW FOR VISUALIZATION
show_masked_img = cv2.bitwise_and(show_rainbow, show_rainbow, mask=mask)

In [ ]:
plt.imshow(show_masked_img)

> **🎭 Masked region selection**
>
> A **mask** is a grayscale image of the same size as the original where:
> - `255` (white) = include this pixel
> - `0` (black) = exclude this pixel
>
> `cv2.bitwise_and(img, img, mask=mask)` zeroes out all pixels where the mask is black, keeping only the masked region visible.
>
> In cells 265–268, a white rectangle was drawn on a black mask at `[300:400, 100:400]`.  
> **Expected result**: only the masked rectangular band of the rainbow is visible, the rest is black.


In [ ]:
# B G R
hist_mask_values_red = cv2.calcHist([rainbow], channels=[2], mask=mask, histSize =[256], ranges=[0,256])

In [ ]:
hist_values_red = cv2.calcHist([rainbow], channels=[2], mask=None, histSize =[256], ranges=[0,256])

In [ ]:
plt.plot(hist_mask_values_red)
plt.title('RED HISTOGRAM FOR MASKED RAINBOW')

In [ ]:
plt.plot(hist_values_red)
plt.title('RED HISTOGRAM FOR FULL RAINBOW')

In [ ]:
gorilla = cv2.imread('data/gorilla.jpg', 0)

In [ ]:
def display_img(img):
    fig = plt.figure(figsize=(12,10))
    ax = fig.add_subplot(111)
    ax.imshow(img, cmap='gray')

In [ ]:
display_img(gorilla)

In [ ]:
gorilla.shape

In [ ]:
hist_values = cv2.calcHist([gorilla], channels=[0], mask=None, histSize=[256], ranges=[0,256])

In [ ]:
plt.plot(hist_values)

In [ ]:
eq_gorilla = cv2.equalizeHist(gorilla)

In [ ]:
display_img(eq_gorilla)

> **📊 Histogram Equalization**
>
> `cv2.equalizeHist(gray_img)` redistributes pixel intensities so that the histogram becomes more uniform across the full `[0, 255]` range.
>
> **Effect**: increases contrast in dark or low-contrast images by stretching out the intensity distribution.
>
> **How it works**: it uses the cumulative distribution function (CDF) of the histogram to remap pixel values.
>
> **Expected result**: the gorilla image (which was likely dark/low-contrast) should appear with more visible detail and higher contrast after equalization.


In [ ]:
hist_values = cv2.calcHist([eq_gorilla], channels=[0], mask=None, histSize=[256], ranges=[0,256])

In [ ]:
plt.plot(hist_values)

In [ ]:
color_gorilla = cv2.imread('data/gorilla.jpg')

In [ ]:
show_gorilla = cv2.cvtColor(color_gorilla, cv2.COLOR_BGR2RGB)

In [ ]:
display_img(show_gorilla)

In [ ]:
hsv = cv2.cvtColor(color_gorilla, cv2.COLOR_BGR2HSV)

In [ ]:
hsv[:,:,2] = cv2.equalizeHist(hsv[:,:,2])

In [ ]:
eq_color_gorilla = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

In [ ]:
display_img(eq_color_gorilla)

> **🎨 Histogram equalization on color images**
>
> Directly applying `equalizeHist` to all 3 RGB channels distorts colors (each channel is equalized independently).
>
> The correct approach for color images:
> 1. Convert BGR → HSV
> 2. Equalize **only the V (Value/brightness) channel**: `cv2.equalizeHist(hsv[:,:,2])`
> 3. Convert back HSV → RGB
>
> This preserves the original hue and saturation (the "actual color") while only boosting brightness.
>
> **Expected result**: the gorilla image appears brighter and more detailed without unnatural color shifts.


## Contrast Enhancement: Histogram Equalization vs CLAHE

- Histogram equalization spreads intensities globally (can over-amplify noise)
- CLAHE spreads intensities locally with clipping (often better in practice)


In [ ]:
eq = cv2.equalizeHist(gray)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)

show(gray, title='Original gray')
show(eq, title='Equalized')
show(clahe, title='CLAHE')

plot_gray_hist(eq, title='Histogram after equalization')
plot_gray_hist(clahe, title='Histogram after CLAHE')

## Exercise 2 : Color Segmentation in HSV

Goal: create a mask that selects the yellow circle in the synthetic image (or anything yellow-ish in your own image).

Steps:
1) Convert RGB->BGR->HSV
2) Use `cv2.inRange(hsv, lower, upper)`
3) Visualize the mask and the segmented result

Tip: print pixel HSV values by clicking? (We won't use GUI). Instead, inspect a few pixels manually around the region or adjust ranges gradually.


In [ ]:
img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

# TODO: tune these ranges (H in [0,179] in OpenCV)
lower = np.array([20, 80, 80])
upper = np.array([40, 255, 255])
mask = cv2.inRange(hsv, lower, upper)

seg = cv2.bitwise_and(img_rgb, img_rgb, mask=mask)

show(mask, title='Mask')
show(seg, title='Segmented')
print('mask coverage:', mask.mean() / 255.0)

## Geometric Transforms

You will use these constantly in CV pipelines:
- Resize + preserve aspect ratio
- Translation, rotation, affine transform
- Perspective transform (homography) for document scanning / planar objects


In [ ]:
def resize_max_side(img, max_side=512):
    h, w = img.shape[:2]
    scale = max_side / max(h, w)
    if scale >= 1:
        return img.copy()
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))
    return cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

small = resize_max_side(img_rgb, 320)
show(small, title='Resized (keep aspect)')
print('old:', img_rgb.shape, 'new:', small.shape)

In [ ]:
def rotate_about_center(img, angle_deg, scale=1.0):
    h, w = img.shape[:2]
    center = (w / 2, h / 2)
    M = cv2.getRotationMatrix2D(center, angle_deg, scale)
    out = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    return out

rot = rotate_about_center(img_rgb, 25)
show(rot, title='Rotation (25 deg)')

## Perspective Transform (Practical Homography)

When an object is planar (paper, screen, book cover), you can map 4 corners to a rectangle.

We will demonstrate on a synthetic 'document' created by warping a rectangle into perspective, then unwarping it back.


In [ ]:
def draw_document(h=420, w=600):
    doc = np.full((h, w, 3), 245, dtype=np.uint8)
    cv2.rectangle(doc, (50, 60), (w - 50, h - 60), (30, 30, 30), 3)
    cv2.putText(doc, 'INVOICE #123', (80, 130), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (10, 10, 10), 2, cv2.LINE_AA)
    cv2.putText(doc, 'Total: $42.00', (80, 200), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (10, 10, 10), 2, cv2.LINE_AA)
    for i in range(6):
        y = 260 + i * 22
        cv2.line(doc, (80, y), (w - 80, y), (120, 120, 120), 1)
    return cv2.cvtColor(doc, cv2.COLOR_BGR2RGB)

doc_rgb = draw_document()
show(doc_rgb, title='Synthetic document (fronto-parallel)')

# Warp it into a perspective view
doc_bgr = cv2.cvtColor(doc_rgb, cv2.COLOR_RGB2BGR)
h, w = doc_bgr.shape[:2]
src = np.float32([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]])
dst = np.float32([[80, 40], [w - 120, 70], [w - 40, h - 60], [60, h - 20]])
H = cv2.getPerspectiveTransform(src, dst)
warped = cv2.warpPerspective(doc_bgr, H, (w, h), borderMode=cv2.BORDER_CONSTANT, borderValue=(220, 220, 220))
warped_rgb = cv2.cvtColor(warped, cv2.COLOR_BGR2RGB)
show(warped_rgb, title='Document with perspective distortion')

In [ ]:
# Rectify back using the inverse mapping (dst -> src)
H_inv = cv2.getPerspectiveTransform(dst, src)
rectified = cv2.warpPerspective(warped, H_inv, (w, h))
rectified_rgb = cv2.cvtColor(rectified, cv2.COLOR_BGR2RGB)
show(rectified_rgb, title='Rectified back (should look like original)')

## Exercise 3: Build a Simple Document Scanner (Manual Corners)

Use the `warped_rgb` image above. Your goal is to produce a top-down view (rectified) with a *tight crop* around the document.

Steps:
1) Choose 4 corner points on the perspective image (hardcode them)
2) Map to a rectangle of size `(out_w, out_h)`
3) Apply `cv2.getPerspectiveTransform` + `cv2.warpPerspective`

Stretch goal: compute `out_w/out_h` based on distances between corners.


In [ ]:
# TODO: pick points in the warped image coordinate system
# Example format: pts = np.float32([[x1,y1],[x2,y2],[x3,y3],[x4,y4]])
pts = None

# TODO: define output rectangle
out_w, out_h = 500, 360
dst_rect = np.float32([[0, 0], [out_w - 1, 0], [out_w - 1, out_h - 1], [0, out_h - 1]])

# TODO: compute and warp
# H = cv2.getPerspectiveTransform(pts, dst_rect)
# scan = cv2.warpPerspective(cv2.cvtColor(warped_rgb, cv2.COLOR_RGB2BGR), H, (out_w, out_h))
# show(cv2.cvtColor(scan, cv2.COLOR_BGR2RGB), title='Scanned (rectified)')

print('Fill TODOs to run this cell.')

## Filtering, Noise, and Edge Detection

Core idea: many CV algorithms start with smoothing (reduce noise) then compute derivatives (edges).

We will cover:
- Convolution intuition
- Mean / Gaussian / Median / Bilateral filtering
- Gradients: Sobel
- Canny edge detector


In [ ]:
rng = np.random.default_rng(0)
noisy = gray.copy().astype(np.float32)
noisy += rng.normal(0, 18, size=noisy.shape).astype(np.float32)
noisy = np.clip(noisy, 0, 255).astype(np.uint8)

mean_blur = cv2.blur(noisy, (5, 5))
gauss = cv2.GaussianBlur(noisy, (5, 5), 1.2)
median = cv2.medianBlur(noisy, 5)
bilateral = cv2.bilateralFilter(noisy, d=7, sigmaColor=30, sigmaSpace=30)

show(noisy, title='Noisy')
show(mean_blur, title='Mean blur')
show(gauss, title='Gaussian blur')
show(median, title='Median blur')
show(bilateral, title='Bilateral filter')

In [ ]:
g = gauss
sx = cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3)
sy = cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3)
mag = cv2.magnitude(sx, sy)
mag_u8 = np.clip(mag / (mag.max() + 1e-6) * 255, 0, 255).astype(np.uint8)
edges = cv2.Canny(g, threshold1=50, threshold2=120)

show(g, title='Smoothed gray')
show(mag_u8, title='Sobel magnitude')
show(edges, title='Canny edges')

## Thresholding (Binary Segmentation)

Thresholding converts grayscale to black/white.
- Global threshold: one value for the whole image
- Otsu: chooses a value automatically (works when histogram is bimodal)
- Adaptive threshold: different value per local neighborhood (good for uneven lighting)


In [ ]:
g = cv2.GaussianBlur(gray, (5, 5), 0)
_, th_fixed = cv2.threshold(g, 120, 255, cv2.THRESH_BINARY)
_, th_otsu = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
th_adapt = cv2.adaptiveThreshold(g, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 21, 5)

show(g, title='Gray (blurred)')
show(th_fixed, title='Fixed threshold')
show(th_otsu, title='Otsu threshold')
show(th_adapt, title='Adaptive threshold')

## Exercise 4: Canny Threshold Exploration

Try 3 different pairs of thresholds `(t1, t2)` and compare:
- Too low: lots of noise edges
- Too high: missing true edges

Rule of thumb: `t2` is often ~2x to 3x `t1`, but it depends on the image.


In [ ]:
for t1, t2 in [(30, 90), (60, 180), (100, 240)]:
    e = cv2.Canny(gauss, t1, t2)
    show(e, title=f'Canny t1={t1}, t2={t2}', figsize=(5, 3))

## Morphology, Components, and Contours

Binary images become useful when you can clean them and measure shapes.

You will learn:
- Erosion / dilation
- Opening / closing
- Connected components labeling
- Contours, bounding boxes, area, perimeter


In [ ]:
# Make a simple binary mask from the HSV segmentation earlier (or threshold)
img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
mask = cv2.inRange(hsv, np.array([20, 80, 80]), np.array([40, 255, 255]))

kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
opened = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel)

show(mask, title='Raw mask')
show(opened, title='Opening (remove small noise)')
show(closed, title='Closing (fill small holes)')

In [ ]:
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(closed)
print('num_labels (including background):', num_labels)

# Visualize labels (skip background=0)
vis = np.zeros((*labels.shape, 3), dtype=np.uint8)
colors = np.array([[0, 0, 0]] + [list(map(int, rng.integers(0, 255, size=3))) for _ in range(num_labels - 1)], dtype=np.uint8)
vis = colors[labels]
show(vis, title='Connected component labels')

# Print areas
for i in range(1, num_labels):
    x, y, w, h, area = stats[i]
    cx, cy = centroids[i]
    print(f'label {i}: area={area}, bbox=({x},{y},{w},{h}), centroid=({cx:.1f},{cy:.1f})')

In [ ]:
contours, hierarchy = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
overlay = img_rgb.copy()
overlay_bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)

for cnt in contours:
    area = cv2.contourArea(cnt)
    if area < 50:
        continue
    x, y, w, h = cv2.boundingRect(cnt)
    cv2.rectangle(overlay_bgr, (x, y), (x + w, y + h), (0, 255, 0), 2)

show(cv2.cvtColor(overlay_bgr, cv2.COLOR_BGR2RGB), title='Contours bounding boxes')
print('contours found:', len(contours))

## Exercise 5: Shape Features on Synthetic Shapes

We'll generate a binary image with circles and rectangles, then classify each contour by a simple heuristic using:
- Area
- Perimeter
- Circularity: `4*pi*area / perimeter^2`

Task: label each detected object as `circle` or `rectangle` and draw the label on the image.


In [ ]:
canvas = np.zeros((420, 620), dtype=np.uint8)
cv2.circle(canvas, (140, 140), 60, 255, -1)
cv2.circle(canvas, (460, 120), 40, 255, -1)
cv2.rectangle(canvas, (80, 260), (220, 380), 255, -1)
cv2.rectangle(canvas, (360, 260), (560, 360), 255, -1)

# Add small noise
noise = (rng.random(canvas.shape) > 0.995).astype(np.uint8) * 255
canvas_noisy = cv2.bitwise_or(canvas, noise)

kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
clean = cv2.morphologyEx(canvas_noisy, cv2.MORPH_OPEN, kernel)

contours, _ = cv2.findContours(clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
out = cv2.cvtColor(clean, cv2.COLOR_GRAY2BGR)

import math

for cnt in contours:
    area = cv2.contourArea(cnt)
    if area < 200:
        continue
    peri = cv2.arcLength(cnt, True)
    circ = 4 * math.pi * area / (peri * peri + 1e-6)
    x, y, w, h = cv2.boundingRect(cnt)
    label = 'circle' if circ > 0.78 else 'rectangle'
    cv2.rectangle(out, (x, y), (x + w, y + h), (0, 255, 0), 2)
    cv2.putText(out, f'{label} ({circ:.2f})', (x, y - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA)

show(clean, title='Clean binary')
show(cv2.cvtColor(out, cv2.COLOR_BGR2RGB), title='Labeled shapes')

## Template Matching, Keypoints, and Homography

Two common ways to locate an object:
- Template matching (works for same scale/rotation)
- Feature matching (ORB/SIFT-like) + RANSAC homography (works under viewpoint changes for planar objects)

Note: SIFT may not be available in all OpenCV builds. We'll use ORB (free, fast).


In [ ]:
# Build a simple template matching demo using the synthetic image
scene = img_rgb.copy()
scene_gray = cv2.cvtColor(cv2.cvtColor(scene, cv2.COLOR_RGB2BGR), cv2.COLOR_BGR2GRAY)

# Template: a crop around the yellow circle area (adjust if you use your own image)
tmpl = scene_gray[70:190, 40:170]
res = cv2.matchTemplate(scene_gray, tmpl, cv2.TM_CCOEFF_NORMED)
min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(res)
top_left = max_loc
h, w = tmpl.shape

vis = cv2.cvtColor(scene_gray, cv2.COLOR_GRAY2BGR)
cv2.rectangle(vis, top_left, (top_left[0] + w, top_left[1] + h), (0, 255, 0), 2)

show(tmpl, title='Template')
show(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB), title=f'Match (score={max_val:.3f})')
print('best match:', top_left, 'score:', max_val)

In [ ]:
# ORB matching demo (scene vs rotated version)
img1 = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
img2 = cv2.cvtColor(rotate_about_center(img_rgb, 18), cv2.COLOR_RGB2GRAY)

orb = cv2.ORB_create(nfeatures=800)
kp1, des1 = orb.detectAndCompute(img1, None)
kp2, des2 = orb.detectAndCompute(img2, None)

bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = bf.match(des1, des2)
matches = sorted(matches, key=lambda m: m.distance)

vis = cv2.drawMatches(cv2.cvtColor(img1, cv2.COLOR_GRAY2BGR), kp1, cv2.cvtColor(img2, cv2.COLOR_GRAY2BGR), kp2, matches[:40], None, flags=2)
show(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB), title=f'ORB matches: {len(matches)} (showing 40)', figsize=(10, 5))
print('kp1:', len(kp1), 'kp2:', len(kp2), 'matches:', len(matches))

## Motion and Tracking (Intro)

We will simulate a simple video (moving dot) so you can learn motion methods without downloading videos:
- Background subtraction (conceptual)
- Optical flow (Farneback)
- Tracking by detecting the moving object per frame


In [ ]:
def make_moving_dot_video(num_frames=40, h=240, w=320):
    frames = []
    for t in range(num_frames):
        img = np.zeros((h, w), dtype=np.uint8)
        x = int(30 + t * (w - 60) / (num_frames - 1))
        y = int(h * 0.5 + 40 * np.sin(t * 0.25))
        cv2.circle(img, (x, y), 10, 255, -1)
        frames.append(img)
    return frames

frames = make_moving_dot_video()
show(frames[0], title='Frame 0')
show(frames[len(frames)//2], title='Middle frame')
show(frames[-1], title='Last frame')

In [ ]:
prev = frames[10]
nxt = frames[11]
flow = cv2.calcOpticalFlowFarneback(prev, nxt, None, 0.5, 3, 15, 3, 5, 1.2, 0)
fx, fy = flow[..., 0], flow[..., 1]
mag = np.sqrt(fx * fx + fy * fy)
mag_u8 = np.clip(mag / (mag.max() + 1e-6) * 255, 0, 255).astype(np.uint8)
show(prev, title='Prev')
show(nxt, title='Next')
show(mag_u8, title='Flow magnitude')
print('mean flow magnitude:', float(mag.mean()))

## Exercise 6: Track the Moving Dot

For each frame:
1) Threshold to get a binary mask
2) Find the centroid (mean x/y of white pixels)
3) Store the trajectory and plot it

Optional: draw the trajectory on the last frame.


### Video loop checklist

- Always call `cap.release()` when done
- Use `cv2.waitKey(...)` to process UI events
- If the camera stays busy, restart the kernel and close other apps using the webcam


## Video basics

Working with video means reading frames in a loop and responding to keys/events in real time.


Important: avoid having multiple notebooks/kernels trying to access the same webcam at the same time. Close any other camera-using process first.


In [ ]:
# check the main python file :
# python3 main.py --type="video"
# testVideo.mp4

In [ ]:
# check the main python file :
# python3 main.py --type="interactive_draw"

## Object detection

Here we'll look at classical (non-deep-learning) ways to locate objects or patterns in an image.


### Template matching

Template matching looks for a small template inside a larger image by sliding it over every possible position and scoring how well it matches.

Notes:

- it works best when the object appears at the same scale/rotation as the template
- the main choice is the matching method (a correlation-like score)


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
full = cv2.imread('data/sammy.jpg')
full = cv2.cvtColor(full, cv2.COLOR_BGR2RGB)

In [ ]:
plt.imshow(full)

In [ ]:
face = cv2.imread('data/sammy_face.jpg')
face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)

In [ ]:
plt.imshow(face)

In [ ]:
# All the 6 methods for comparison in a list
# Note how we are using strings, later on we'll use the eval() function to convert to function
methods = ['cv2.TM_CCOEFF', 'cv2.TM_CCOEFF_NORMED', 'cv2.TM_CCORR','cv2.TM_CCORR_NORMED', 'cv2.TM_SQDIFF', 'cv2.TM_SQDIFF_NORMED']

In [ ]:
# Here we are going to find the face in the full image using template matching with the 6 different methods, and show the results for each method

# How It Works ?
# 1 - Place the template at top-left of the image
# 2 - Compute similarity score
# 3 - Move template one pixel to the right
# 4 - Repeat for entire image

# It’s like sliding a small window across the image.

for m in methods:
    
    # CREATE A COPY
    full_copy = full.copy()
    method = eval(m)
    
    # TEMPLATE MATCHING
    res = cv2.matchTemplate(full_copy, face, method)
    
    min_val, max_value, min_loc, max_loc = cv2.minMaxLoc(res)
    
    if method in [cv2.TM_SQDIFF, cv2.TM_SQDIFF_NORMED]:
        top_left = min_loc
    else:
        top_left = max_loc

    height, width, channels = face.shape    
    
    bottom_right = (top_left[0]+width, top_left[1] + height)
    
    cv2.rectangle(full_copy, top_left, bottom_right, (255,0,0), 10)
    
    # PLOT AND SHOW THE IMAGES
    plt.subplot(121)
    plt.imshow(res)
    plt.title('HEATMAP OF TEMPLATE MATCHING')
    
    plt.subplot(122)
    plt.imshow(full_copy)
    plt.title('DETECTION OF TEMPLATE')
    plt.suptitle(m)
    
    plt.show()
    
    print('\n')
    print('\n')

> **🔍 Template Matching — how it works**
>
> `cv2.matchTemplate(image, template, method)` slides the template over the image and computes a similarity score at each position.
>
> The result `res` is a **2D score map** — each pixel represents how well the template matches at that location.
>
> **The 6 methods differ in how similarity is computed:**
>
> | Method | Best match |
> |---|---|
> | `TM_CCOEFF` / `TM_CCOEFF_NORMED` | Maximum value (`max_loc`) |
> | `TM_CCORR` / `TM_CCORR_NORMED` | Maximum value (`max_loc`) |
> | `TM_SQDIFF` / `TM_SQDIFF_NORMED` | Minimum value (`min_loc`) |
>
> `NORMED` variants normalize the score to `[0, 1]` — more robust across different image brightnesses.
>
> **Expected result**: a rectangle is drawn on the full image at the location where the face template best matches.


Example (choose a template matching method and run the match):

```python
myMethod = eval('cv2.TM_CCOEFF')
res = cv2.matchTemplate(full_copy, face, myMethod)
```

`res` is a 2D score map: each location contains the match score for placing the template there.


In [ ]:
plt.imshow(res)

### Corner detection

A corner is a point where intensity changes strongly in (at least) two different directions (think: a junction of two edges).

Two common methods:

- Harris: classic detector based on local intensity changes
- Shi-Tomasi: a refinement of Harris that often gives more stable points


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
flat_chess = cv2.imread('data/flat_chessboard.png')
flat_chess = cv2.cvtColor(flat_chess, cv2.COLOR_BGR2RGB)

In [ ]:
plt.imshow(flat_chess)

In [ ]:
gray_flat_chess = cv2.cvtColor(flat_chess, cv2.COLOR_BGR2GRAY)

In [ ]:
plt.imshow(gray_flat_chess, cmap='gray')

In [ ]:
real_chess = cv2.imread('data/real_chessboard.jpg')
real_chess = cv2.cvtColor(real_chess, cv2.COLOR_BGR2RGB)

In [ ]:
plt.imshow(real_chess)

In [ ]:
gray_real_chess = cv2.cvtColor(real_chess, cv2.COLOR_BGR2GRAY)

In [ ]:
plt.imshow(gray_real_chess, cmap='gray')

In [ ]:
gray = np.float32(gray_flat_chess)

In [ ]:
# Harris Corner Detection Algorithm
dst = cv2.cornerHarris(src=gray, blockSize=2, ksize=3, k=0.04)

In [ ]:
# Needed to show the results
dst = cv2.dilate(dst, None)

In [ ]:
# Whenever our HCA is greater than 10 % of our max value, set it to red (for visualization)
flat_chess[dst>0.01 * dst.max()] = [255,0,0] #RGB

In [ ]:
# Can't detect on edge because there is nothing on the edge to see to
plt.imshow(flat_chess)

> **🔵 Harris Corner Detection**
>
> `cv2.cornerHarris(src, blockSize, ksize, k)` detects corners by measuring how much intensity changes in multiple directions.
>
> **Parameters:**
> - `blockSize=2` — neighborhood size considered around each pixel
> - `ksize=3` — Sobel kernel size used internally for derivatives
> - `k=0.04` — Harris detector free parameter (typically 0.04–0.06)
>
> A **corner** is where intensity changes strongly in at least 2 different directions (unlike an edge, which only changes in 1 direction).
>
> `cv2.dilate(dst, None)` enlarges the detected corner spots so they are visible on the image.
>
> `flat_chess[dst > 0.01 * dst.max()] = [255,0,0]` colors all pixels that are corners with red.
>
> **Expected result**: red dots at the internal corners of the chessboard. Note that border corners may be missed (no neighborhood on one side).


In [ ]:
gray = np.float32(gray_real_chess)

dst = cv2.cornerHarris(src=gray, blockSize=2, ksize=3, k=0.04)

In [ ]:
dst = cv2.dilate(dst, None)

In [ ]:
real_chess[dst>0.01 * dst.max()] = [255,0,0] #RGB

In [ ]:
plt.imshow(real_chess)

In [ ]:
# Shi-Tomasi Corner Detection Algorithm

In [ ]:
flat_chess = cv2.imread('data/flat_chessboard.png')
flat_chess = cv2.cvtColor(flat_chess, cv2.COLOR_BGR2RGB)
gray_flat_chess = cv2.cvtColor(flat_chess, cv2.COLOR_BGR2GRAY)

real_chess = cv2.imread('data/real_chessboard.jpg')
real_chess = cv2.cvtColor(real_chess, cv2.COLOR_BGR2RGB)
gray_real_chess = cv2.cvtColor(real_chess, cv2.COLOR_BGR2GRAY)

In [ ]:
# 5 - how many corners to detect
# downside to this method is that it doesn't automatically mark corners
corners = cv2.goodFeaturesToTrack(gray_flat_chess, 64, 0.01, 10)

In [ ]:
corners = np.int32(corners)

In [ ]:
for i in corners:
    x,y = i.ravel()
    cv2.circle(flat_chess, (x,y), 3, (255,0,0), -1)

In [ ]:
plt.imshow(flat_chess)

> **🔵 Shi-Tomasi Corner Detection**
>
> `cv2.goodFeaturesToTrack(gray, maxCorners, qualityLevel, minDistance)` is an improved version of Harris corners.
>
> **Parameters:**
> - `maxCorners=64` — maximum number of corners to return
> - `qualityLevel=0.01` — minimum quality relative to the best corner (filters out weak corners)
> - `minDistance=10` — minimum pixel distance between returned corners (avoids clusters)
>
> **Advantage over Harris**: automatically selects the N best corners; easier to use for tracking applications.
>
> The detected corners are then drawn as red circles using a loop.
>
> **Expected result**: red dots on the flat chessboard at the grid intersections.


In [ ]:
corners = cv2.goodFeaturesToTrack(gray_real_chess, 100, 0.01, 10)

In [ ]:
corners = np.int32(corners)

In [ ]:
for i in corners:
    x,y = i.ravel()
    cv2.circle(real_chess, (x,y), 3, (255,0,0), -1)

In [ ]:
plt.imshow(real_chess)

### Edge detection

Edges are locations with strong intensity change. A popular detector is Canny, which is a multi-stage process:

1) blur (often Gaussian) to reduce noise
2) compute image gradients
3) non-maximum suppression (thin edges)
4) double threshold (strong vs weak edges)
5) hysteresis (keep weak edges connected to strong ones)

Practical tips:

- for high-resolution images, applying a custom blur first can help
- Canny requires choosing low/high thresholds; the right values depend on contrast and noise


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
img = cv2.imread('data/sammy_face.jpg')

In [ ]:
plt.imshow(img)

In [ ]:
edges = cv2.Canny(image=img, threshold1=127, threshold2=127)

In [ ]:
plt.imshow(edges)

> **🔍 Canny Edge Detection**
>
> `cv2.Canny(image, threshold1, threshold2)` is a multi-stage edge detector:
>
> 1. **Gaussian blur** (internal) — reduces noise
> 2. **Gradient computation** — finds intensity changes
> 3. **Non-maximum suppression** — thins edges to 1 pixel
> 4. **Double thresholding**:
>    - Pixels above `threshold2` → definite edges (strong)
>    - Pixels between `threshold1` and `threshold2` → weak edges
>    - Pixels below `threshold1` → discarded
> 5. **Hysteresis** — weak edges are kept only if connected to strong edges
>
> **Effect of threshold values:**
> - Both set to `127` → balanced
> - `threshold1=0, threshold2=255` → very few edges (only very strong ones pass)
> - Low values → many edges including noise


In [ ]:
edges = cv2.Canny(image=img, threshold1=0, threshold2=255)

In [ ]:
plt.imshow(edges)

In [ ]:
# Median pixel value
med_val = np.median(img)
med_val

In [ ]:
# Set the lower threshold to 0 or 70% of the median value, whichever is greater
lower = int(max(0, 0.7*med_val))
# Set the upper threshold to max of 255 or 130% of the median value, whichever is smaller
upper = int(min(255, 1.3*med_val))

In [ ]:
edges = cv2.Canny(image=img, threshold1=lower, threshold2=upper+100)

In [ ]:
plt.imshow(edges)

> **🎯 Automatic threshold selection for Canny**
>
> Instead of guessing thresholds, we compute them from the **median pixel intensity** of the image:
>
> ```python
> lower = max(0, 0.7 × median)
> upper = min(255, 1.3 × median)
> ```
>
> This heuristic adapts to the overall brightness of the image:
> - Dark images → lower thresholds (needed to detect subtle edges)
> - Bright images → higher thresholds
>
> The `+100` on the upper threshold in cell 388 makes it more selective (only catches strong edges).
>
> **Best practice**: always blur slightly before Canny (cell 390) to reduce noise-induced false edges.


In [ ]:
blurred_img = cv2.blur(img, ksize=(5,5))
edges = cv2.Canny(image=blurred_img, threshold1=lower, threshold2=upper+50)

In [ ]:
plt.imshow(edges)

### Grid detection

Grid patterns (checkerboards, dot grids) are commonly used for camera calibration and tracking because they provide many reliable corners/points.

Calibration helps correct lens distortion (radial/tangential), which can improve downstream tasks like measurement and tracking.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
flat_chess = cv2.imread('data/flat_chessboard.png')

In [ ]:
plt.imshow(flat_chess)

In [ ]:
# found is if it was able to find a chessboard
# corners is a list of coordinates that it found a chess board
found, corners = cv2.findChessboardCorners(flat_chess, (7,7))

In [ ]:
cv2.drawChessboardCorners(flat_chess, (7,7), corners, found)
print ('suppressed')

In [ ]:
plt.imshow(flat_chess)

> **♟️ Chessboard corner detection**
>
> `cv2.findChessboardCorners(img, (7,7))` automatically detects a **7×7 grid** of internal corners on a chessboard pattern.
>
> Note: `(7,7)` refers to the number of **inner corners per row/column** — an 8×8 chessboard has 7×7 internal corners.
>
> `cv2.drawChessboardCorners(img, (7,7), corners, found)` draws the detected corners with connecting lines.
>
> **Why is this useful?**  
> Chessboard patterns are the standard tool for **camera calibration** — knowing where these corners are in the image vs. in 3D space lets you estimate lens distortion and camera parameters.


In [ ]:
dots = cv2.imread('data/dot_grid.png')

In [ ]:
plt.imshow(dots)

In [ ]:
found, corners = cv2.findCirclesGrid(dots, (10,10), cv2.CALIB_CB_SYMMETRIC_GRID)
found

In [ ]:
cv2.drawChessboardCorners(dots, (10,10), corners, found)
print('suppressed')

In [ ]:
plt.imshow(dots)

> **🔵 Circular grid detection**
>
> `cv2.findCirclesGrid(img, (10,10), cv2.CALIB_CB_SYMMETRIC_GRID)` detects a 10×10 grid of circles.
>
> Like chessboard corners, circular grids are used for camera calibration — they are often preferred because circle centers can be localized more precisely than corner intersections.
>
> **Expected result**: the 100 detected circle centers are highlighted and connected on the dot grid image.


### Contour detection

Contours represent curves along object boundaries (typically extracted from a binary image).

They are useful for:

- shape analysis (area, perimeter, bounding boxes)
- filtering by size/shape
- separating external vs internal boundaries

Terminology:

- external contours: outer boundary
- internal contours: holes inside an object


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
img = cv2.imread('data/internal_external.png', 0)

In [ ]:
plt.imshow(img, cmap='gray')

In [ ]:
image = img.copy()

In [ ]:
contours, hierarchy = cv2.findContours(img, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)

In [ ]:
contours

In [ ]:
hierarchy

In [ ]:
external_contours = np.zeros(image.shape)

In [ ]:
for i in range(len(contours)):
    # External Contour
    if hierarchy[0][i][3] == -1:
        cv2.drawContours(external_contours, contours, i, 255, -1)

In [ ]:
plt.imshow(external_contours, cmap='gray')

In [ ]:
internal_contours = np.zeros(image.shape)

for i in range(len(contours)):
    # External Contour
    if hierarchy[0][i][3] != -1:
        cv2.drawContours(internal_contours, contours, i, 255, -1)

In [ ]:
plt.imshow(internal_contours, cmap='gray')

> **📐 External vs Internal contours**
>
> `cv2.findContours(img, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)` finds all contours and organizes them into a 2-level hierarchy:
>
> - **Level 0** (top) = external contours (outer boundaries)
> - **Level 1** = internal contours (holes inside objects)
>
> **Understanding `hierarchy[0][i]`**: for each contour `i`, it stores `[next, prev, first_child, parent]`.  
> - `hierarchy[0][i][3] == -1` → no parent → **external contour**
> - `hierarchy[0][i][3] != -1` → has a parent → **internal contour (hole)**
>
> **Expected results:**
> - Cell 416: external boundaries filled in white
> - Cell 418: holes/internal boundaries filled in white


### Feature matching

Feature matching finds correspondences between two images even when the object is not an exact pixel copy (changes in viewpoint, lighting, partial occlusion).

Typical pipeline:

1) detect keypoints
2) compute descriptors
3) match descriptors using a distance metric
4) optionally filter matches (ratio test, RANSAC)


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def display(img, cmap='gray'):
    fig = plt.figure(figsize=(12,10))
    ax = fig.add_subplot(111)
    ax.imshow(img, cmap='gray')

In [ ]:
reeses = cv2.imread('data/reeses_puffs.png', 0)

In [ ]:
display(reeses)

In [ ]:
cereals = cv2.imread('data/many_cereals.jpg', 0)

In [ ]:
display(cereals)

In [ ]:
# ORB Descriptors
orb = cv2.ORB_create()

In [ ]:
# Get keypoints and descriptors
kp1, des1 = orb.detectAndCompute(reeses, None)
kp2, des2 = orb.detectAndCompute(cereals, None)

In [ ]:
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)

In [ ]:
matches = bf.match(des1, des2)

In [ ]:
# Sort in order of distance
matches = sorted(matches, key=lambda x:x.distance)

In [ ]:
reeses_matches = cv2.drawMatches(reeses, kp1, cereals, kp2, matches[:25], None, flags=2)

In [ ]:
display(reeses_matches)

> **🔑 Feature matching with ORB + Brute-Force Matcher**
>
> **ORB (Oriented FAST and Rotated BRIEF)** is a fast, patent-free feature detector and descriptor:
> 1. `orb.detectAndCompute(img, None)` → finds keypoints (interesting points) + computes binary descriptors
> 2. `cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)` → matches descriptors using Hamming distance (counts different bits)
> 3. `bf.match(des1, des2)` → finds the best match for each descriptor
> 4. Sorting by `distance` → best matches first
>
> The `crossCheck=True` option ensures a match is only accepted if descriptor A's best match is B, **and** B's best match is also A.
>
> **Expected result**: lines connecting matching keypoints between the Reese's Puffs box and the cereals shelf image.  
> Some matches may be wrong (outliers) — this is normal with brute-force matching.


In [ ]:
# Scale Invariant Feature Transfer (SIFT) Descriptors
sift = cv2.xfeatures2d.SIFT_create()

In [ ]:
kp1, des1 = sift.detectAndCompute(reeses, None)
kp2, des2 = sift.detectAndCompute(cereals, None)

In [ ]:
bf = cv2.BFMatcher()

In [ ]:
matches = bf.knnMatch(des1, des2, k=2)

In [ ]:
# Ratio Test
# Less distance == better match
good = []

for match1, match2 in matches:
    # If match 1 distance is less than 75% of match 2 distance
    # then descriptor for that row is a good match, keep it
    if match1.distance < 0.75*match2.distance:
        good.append([match1])

In [ ]:
sift_matches = cv2.drawMatchesKnn(reeses, kp1, cereals, kp2, good, None, flags=2)

In [ ]:
display(sift_matches)

> **🎯 SIFT + Lowe's Ratio Test**
>
> **SIFT (Scale-Invariant Feature Transform)** produces more distinctive (but floating-point, larger) descriptors than ORB.  
> Note: SIFT requires `opencv-contrib-python`.
>
> **knnMatch(des1, des2, k=2)** returns the **2 best matches** for each descriptor.
>
> **Lowe's Ratio Test** (the filter in cell 438):
> ```python
> if match1.distance < 0.75 × match2.distance:
>     keep match1  # it's clearly better than the 2nd best
> ```
>
> If the best match is much better than the second best, it's likely a **true match**.  
> If they're similar in distance, the match is ambiguous → discard it.
>
> **Expected result**: fewer but more reliable matches compared to the brute-force ORB approach.


In [ ]:
# FLANN (Fast Library For Approximate Near Neighbors) Based Measure

In [ ]:
sift = cv2.xfeatures2d.SIFT_create()

In [ ]:
kp1, des1 = sift.detectAndCompute(reeses, None)
kp2, des2 = sift.detectAndCompute(cereals, None)

In [ ]:
FLANN_INDEX_KDTREE = 0
index_params = dict(algorithm=FLANN_INDEX_KDTREE,trees=5)
search_params = dict(checks=50)

In [ ]:
flann = cv2.FlannBasedMatcher(index_params, search_params)

In [ ]:
matches = flann.knnMatch(des1, des2, k=2)

In [ ]:
matchesMask = [[0,0] for i in range(len(matches))]

In [ ]:
# Ratio Test
# Less distance == better match
for i,(match1, match2) in enumerate(matches):
    # If match 1 distance is less than 70% of match 2 distance
    # then descriptor for that row is a good match, keep it
    if match1.distance < 0.7*match2.distance:
        matchesMask[i] = [1,0]

In [ ]:
draw_params = dict(matchColor=(0,255,0), 
                   singlePointColor=(255,0,0),
                   matchesMask=matchesMask,
                   flags=0)

In [ ]:
flann_matches = cv2.drawMatchesKnn(reeses, kp1, cereals, kp2, matches, None, **draw_params)

In [ ]:
display(flann_matches)

> **⚡ FLANN-based matching**
>
> **FLANN (Fast Library for Approximate Nearest Neighbors)** is faster than brute-force matching for large descriptor sets.
>
> - `FLANN_INDEX_KDTREE=0` with `trees=5` → builds a KD-tree index for efficient search
> - `checks=50` → number of tree traversals per query (higher = more accurate but slower)
>
> Same ratio test as before (`0.7` threshold), but results are stored in `matchesMask` (list of `[1,0]` / `[0,0]`) to control which matches are drawn.
>
> **Expected result**: matches visualized with green lines (good matches) and single red dots (unmatched keypoints) — similar to SIFT but computed faster for large datasets.


### Watershed algorithm

Watershed is a segmentation method that treats a grayscale image like a topographic surface:

- high intensity = peaks
- low intensity = valleys

If you place seed markers for different regions, the algorithm "floods" the valleys outward until different floods meet. The meeting lines become the region boundaries.


Analogy:

Imagine water rising in a landscape. Basins fill first (local minima). When water from two basins would merge, a ridge line is created to keep them separate. Those ridge lines are the segmentation boundaries.


In [ ]:
import cv2 
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def display(img, cmap='gray'):
    fig = plt.figure(figsize=(12,10))
    ax = fig.add_subplot(111)
    ax.imshow(img, cmap='gray')

In [ ]:
sep_coins = cv2.imread('data/pennies.jpg')

In [ ]:
plt.imshow(sep_coins)

In [ ]:
# Median Blur
sep_blur = cv2.medianBlur(sep_coins, 25)

In [ ]:
display(sep_blur)

In [ ]:
# Grayscale
gray_sep_coins = cv2.cvtColor(sep_blur, cv2.COLOR_BGR2GRAY)

In [ ]:
display(gray_sep_coins)

In [ ]:
# Binary Threshold
ret, sep_thresh = cv2.threshold(gray_sep_coins, 160, 255, cv2.THRESH_BINARY_INV)

In [ ]:
display(sep_thresh)

In [ ]:
# Find Contours
image = sep_thresh.copy()
contours, hierarchy = cv2.findContours(image, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)

In [ ]:
for i in range(len(contours)):
    if hierarchy[0][i][3] == -1:
        cv2.drawContours(sep_coins, contours, i, (255,0,0), 10)

In [ ]:
display(sep_coins)

> **🪙 Simple coin detection with thresholding + contours**
>
> This is a first attempt at separating touching coins:
> 1. **Median blur** (cell 459) — smooths the image while preserving coin boundaries
> 2. **Grayscale** (cell 461)
> 3. **Binary inverse threshold** (cell 463) — coins become white, background becomes black
> 4. **Find + draw contours** (cells 465–466)
>
> **Limitation**: if coins are touching, `findContours` may detect them as a single blob.  
> The Watershed algorithm (next section) solves this problem.


In [ ]:
# WATERSHED ALGORITHM
img = cv2.imread('data/pennies.jpg')

In [ ]:
img = cv2.medianBlur(img, 35)

In [ ]:
display(img)

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

In [ ]:
# Otsu's Method
ret, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)

In [ ]:
display(thresh)

In [ ]:
# NOISE REMOVAL (OPTIONAL)  
kernel = np.ones((3,3), np.uint8)

In [ ]:
opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)

In [ ]:
display(opening)

> **🧹 Watershed step 1 — Noise removal with Opening**
>
> `cv2.THRESH_OTSU` automatically determines the best global threshold using Otsu's method:  
> it finds the value that minimizes the within-class intensity variance (works best for bimodal histograms).
>
> `cv2.THRESH_BINARY_INV` inverts the result: coins become white, background becomes black.
>
> `cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)` removes small noise blobs that could confuse the Watershed algorithm.


In [ ]:
sure_bg = cv2.dilate(opening, kernel, iterations=3)

In [ ]:
display(sure_bg, cmap='gray')

In [ ]:
# DISTANCE TRANSFORM
dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)

In [ ]:
display(dist_transform)

In [ ]:
ret, sure_fg = cv2.threshold(dist_transform, 0.7*dist_transform.max(), 255,0)

In [ ]:
display(sure_fg)

In [ ]:
sure_fg = np.uint8(sure_fg)

In [ ]:
unknown = cv2.subtract(sure_bg, sure_fg)

In [ ]:
display(unknown)

> **📍 Watershed step 2 — Distance Transform & Sure Foreground/Background**
>
> **Distance Transform** (`cv2.distanceTransform`): each white pixel is replaced by its distance to the nearest black pixel.  
> → Centers of coins appear bright (far from background); coin edges appear dark.
>
> **Sure foreground** (cell 481): threshold at `0.7 × max(dist_transform)` → only keep the coin centers that are definitely foreground.
>
> **Sure background** (cell 477): dilate the opened mask → expand to get pixels that are definitely background.
>
> **Unknown region** (cell 484): `sure_bg - sure_fg` → the border zone between coins (where we're unsure) — this is exactly where Watershed will place its boundaries.


In [ ]:
ret, markers = cv2.connectedComponents(sure_fg)

In [ ]:
markers = markers + 1

In [ ]:
markers[unknown==255] = 0

In [ ]:
display(markers)

In [ ]:
markers = cv2.watershed(img, markers)

In [ ]:
display(markers)

> **🌊 Watershed step 3 — Running the algorithm**
>
> `cv2.watershed(img, markers)` floods from the seed markers:
> - Regions marked `≥ 2` → known coin regions (seeds)
> - Region marked `1` → known background
> - Region marked `0` → unknown (to be determined)
>
> After running, **boundary pixels are marked with `-1`** — these are the watershed lines separating touching coins.
>
> **Expected result in cell 491**: a marker image where each coin has a unique label ID and boundaries are -1.  
> The final cell 494 draws the detected contours in blue on the original coin image — each touching coin should be separated.


In [ ]:
image = markers.copy()
contours, hierarchy = cv2.findContours(image, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)

In [ ]:
for i in range(len(contours)):
    if hierarchy[0][i][3] == -1:
        cv2.drawContours(sep_coins, contours, i, (255,0,0), 10)

In [ ]:
display(sep_coins)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
road = cv2.imread('data/road_image.jpg')

In [ ]:
road_copy = np.copy(road)

In [ ]:
plt.imshow(road)

In [ ]:
road.shape

In [ ]:
marker_image = np.zeros(road.shape[:2], dtype=np.int32)

In [ ]:
segments = np.zeros(road.shape, dtype=np.uint8)

In [ ]:
from matplotlib import cm

In [ ]:
cm.tab10(0)

In [ ]:
def create_rgb(i):
    return tuple(np.array(cm.tab10(i)[:3]) * 255)

In [ ]:
colors = []
for i in range(10):
    colors.append(create_rgb(i))

In [ ]:
colors

In [ ]:
# check the main python file :
# python3 main.py --type="watershed"

In this demo you:

- load a road image
- mark regions with the mouse (seed labels)
- run watershed using those markers
- visualize each region with a different color
- optionally clear markers or switch label IDs interactively


### Haar cascades and face detection

Haar cascades (Viola-Jones) can detect faces (or other objects) by scanning an image with many simple rectangular features.

Important distinction:

- detection: "is there a face and where is it?"
- recognition: "whose face is it?" (this requires a different approach, often deep learning)

Why it can be fast:

- integral images allow fast feature sums
- a cascade of classifiers quickly rejects most non-face windows

Limitations:

- sensitive to pose/lighting/occlusion
- training your own cascades is data- and time-intensive


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
nadia = cv2.imread('data/Nadia_Murad.jpg', 0)
dennis = cv2.imread('data/Denis_Mukwege.jpg', 0)
solvay = cv2.imread('data/solvay_conference.jpg', 0)

In [ ]:
face_cascade = cv2.CascadeClassifier('data/haarcascades/haarcascade_frontalface_default.xml')

In [ ]:
def detect_face(img):
    face_img = img.copy()
    
    face_rects = face_cascade.detectMultiScale(face_img)
    
    for (x,y,w,h) in face_rects:
        cv2.rectangle(face_img, (x,y), (x+w, y+h), (255,255,255), 10)
        
    return face_img

In [ ]:
result = detect_face(dennis)

In [ ]:
plt.imshow(result, cmap='gray')

> **👤 Face detection with Haar Cascades**
>
> `cv2.CascadeClassifier` loads a pre-trained cascade model from an XML file.  
> `face_cascade.detectMultiScale(img)` scans the image at multiple scales and returns a list of rectangles `(x, y, w, h)` where faces were detected.
>
> **How it works:**
> 1. The image is scanned with a sliding window at multiple sizes
> 2. At each position, ~6000 simple rectangular features are evaluated quickly using **integral images**
> 3. The cascade structure rejects most non-face windows after just a few features
> 4. Only windows that pass all stages are reported as faces
>
> **Expected result**: white rectangle(s) drawn around detected faces.


In [ ]:
result = detect_face(solvay)

In [ ]:
plt.imshow(result, cmap='gray')

In [ ]:
def adj_detect_face(img):
    face_img = img.copy()
    
    face_rects = face_cascade.detectMultiScale(face_img, scaleFactor=1.2, minNeighbors=5)
    
    for (x,y,w,h) in face_rects:
        cv2.rectangle(face_img, (x,y), (x+w, y+h), (255,255,255), 10)
        
    return face_img

In [ ]:
result = adj_detect_face(solvay)

In [ ]:
plt.imshow(result, cmap='gray')

> **⚙️ Tuning detection parameters**
>
> `detectMultiScale(img, scaleFactor=1.2, minNeighbors=5)`:
>
> | Parameter | Effect |
> |---|---|
> | `scaleFactor=1.2` | Image is shrunk by 20% at each scale step; smaller value → more scales checked → slower but catches more sizes |
> | `minNeighbors=5` | A detection is kept only if it has at least 5 overlapping neighbor detections; higher → fewer but more reliable detections |
>
> **When to tune**: if you're getting too many false positives (random things detected as faces), increase `minNeighbors`. If real faces are being missed, decrease it.


In [ ]:
eye_cascade = cv2.CascadeClassifier('data/haarcascades/haarcascade_eye.xml')

In [ ]:
def detect_eyes(img):
    face_img = img.copy()
    
    eye_rects = eye_cascade.detectMultiScale(face_img, scaleFactor=1.2, minNeighbors=5)
    
    for (x,y,w,h) in eye_rects:
        cv2.rectangle(face_img, (x,y), (x+w, y+h), (255,255,255), 10)
        
    return face_img

In [ ]:
result = detect_eyes(nadia)

In [ ]:
plt.imshow(result, cmap='gray')

In [ ]:
result = detect_eyes(dennis)
plt.imshow(result, cmap='gray')

In [ ]:
# check the main python file :
# python3 main.py --type="detect_face"

## Object tracking

Tracking follows an object through a video over time once you have an initial location.


### Optical flow

Optical flow estimates motion between two consecutive frames.

Common assumptions:

1) pixel intensities of a moving point do not change much between frames
2) neighboring pixels move similarly

In OpenCV:

- Lucas-Kanade computes **sparse** optical flow (for selected points)
- Farneback computes **dense** optical flow (for many/all pixels)

A common visualization converts flow vectors to HSV: hue encodes direction, value encodes magnitude.

Image pyramids can improve tracking across scales: https://en.wikipedia.org/wiki/Pyramid_(image_processing)


In [ ]:
import cv2
import numpy as np

In [ ]:
# SPARSE OPTICAL FLOW (LUCAS KANADE)
corner_track_params = dict(maxCorners=25, qualityLevel=0.3, minDistance=7, blockSize=7)

In [ ]:
corner_track_params

In [ ]:
lk_params = dict(winSize=(200,200), maxLevel=2,  criteria =(cv2.TERM_CRITERIA_EPS | cv2.TermCriteria_COUNT, 10, 0.03))

> **🏃 Sparse Optical Flow — Lucas-Kanade parameters**
>
> **Shi-Tomasi params** (`corner_track_params`):
> - `maxCorners=25` — track at most 25 points
> - `qualityLevel=0.3` — only keep corners with at least 30% of the best corner quality
> - `minDistance=7` — minimum pixel distance between tracked points
>
> **Lucas-Kanade params** (`lk_params`):
> - `winSize=(200, 200)` — search window around each point (large = can handle fast motion)
> - `maxLevel=2` — use 3-level image pyramid (handles motion at different scales)
> - `criteria` — stopping condition (stop when moving < 0.03 px OR after 10 iterations)
>
> **What it does**: for each selected corner, LK estimates where it moved in the next frame by minimizing intensity difference inside the window.


In [ ]:
# Lucas-Kanade Sparse Optical Flow
# check the main python file :
# python3 main.py --type="sparse_optical_flow"

In [ ]:
# DENSE OPTICAL FLOW (GUNNER FARNEBACK)
# check the main python file :
# python3 main.py --type="dense_optical_flow2"

### MeanShift and CamShift

MeanShift iteratively moves a window toward the region with the highest density (in this context, the best match to a color distribution).

Typical tracking idea:

- select a region of interest (ROI)
- build a color histogram for that ROI
- compute back-projection on each frame
- shift the window toward the peak response until convergence

CamShift (Continuously Adaptive MeanShift) extends this by adapting the window size (and rotation in some variants) as the target changes scale.


In [ ]:
import cv2
import numpy as np

This function detects a face once, then tracks it in real time using MeanShift (optionally CamShift).

Tracking is based on a color-histogram back-projection of the detected face region.


In [ ]:
# check the main python file :
# python3 main.py --type="meanshift"

> **🎯 MeanShift and CamShift tracking**
>
> **MeanShift tracking** workflow:
> 1. Select a ROI (region of interest) around the object to track (e.g., a face)
> 2. Compute a **color histogram** (usually in Hue channel of HSV) for that ROI
> 3. For each new frame, compute the **back-projection** — a map showing how likely each pixel belongs to the target color distribution
> 4. **MeanShift** iteratively moves the search window toward the peak of the back-projection
>
> **CamShift** extends MeanShift by also **adapting the window size** as the object gets closer/farther.
>
> **Expected behavior**: the window follows the face across frames even as it moves.  
> Run via `python3 main.py --type="meanshift"` as it requires a live video window.


### Note on OpenCV versions

Some trackers live in `opencv-contrib-python` and APIs differ across OpenCV versions. If a tracker constructor is missing, check your OpenCV version and whether contrib modules are installed.


### Built-in Tracking APIs (OpenCV)

OpenCV provides several **ready-to-use object tracking algorithms**, accessible through simple API calls:

- **Boosting Tracker**  
  - Based on the **AdaBoost algorithm** (same principle as Haar Cascade face detection)  
  - Evaluates the object across **multiple frames**  

- **MIL Tracker (Multiple Instance Learning)**  
  - Extends Boosting by considering a **neighborhood of points** around the current location  
  - Creates **multiple instances** for more robust tracking  

- **KCF Tracker (Kernelized Correlation Filters)**  
  - Builds on MIL principles and exploits overlapping data points  
  - **Faster and more accurate**, often a **good first choice** for tracking  

- **TLD Tracker (Tracking, Learning, Detection)**  
  - Handles **occlusions** and **large scale changes** well  
  - Can produce **false positives**, but robust under challenging conditions  

- **MedianFlow Tracker**  
  - Excellent at **detecting tracking failures**  
  - Works well with **predictable motion**, but fails with **fast-moving objects**


# Capstone Projects (Pick 1)

Deliverable format (recommended):
- A single notebook section with: problem statement, approach, results, and failure cases
- Clear visualizations + a few quantitative metrics

Project A: Document Scanner
- Input: photo of a document
- Output: rectified scan + optional contrast enhancement (CLAHE)
- Bonus: automatically find corners using edges + contours

Project B: Color-Based Object Counter
- Input: image with objects of a specific color (e.g., candies, LEGO, tokens)
- Output: mask cleaning (morphology) + connected components count + bounding boxes
- Bonus: report size distribution (areas histogram)

Project C: Classical 'Find My Logo'
- Input: a template image + a larger scene image
- Output: detected location using ORB matching + RANSAC homography
- Bonus: compare to template matching and explain when each fails
